

# Technical Report: Systematic Parallelism Strategy for Large-Scale Distributed LLM Training

## A Principal-Engineer Reference for Fitting, Scaling, and Optimizing Training Across GPU Clusters

---

## Table of Contents

1. [Executive Summary](#1-executive-summary)
2. [Foundational Principles: Memory Anatomy and Compute Cost Models](#2-foundational-principles)
3. [Step 1 — Fitting the Model into GPU Memory](#3-step-1)
4. [Step 2 — Satisfying the Target Global Batch Size](#4-step-2)
5. [Step 3 — Optimizing Training Throughput](#5-step-3)
6. [Parallelization Strategy Reference Matrix](#6-parallelization-strategy-reference-matrix)
7. [Framework-Specific Implementation Guidance](#7-framework-specific-implementation)
8. [Kernel-Level Optimization and Numerical Stability](#8-kernel-level-optimization)
9. [Data Pipeline Engineering for Distributed Training](#9-data-pipeline-engineering)
10. [Fault Tolerance, Automation, and Production Resilience](#10-fault-tolerance)
11. [Profiling, Instrumentation, and Regression Gating](#11-profiling-and-instrumentation)
12. [Cross-Vendor Portability: NVIDIA and AMD](#12-cross-vendor-portability)
13. [Glossary and Reference Formulae](#13-glossary)
14. [Conclusion](#14-conclusion)

---

## 1. Executive Summary

Training large language models at scale is fundamentally a **systems engineering problem** governed by three hard constraints: GPU memory capacity, interconnect bandwidth, and target convergence quality. Every architectural decision—parallelism degree, micro-batch size, activation checkpointing strategy, optimizer shard placement—derives from these constraints through quantitative analysis, not heuristic guessing.

This report presents a rigorous, three-step methodology for configuring distributed training:

1. **Fit the model into memory** by selecting the minimum necessary parallelism and memory-reduction techniques.
2. **Satisfy the target global batch size** by adjusting data-parallel width, context parallelism, and gradient accumulation.
3. **Optimize training throughput** by experimentally tuning parallelism degrees, micro-batch sizes, kernel selection, and compute-communication overlap.

Every recommendation is grounded in first-principles memory formulae, communication cost models, and production experience across A100 (80 GB HBM2e), H100 (80 GB HBM3), B200 (192 GB HBM3e), MI300X (192 GB HBM3), and MI350-class clusters. Implementation guidance covers **Megatron-LM / Megatron-Core**, **DeepSpeed**, **PyTorch FSDP / DTensor**, **Triton kernels**, and **torchrun** orchestration.

---

## 2. Foundational Principles: Memory Anatomy and Compute Cost Models

Before any parallelism decision can be made, the engineer must derive exact memory and compute requirements from model configuration. All downstream choices are algebraic consequences of these quantities.

### 2.1 Memory Decomposition Per GPU

The peak memory consumption during a single training step on one GPU is decomposed as:

$$
\text{peak\_memory} = M_{\text{model\_bf16}} + M_{\text{model\_fp32}} + M_{\text{grads\_fp32}} + M_{\text{optimstates}} + M_{\text{activations}} + M_{\text{transient}}
$$

Each term is defined precisely:

| Component | Formula | Description |
|-----------|---------|-------------|
| **model_bf16** | $2 \times N_{\text{params}}$ bytes | BF16 model weights used in forward/backward |
| **model_fp32** | $4 \times N_{\text{params}} / \text{dp\_if\_zero1}$ bytes | FP32 master copy held by optimizer |
| **grads_fp32** | $4 \times N_{\text{params}} / \text{dp\_if\_zero2}$ bytes | FP32 gradients (reduced or scattered) |
| **optimstates** | $8 \times N_{\text{params}} / \text{dp\_if\_zero1}$ bytes | Adam first/second moments (2 × FP32 per param) |
| **activations** | $f(\text{config}, \text{mseqlen}, \text{mbs}, \text{tp}, \text{cp}, \text{pp}, \text{recomp})$ | Intermediate tensors from forward pass |
| **transient** | Variable | Workspace buffers, NCCL scratch, fragmentation |

Where the parameter count for a standard Transformer is:

$$
N_{\text{params}} \approx N_{\text{layers}} \times 12 \times h^2
$$

with $h$ being the hidden dimension. This accounts for the four weight matrices in self-attention ($W_Q, W_K, W_V, W_O$, each $h \times h$) and the two MLP matrices (with intermediate size $4h$, giving $h \times 4h$ and $4h \times h$), plus biases and layer norm parameters (which are negligible at scale but included in precise accounting).

Therefore:

$$
M_{\text{model\_bf16}} = 2 \times N_{\text{layers}} \times 12 \times h^2 \text{ bytes}
$$

### 2.2 Activation Memory

Activation memory is the dominant variable the engineer controls. For a single Transformer layer with sequence length $s$, micro-batch size $b$, hidden dimension $h$, and number of attention heads $a$:

$$
M_{\text{act\_per\_layer}} = s \times b \times h \times \left(34 + 5 \frac{a \times s}{h}\right) \text{ bytes (mixed precision, no recompute)}
$$

With **full activation recomputation**, only the input activation to each layer is stored:

$$
M_{\text{act\_per\_layer}}^{\text{full\_recomp}} = 2 \times s \times b \times h \text{ bytes}
$$

With **selective recomputation** (recomputing only attention softmax and dropout), the savings are intermediate—typically 60–70% of full activation memory is recovered at roughly 5–8% compute overhead versus the 30–35% overhead of full recomputation.

Parallelism further divides activations:

$$
M_{\text{activations}} = \frac{N_{\text{layers}}}{\text{pp}} \times \frac{M_{\text{act\_per\_layer}}}{\text{tp} \times \text{cp}}
$$

### 2.3 Compute Cost Per Training Step

The total floating-point operations for a single forward and backward pass (the backward costs approximately 2× the forward) is:

$$
C_{\text{step}} = 6 \times N_{\text{params}} \times \text{mbs} \times s \times \text{gas}
$$

The factor of 6 arises from: 2 FLOPs per parameter per token for forward (multiply-accumulate), multiplied by 3 (one forward + two backward passes for weight and input gradients).

**Model FLOPS Utilization (MFU)** is then:

$$
\text{MFU} = \frac{C_{\text{step}}}{\text{step\_time} \times N_{\text{GPUs}} \times \text{peak\_FLOPS\_per\_GPU}}
$$

Target MFU benchmarks for dense models:

| GPU Class | BF16 Peak TFLOPS | Good MFU Range | Excellent MFU |
|-----------|-------------------|-----------------|----------------|
| A100 SXM | 312 | 38–45% | >48% |
| H100 SXM | 989 (with sparsity: 1979) | 40–48% | >52% |
| B200 SXM | ~2,250 | 42–50% | >54% |
| MI300X | 1,307 | 35–42% | >45% |

> **Engineering Principle:** MFU below 35% on a well-tuned cluster indicates a systemic inefficiency—typically communication bottleneck, pipeline bubble overhead, dataloader starvation, or kernel underutilization. Never accept low MFU without a root-cause decomposition.

---

## 3. Step 1 — Fitting the Model into GPU Memory

The first and non-negotiable requirement is that the model, its optimizer state, gradients, and activations must physically fit within the aggregate HBM of the allocated GPUs. This step determines the **minimum parallelism** required before any throughput optimization.

### 3.1 Decision Framework

The decision tree is governed by two variables: **model size** and **GPU count availability**.

#### 3.1.1 GPU-Rich Scenario (Sufficient GPUs Available)

**Case A: Small Models (< 10B Parameters)**

For models under approximately 10 billion parameters, a single parallelism dimension typically suffices across a single 8-GPU node:

- **Option 1: ZeRO-3 / FSDP with DP=8.** Shards all model states (weights, gradients, optimizer states) across 8 GPUs. Combined with full activation recomputation, this fits models up to ~10B on 80 GB HBM GPUs.
- **Option 2: TP=8 within a single node.** Shards weights and activations across 8 GPUs connected via NVLink/NVSwitch. Lower communication overhead than ZeRO-3 for small DP worlds, but requires code that supports TP (Megatron-LM, Megatron-Core).

**Quantitative justification for 7B model on 8× A100-80GB:**

```
N_params = 32 layers × 12 × 4096² = 6.44B parameters

Without any parallelism (single GPU):
  model_bf16  = 2 × 6.44B = 12.88 GB
  model_fp32  = 4 × 6.44B = 25.76 GB
  grads_fp32  = 4 × 6.44B = 25.76 GB
  optimstates = 8 × 6.44B = 51.52 GB
  Total (no activations) = 115.9 GB → Does not fit on 80 GB

With ZeRO-3, DP=8:
  model_bf16  = 12.88 / 8 = 1.61 GB
  model_fp32  = 25.76 / 8 = 3.22 GB
  grads_fp32  = 25.76 / 8 = 3.22 GB
  optimstates = 51.52 / 8 = 6.44 GB
  Subtotal = 14.49 GB
  Activations (full recomp, mbs=1, seq=4096) ≈ 2 × 4096 × 1 × 4096 × 32 = 1.07 GB
  Total ≈ 15.6 GB → Fits comfortably on 80 GB
```

**Case B: Large Models (10B–100B+ Parameters)**

Models exceeding 10B parameters require more than 8 GPUs, which introduces a critical topological boundary: **inter-node communication**. The engineer must select a combination strategy:

- **Strategy 1: TP=8 (intra-node via NVLink) + PP (inter-node via InfiniBand/RoCE).** Pipeline parallelism partitions layers across nodes. Each node runs TP=8 internally. PP communication is point-to-point (send/recv of activation tensors), which is far more bandwidth-efficient than collective operations over the inter-node fabric.

- **Strategy 2: TP=8 (intra-node) + ZeRO-3 / FSDP (inter-node).** ZeRO-3 shards optimizer states across all GPUs globally. This avoids the pipeline bubble but introduces all-gather and reduce-scatter collectives across the inter-node fabric for every layer, forward and backward.

- **Strategy 3: Pure ZeRO-3 across all GPUs.** Simplest to configure but places the heaviest communication burden on the inter-node interconnect, as every forward and backward layer requires an all-gather of the full layer weights.

> **Critical Topological Constraint:** Tensor Parallelism (TP) must be confined to GPUs connected by the highest-bandwidth interconnect—NVLink/NVSwitch within a node (900 GB/s bidirectional on H100 SXM). Extending TP across InfiniBand (400 Gb/s = 50 GB/s per port, typically 8 ports = 400 GB/s per node) incurs 2–4× latency increase per collective and collapses MFU. The only exception is clusters with dedicated NVLink-connected multi-node domains (e.g., GB200 NVL72 or DGX SuperPOD with NVSwitch fabric).

**Quantitative justification for 70B model on 64× H100:**

```
N_params ≈ 80 layers × 12 × 8192² ≈ 64.4B parameters (approximation; actual 70B uses GQA)

TP=8, PP=4, DP=2 → 8 × 4 × 2 = 64 GPUs

Per-GPU memory:
  model_bf16  = 2 × 64.4B / (8 × 4) = 4.03 GB
  model_fp32  = 4 × 64.4B / 8 = 32.2 GB (ZeRO-0, no sharding of optimizer across DP)
  
  With ZeRO-1 on DP=2:
  model_fp32  = 32.2 / 2 = 16.1 GB
  optimstates = 64.4 / 2 = 32.2 GB → still 32.2 / 2 = 16.1 GB per DP rank

  Total model states ≈ 4.03 + 16.1 + 4.03 + 16.1 = 40.3 GB
  Activations (selective recomp, mbs=1, seq=8192, 20 layers/pp_stage) ≈ 12–18 GB
  Total ≈ 52–58 GB → Fits on 80 GB H100
```

**Case C: 512+ GPU Scale**

At 512+ GPUs, pure DP/ZeRO-3 becomes communication-inefficient because:

1. **All-gather volume per layer per step scales as** $(1 - 1/\text{DP}) \times M_{\text{layer\_params}}$. With DP=512, each of the $2 \times N_{\text{layers}}$ all-gather operations (forward + backward) transfers nearly the full layer weight. For an 8192-hidden model with 80 layers, this is $\sim 2 \times 80 \times 2 \times 12 \times 8192^2 / (8 \times 10^9) \approx 25.8$ GB of data moved per step per GPU—far exceeding what inter-node InfiniBand can sustain without becoming the bottleneck.

2. **Latency of collectives grows logarithmically with world size** in ring or tree topologies. At 512 ranks, each all-gather has $O(\log 512) = 9$ steps of network latency overhead, compounding across hundreds of layers.

**Recommended approach at 512+ GPUs:** Combine TP (intra-node, degree 4 or 8) with PP (4–16 stages) and DP with ZeRO-1 or ZeRO-2 (not ZeRO-3, to avoid per-layer all-gathers). DP handles the remaining parallelism.

**Case D: 1024+ GPU Scale**

The reference configuration at this scale:

$$
\text{TP}=8, \quad \text{PP} \in [4, 16], \quad \text{DP} = \frac{N_{\text{GPUs}}}{\text{TP} \times \text{PP}}, \quad \text{ZeRO-2 on DP dimension}
$$

For 2048 H100 GPUs training a 405B model (Llama-3.1 class):

```
TP=8, PP=16, DP = 2048 / (8 × 16) = 16, ZeRO-1 on DP

Per-GPU parameters: 405B / (8 × 16) = 3.16B
model_bf16 = 6.33 GB
model_fp32 = 12.66 / 16 (ZeRO-1) = 0.79 GB
optimstates = 25.32 / 16 = 1.58 GB
grads_fp32 = 12.66 GB (ZeRO-0 on grads, or /16 with ZeRO-2 = 0.79 GB)

With ZeRO-2: Total model states ≈ 6.33 + 0.79 + 0.79 + 1.58 = 9.49 GB
Activations (selective recomp, 5 layers/stage): ~15–25 GB
Total: 25–35 GB → Ample headroom on 80 GB H100
```

**Case E: Special Architectures**

- **Long Context (≥ 32K tokens):** Activation memory scales linearly with sequence length (and quadratically in the attention score matrix without FlashAttention). **Context Parallelism (CP)** partitions the sequence dimension across GPUs, reducing per-GPU activation memory by a factor of CP. Preferred over increasing TP beyond 8 because CP communication (ring send/recv of KV blocks) overlaps with attention compute.

- **Mixture-of-Experts (MoE):** Expert parameters inflate the model size by the number of experts but each token activates only top-$k$ experts. **Expert Parallelism (EP)** distributes experts across GPUs, with all-to-all collectives routing tokens. EP is orthogonal to TP/PP/DP and typically occupies a dedicated process-group dimension.

#### 3.1.2 GPU-Poor Scenario (Insufficient GPUs Available)

When the GPU count is constrained and the model does not fit even with TP/PP across available devices, the engineer must reduce per-GPU memory demand:

1. **Full Activation Checkpointing (Recomputation):** Discard all intermediate activations during the forward pass; recompute them during the backward pass. Reduces activation memory to $O(N_{\text{layers}} \times s \times b \times h)$ at the cost of ~33% additional compute.

2. **Gradient Accumulation:** Reduce micro-batch size to 1 and accumulate gradients over multiple forward-backward passes. Reduces activation memory proportionally to micro-batch size.

3. **CPU/NVMe Offload (DeepSpeed ZeRO-Infinity):** Offload optimizer states and optionally parameters to host DRAM or NVMe. Trades PCIe bandwidth for HBM capacity. Only viable when compute density is low enough that PCIe transfers do not dominate step time.

4. **Mixed Precision with FP8 Weights:** On H100/B200/MI300X, storing weights in FP8 (E4M3 for forward, E5M2 for backward) reduces model_bf16 by 50%. Requires careful loss-scaling and master weights in BF16/FP32.

**Pseudocode: Memory Feasibility Check**

```
ALGORITHM: CheckMemoryFeasibility
INPUT: model_config, gpu_hbm_capacity, tp, pp, dp, cp, zero_stage,
       recompute_mode, mbs, seq_len
OUTPUT: fits (boolean), peak_memory_gb, headroom_gb

1. N_params ← ComputeParamCount(model_config)
2. layers_per_stage ← model_config.num_layers / pp

3. M_model_bf16 ← 2 × N_params / (tp × pp)
4. IF zero_stage ≥ 3:
       M_model_bf16 ← M_model_bf16 / dp

5. M_model_fp32 ← 2 × M_model_bf16       // FP32 master copy
6. IF zero_stage ≥ 1:
       M_model_fp32 ← M_model_fp32 / dp

7. M_grads ← 2 × M_model_bf16             // FP32 gradients
8. IF zero_stage ≥ 2:
       M_grads ← M_grads / dp

9. M_optim ← 4 × M_model_bf16             // Adam: 2 × FP32 states
10. IF zero_stage ≥ 1:
        M_optim ← M_optim / dp

11. M_act ← ComputeActivationMemory(model_config, seq_len / cp, mbs,
                                      tp, recompute_mode, layers_per_stage)

12. M_transient ← EstimateTransientBuffers(tp, pp, dp, seq_len, mbs)

13. peak_memory ← M_model_bf16 + M_model_fp32 + M_grads + M_optim + M_act + M_transient
14. headroom ← gpu_hbm_capacity - peak_memory
15. fits ← (headroom > 0)

16. RETURN fits, peak_memory, headroom
```

```
ALGORITHM: SelectMinimumParallelism
INPUT: model_config, num_gpus, gpu_hbm_capacity, target_gbs, seq_len
OUTPUT: parallelism_config {tp, pp, dp, cp, ep, zero_stage, mbs, gas, recompute}

1. FOR tp IN [1, 2, 4, 8]:
     FOR pp IN [1, 2, 4, 8, 16, 32]:
       FOR zero_stage IN [0, 1, 2, 3]:
         FOR recompute IN [none, selective, full]:
           FOR mbs IN [4, 2, 1]:
             dp ← num_gpus / (tp × pp)
             IF dp < 1: CONTINUE
             
             fits, peak_mem, headroom ← CheckMemoryFeasibility(
                 model_config, gpu_hbm_capacity, tp, pp, dp, 1, zero_stage,
                 recompute, mbs, seq_len)
             
             IF NOT fits: CONTINUE
             
             gas ← target_gbs / (mbs × dp)
             IF gas < 1: CONTINUE
             
             comm_cost ← EstimateCommunicationCost(tp, pp, dp, zero_stage, model_config)
             bubble_frac ← (pp - 1) / (gas + pp - 1)  // 1F1B schedule
             
             RECORD config {tp, pp, dp, zero_stage, recompute, mbs, gas,
                           peak_mem, headroom, comm_cost, bubble_frac}

2. SORT candidates BY (comm_cost + bubble_penalty) ASCENDING
3. RETURN candidates[0]
```

### 3.2 Topology-Aware Placement Rules

The selection of parallelism degrees is inseparable from the physical topology:

| Parallelism Dimension | Preferred Interconnect | Rationale |
|----------------------|----------------------|-----------|
| **TP** | NVLink / NVSwitch (intra-node) | 4 all-reduce (or all-gather + reduce-scatter) per layer; latency-sensitive; requires 450–900 GB/s |
| **PP** | InfiniBand / RoCE (inter-node) or NVLink | Point-to-point send/recv; modest bandwidth; tolerates 25–50 µs latency per microbatch |
| **DP / ZeRO** | InfiniBand / RoCE (inter-node) | Collective per gradient bucket; overlaps with backward compute; bandwidth-bound |
| **CP** | NVLink preferred; InfiniBand acceptable | Ring send/recv of KV blocks; overlaps with attention FLOPS |
| **EP** | InfiniBand / RoCE | All-to-all routing; bandwidth-bound; scale-sensitive |

> **Rule of Thumb (World-Size Factorization):** For $N$ GPUs, with $G$ GPUs per node:
> $$N = \text{TP} \times \text{PP} \times \text{DP} \times \text{CP} \times \text{EP}$$
> Constrain $\text{TP} \leq G$ (intra-node). Place TP on the innermost (fastest) communication domain. PP and DP span inter-node. CP can share the DP dimension or be factored separately. EP is typically orthogonal, mapped to a subset of DP ranks.

---

## 4. Step 2 — Satisfying the Target Global Batch Size

### 4.1 Why Global Batch Size Matters

Empirical scaling laws and training stability research (Kaplan et al., 2020; Hoffmann et al., 2022; McCandlish et al., 2018) demonstrate that there exists an optimal batch size for a given compute budget that minimizes the total number of training tokens to reach a target loss. For most large-scale LLM pretraining runs, this optimal range is:

$$
\text{GBS}_{\text{optimal}} \in [4\text{M}, 40\text{M}] \text{ tokens}
$$

The global batch size in tokens is:

$$
\text{GBS}_{\text{tokens}} = \text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}
$$

Or equivalently in sequences:

$$
\text{GBS}_{\text{sequences}} = \text{mbs} \times \text{dp} \times \text{gas}
$$

> **Critical Constraint:** GBS is a hyperparameter determined by convergence experiments. The parallelism configuration must achieve exactly this GBS—deviating changes the optimization trajectory and invalidates the learning rate schedule.

### 4.2 Adjusting GBS Upward

If Step 1 produced a configuration with $\text{mbs} \times \text{dp} \times \text{gas} < \text{GBS}_{\text{target}}$, increase GBS by:

1. **Increase gradient accumulation steps (gas).** Zero communication overhead; increases step time linearly. Useful when GPU count is fixed.

2. **Scale up DP.** Requires more GPUs. Each additional DP rank contributes $\text{mbs}$ sequences per step. Adds gradient synchronization cost but overlaps with backward pass.

3. **Scale up CP.** If context parallelism is in use, increasing CP allows the same number of tokens per step while splitting longer sequences. Each CP rank processes $\text{seq\_len} / \text{CP}$ tokens. If GBS is measured in tokens and seq_len is fixed, CP does not directly increase GBS—it is primarily a memory technique. However, if CP enables processing longer sequences, total tokens per step increase.

**Pseudocode: GBS Adjustment**

```
ALGORITHM: AdjustGlobalBatchSize
INPUT: target_gbs_tokens, current_config {mbs, dp, gas, seq_len, cp},
       max_gpus, gpu_hbm_capacity, model_config
OUTPUT: adjusted_config

1. current_gbs ← mbs × dp × gas × seq_len
2. IF current_gbs == target_gbs_tokens: RETURN current_config

3. IF current_gbs < target_gbs_tokens:
     // Strategy 1: Increase gradient accumulation (no extra GPUs)
     required_gas ← CEIL(target_gbs_tokens / (mbs × dp × seq_len))
     IF required_gas ≤ MAX_GAS_THRESHOLD:
         RETURN config WITH gas ← required_gas

     // Strategy 2: Scale DP (requires more GPUs)
     required_dp ← CEIL(target_gbs_tokens / (mbs × gas × seq_len))
     new_total_gpus ← required_dp × tp × pp
     IF new_total_gpus ≤ max_gpus:
         VALIDATE memory feasibility with new dp
         RETURN config WITH dp ← required_dp

     // Strategy 3: Increase mbs if memory permits
     required_mbs ← CEIL(target_gbs_tokens / (dp × gas × seq_len))
     IF CheckMemoryFeasibility(..., mbs=required_mbs, ...):
         RETURN config WITH mbs ← required_mbs

4. IF current_gbs > target_gbs_tokens:
     // Reduce gas first (cheapest adjustment)
     new_gas ← MAX(1, FLOOR(target_gbs_tokens / (mbs × dp × seq_len)))
     remaining ← target_gbs_tokens / (mbs × new_gas × seq_len)
     
     IF remaining < dp:
         // Reduce DP, reallocate GPUs to TP or PP
         new_dp ← FLOOR(remaining)
         freed_gpus ← (dp - new_dp) × tp × pp
         // Consider increasing PP to use freed GPUs for larger models
         RETURN config WITH dp ← new_dp, gas ← new_gas
     
     // Reduce mbs
     new_mbs ← MAX(1, FLOOR(target_gbs_tokens / (dp × new_gas × seq_len)))
     RETURN config WITH mbs ← new_mbs, gas ← new_gas
```

### 4.3 The Interplay Between GBS and Pipeline Parallelism

Pipeline parallelism introduces a **bubble fraction** that depends on the ratio of pipeline stages to gradient accumulation steps:

$$
\text{bubble\_fraction}_{\text{1F1B}} = \frac{\text{pp} - 1}{\text{gas} + \text{pp} - 1}
$$

To keep bubble overhead below 5%, the constraint is:

$$
\text{gas} \geq 19 \times (\text{pp} - 1) \quad \text{(for 5\% bubble)}
$$

This means that PP **prefers large gas**, which in turn prefers large GBS. If GBS is fixed and gas is too small for the chosen PP degree, either:

- Reduce PP (allocate GPUs differently).
- Use interleaved pipeline schedules (e.g., Megatron-LM's interleaved 1F1B with virtual stages), which reduce bubble to:

$$
\text{bubble\_fraction}_{\text{interleaved}} = \frac{\text{pp} - 1}{\text{gas} \times v + \text{pp} - 1}
$$

where $v$ is the number of virtual pipeline stages (model chunks) per physical stage.

- Use zero-bubble pipeline schedules (ZB-H1/ZB-H2) that interleave weight gradient computation into the bubble, achieving near-zero idle time.

---

## 5. Step 3 — Optimizing Training Throughput

With a memory-feasible configuration that achieves the target GBS, the engineer now optimizes for maximum **tokens per second per GPU** (equivalently, MFU) through controlled experimentation.

### 5.1 Experimental Optimization Strategy

There is **no closed-form solution** for the optimal configuration. The search space includes continuous and discrete variables whose interactions depend on hardware-specific constants (kernel efficiency, collective latency, memory allocator behavior). The systematic approach is:

**Phase 1: Establish Baseline**
- Run the Step 1/2 configuration for 50–100 steps.
- Record: step time, MFU, memory utilization, communication time, bubble fraction.
- Profile with Nsight Systems / PyTorch Profiler to decompose step time.

**Phase 2: Explore TP Scaling**
- Increase TP from current value toward node size (e.g., TP=2 → TP=4 → TP=8).
- Each TP increase reduces per-GPU parameter count, gradient size, and activation memory.
- Each TP increase adds 4 × num_layers collectives per step (all-gather/reduce-scatter in attention and MLP).
- **Measure:** Does the memory savings (enabling larger mbs or less recomputation) offset the communication cost?

**Phase 3: Explore DP with ZeRO Stages**
- If TP=8 leaves residual memory, try DP with ZeRO-1 (shard optimizer only) instead of ZeRO-3 (shard everything).
- ZeRO-1 has lower communication volume than ZeRO-3 (one all-gather at step end vs. all-gather per layer per forward+backward).
- **Trade-off:** ZeRO-3 frees more memory but at higher communication cost.

**Phase 4: Introduce or Adjust PP**
- PP is beneficial when DP communication (gradient all-reduce or reduce-scatter) saturates the inter-node bandwidth.
- PP replaces gradient collectives with point-to-point send/recv of activations, which use a fraction of the bandwidth.
- **Cost:** Pipeline bubble. Use interleaved schedule to mitigate.

**Phase 5: Tune Micro-Batch Size**
- Larger mbs → higher arithmetic intensity → better GPU utilization.
- Larger mbs → larger activation memory → may require more recomputation.
- Larger mbs with fixed GBS → fewer gas → larger pipeline bubble.
- **Optimal mbs** is typically the largest value that fits in memory with acceptable recomputation overhead.

**Phase 6: Enable Compute-Communication Overlap**
- DP gradient synchronization overlapped with backward pass of subsequent microbatches.
- TP collectives overlapped with subsequent TP regions (attention → MLP transition).
- PP send/recv overlapped with compute of non-dependent microbatches.
- ZeRO-3 all-gather for next layer overlapped with current layer's compute.

**Pseudocode: Throughput Optimization Search**

```
ALGORITHM: OptimizeThroughput
INPUT: base_config, model_config, cluster_topology, target_gbs
OUTPUT: optimized_config, performance_report

1. baseline ← ProfileTraining(base_config, num_steps=100)
2. candidates ← [base_config]

3. // Phase 1: TP scaling
   FOR tp IN [2, 4, 8] WHERE tp ≤ cluster_topology.gpus_per_node:
     config ← DeriveConfig(model_config, tp, target_gbs, cluster_topology)
     IF CheckMemoryFeasibility(config):
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

4. // Phase 2: ZeRO stage exploration
   FOR zero_stage IN [0, 1, 2, 3]:
     config ← best_tp_config WITH zero_stage
     IF CheckMemoryFeasibility(config):
       // Exploit freed memory to increase mbs
       max_mbs ← BinarySearchMaxMBS(config)
       config.mbs ← max_mbs
       config.gas ← target_gbs / (config.mbs × config.dp × config.seq_len)
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

5. // Phase 3: PP introduction
   FOR pp IN [2, 4, 8, 16]:
     config ← DeriveConfigWithPP(model_config, best_tp, pp, target_gbs)
     IF CheckMemoryFeasibility(config):
       // Ensure bubble fraction < 10%
       IF BubbleFraction(config) > 0.10: TRY interleaved schedule
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

6. // Phase 4: mbs tuning for top-3 candidates
   FOR config IN TopK(candidates, k=3, sort_by=tokens_per_sec):
     FOR mbs IN [1, 2, 4, 8, 16]:
       adjusted ← config WITH mbs, gas adjusted for target_gbs
       IF CheckMemoryFeasibility(adjusted):
         result ← ProfileTraining(adjusted, num_steps=50)
         candidates.APPEND((adjusted, result))

7. // Phase 5: Overlap tuning
   best_config ← TopK(candidates, k=1)
   FOR overlap_mode IN [none, dp_overlap, tp_overlap, full_overlap]:
     result ← ProfileTraining(best_config WITH overlap_mode, num_steps=50)
     IF result.mfu > best_result.mfu:
       best_config ← best_config WITH overlap_mode

8. RETURN best_config, GenerateReport(candidates)
```

### 5.2 Micro-Batch Size and Arithmetic Intensity

The micro-batch size directly controls the **arithmetic intensity** of GEMM operations:

$$
\text{Arithmetic Intensity} = \frac{2 \times M \times N \times K}{2 \times (M \times K + K \times N + M \times N)} \approx \frac{M \times N \times K}{M \times K + K \times N + M \times N}
$$

For a linear layer with input $(b \times s, h)$ and weight $(h, 4h)$:

- $M = b \times s / \text{tp}$, $N = 4h / \text{tp}$, $K = h$
- Increasing $b$ (mbs) increases $M$, pushing the operation deeper into the compute-bound regime.

On H100 with 3.35 TB/s HBM bandwidth and 989 TFLOPS BF16 peak, the compute-memory crossover occurs at arithmetic intensity $\approx 295$ ops/byte. For typical hidden sizes (≥4096), this is achieved at $b \times s \geq 2048$, which is almost always satisfied for seq_len ≥ 2048 even at mbs=1.

However, **small mbs still hurts** due to:
- Kernel launch overhead amortization
- Reduced opportunities for CUDA graph capture (graph capture requires fixed shapes)
- Lower occupancy in attention kernels (FlashAttention efficiency degrades below certain sequence/batch thresholds)

---

## 6. Parallelization Strategy Reference Matrix

The following table provides a comprehensive, production-grade reference for each parallelism strategy. Each row details the impact on batch size, memory, compute, communication pattern, and overlap characteristics.

### 6.1 Complete Strategy Comparison

| Strategy | Batch Size Effect | Memory Reduction | Compute Reduction | Communication Pattern | Compute/Communication Overlap Condition |
|----------|------------------|-------------------|--------------------|-----------------------|----------------------------------------|
| **Data Parallelism (DP)** | GBS scales linearly with DP | Can reduce mbs by increasing DP → reduces activation memory | Can reduce mbs by increasing DP → reduces per-GPU compute | **Backward:** all-reduce of gradients in BF16 | Overlapped with microbatch backward pass. Overlap ratio: $(DP-1) \times N_{\text{params}} \times \text{peak\_flops} / (2 \times \text{peak\_bw} \times N_{\text{tokens}} \times DP)$ |
| **DP + ZeRO-1** | GBS scales with DP | model\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Backward:** all-reduce of gradients in BF16. **Step end:** all-gather of model\_fp32 | Same as DP; additional all-gather at step boundary (non-overlapped but amortized) |
| **DP + ZeRO-2** | GBS scales with DP | model\_fp32 / DP, grads\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Backward:** reduce-scatter of gradients in BF16. **Step end:** all-gather of model\_fp32 | Overlapped with backward: $(DP-1) \times N_{\text{params}} \times \text{peak\_flops} / (4 \times \text{peak\_bw} \times N_{\text{tokens}} \times DP)$. Improvement over DP because reduce-scatter has half the volume of all-reduce. |
| **DP + ZeRO-3 (FSDP)** | GBS scales with DP | model\_bf16 / DP, model\_fp32 / DP, grads\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Per layer (×num\_layers):** Forward: all-gather model\_fp32. Backward: all-gather model\_fp32 + reduce-scatter grads\_fp32 | Overlapped with next layer's forward/backward: $(DP-1) \times \text{peak\_flops} / (2 \times s \times \text{mbs} \times \text{peak\_bw})$ |
| **Tensor Parallelism (TP)** | No effect on GBS | model\_bf16 / TP, model\_fp32 / TP, grads\_fp32 / TP, optimstates / TP, activations / TP | Compute per GPU reduces by 1/TP (weight FLOPS) | **Per TP region (×4 × num\_layers):** Forward: all-reduce or all-gather of activations. Backward: all-reduce or reduce-scatter of gradients. | Overlapped with next TP region (attention↔MLP↔LayerNorm): $(TP-1) \times \text{peak\_flops} / (24 \times h \times \text{peak\_bw})$ |
| **Pipeline Parallelism (1F1B)** | Prefers large gas to minimize bubble | model\_bf16 / PP, model\_fp32 / PP, grads\_fp32 / PP, optimstates / PP | Compute per GPU reduces by 1/PP (fewer layers) | **Per microbatch (×gas):** Forward: recv + send activation tensors. Backward: recv + send gradient tensors. | Overlapped with next microbatch's compute: $PP \times \text{peak\_flops} / (32 \times h \times N_{\text{layers}} \times \text{peak\_bw})$ |
| **Context Parallelism (CP)** | Prefers large seq\_len for efficient overlap | activations / CP (sequence dimension split) | Attention FLOPS per GPU reduce (shorter local sequence) | **Per layer (×(CP-1) × num\_layers):** Forward: ring send/recv of KV blocks. Backward: ring send/recv of gradient blocks. | Overlapped with attention computation (Ring Attention): communication volume $= (CP-1) \times h \times (L/CP) \times H_{kv} \times (n_k + n_v) \times 2$ bytes per direction |
| **Expert Parallelism (EP)** | Batch size scales with EP (each expert sees fewer tokens) | expert\_params / EP | expert\_compute / EP | **Per MoE layer (×num\_moe\_layers):** Forward: all-to-all dispatch of tokens. Backward: all-to-all of gradients. | Overlapped with MoE block computation: $(EP-1) \times \text{peak\_flops} / (12 \times N_{\text{experts}} \times h \times \text{peak\_bw})$ |

### 6.2 Communication Volume Formulae

For rigorous capacity planning, the per-step communication volume per GPU for each strategy:

| Strategy | Per-Step Volume Per GPU | Direction |
|----------|------------------------|-----------|
| **DP (all-reduce)** | $2 \times (DP-1)/DP \times 2 \times N_{\text{params}}$ bytes | Bidirectional (ring) |
| **ZeRO-2 (reduce-scatter + all-gather)** | $(DP-1)/DP \times 2 \times N_{\text{params}} + (DP-1)/DP \times 4 \times N_{\text{params}}$ | Bidirectional |
| **ZeRO-3 (per-layer all-gather ×2 + reduce-scatter)** | $3 \times N_{\text{layers}} \times (DP-1)/DP \times 4 \times N_{\text{params\_per\_layer}}$ | Bidirectional |
| **TP (per-layer all-reduce ×4)** | $4 \times N_{\text{layers}} \times 2 \times (TP-1)/TP \times 2 \times s \times b \times h / TP$ | Intra-node |
| **PP (per-microbatch send/recv ×2)** | $2 \times \text{gas} \times 2 \times s \times b \times h$ bytes | Point-to-point |
| **CP (ring KV exchange)** | $(CP-1) \times N_{\text{layers}} \times 2 \times s/CP \times b \times 2 \times h_{kv} \times 2$ bytes | Ring |
| **EP (all-to-all)** | $N_{\text{moe\_layers}} \times 2 \times \text{tokens\_dispatched} \times h \times 2$ bytes | All-to-all |

---

## 7. Framework-Specific Implementation Guidance

### 7.1 Megatron-LM / Megatron-Core

Megatron-Core is the reference implementation for combined TP+PP+DP+CP+EP training. It provides:

- **Column- and row-parallel linear layers** for TP, with fused kernels for GEMMs.
- **Interleaved 1F1B pipeline schedule** with configurable virtual pipeline stages.
- **Distributed optimizer** (equivalent to ZeRO-1) that shards optimizer states across DP ranks.
- **Context parallelism** via Ring Attention or Ulysses (sequence-parallel attention).
- **Expert parallelism** with configurable expert routing, token dropping, and load balancing.
- **Sequence parallelism (SP)**: partitions LayerNorm and Dropout activations along the sequence dimension across TP ranks, reducing activation memory by 1/TP for these operations.

**Launch Configuration (torchrun):**

```
ALGORITHM: LaunchMegatronTraining
INPUT: model_config, parallelism_config, cluster_config, checkpoint_path
OUTPUT: launched training process group

1. // Compute world size
   world_size ← tp × pp × dp × cp × ep_if_orthogonal
   num_nodes ← world_size / cluster_config.gpus_per_node
   
2. // Generate torchrun command
   launch_cmd ← "torchrun"
     + " --nproc_per_node=" + cluster_config.gpus_per_node
     + " --nnodes=" + num_nodes
     + " --node_rank=$NODE_RANK"
     + " --master_addr=$MASTER_ADDR"
     + " --master_port=$MASTER_PORT"
     + " --rdzv_backend=c10d"
     + " --rdzv_endpoint=$MASTER_ADDR:$MASTER_PORT"

3. // Megatron arguments
   megatron_args ← {
     "--tensor-model-parallel-size": tp,
     "--pipeline-model-parallel-size": pp,
     "--context-parallel-size": cp,
     "--num-layers-per-virtual-pipeline-stage": virtual_chunks,
     "--sequence-parallel": True,     // SP coupled with TP
     "--use-distributed-optimizer": True,  // ZeRO-1 on DP dim
     "--overlap-grad-reduce": True,
     "--overlap-param-gather": True,
     "--use-flash-attn": True,
     "--recompute-granularity": "selective",  // or "full"
     "--recompute-method": "uniform",
     "--bf16": True,
     "--micro-batch-size": mbs,
     "--global-batch-size": target_gbs,
     "--accumulate-allreduce-grads-in-fp32": True,
     "--load": checkpoint_path,
     "--save": checkpoint_save_path,
     "--save-interval": save_interval,
   }
   
4. // Expert parallelism (MoE models)
   IF model_config.is_moe:
     megatron_args += {
       "--num-experts": model_config.num_experts,
       "--expert-model-parallel-size": ep,
       "--moe-router-topk": model_config.top_k,
       "--moe-token-dispatcher-type": "alltoall",
       "--moe-aux-loss-coeff": 0.01,
     }

5. // Process group initialization
   // Megatron-Core creates nested process groups:
   //   - TP group: ranks sharing TP dimension (size=tp)
   //   - PP group: ranks sharing PP dimension (size=pp)
   //   - DP group: ranks sharing DP dimension (size=dp)
   //   - CP group: ranks sharing CP dimension (size=cp)
   //   - EP group: subset of DP ranks for expert sharding (size=ep)
   // Placement: TP on innermost (consecutive GPU IDs within node)
   //            PP across nodes, DP across remaining

6. EXECUTE launch_cmd WITH megatron_args
```

**Checkpoint Interoperability:**

Megatron-Core checkpoints store sharded state dictionaries indexed by TP rank, PP stage, and DP rank (for distributed optimizer). Converting between parallelism configurations requires:

```
ALGORITHM: ReshardCheckpoint
INPUT: source_ckpt_path, source_config {tp_s, pp_s, dp_s},
       target_config {tp_t, pp_t, dp_t}
OUTPUT: resharded checkpoint at target_ckpt_path

1. // Load all shards into a single consolidated state
   full_state ← {}
   FOR tp_rank IN [0, tp_s):
     FOR pp_rank IN [0, pp_s):
       shard ← LoadShard(source_ckpt_path, tp_rank, pp_rank)
       // Concatenate TP-sharded weight matrices along partition dim
       // Assign layers to PP stages based on pp_rank
       MergeInto(full_state, shard, tp_rank, pp_rank, tp_s, pp_s)

2. // Re-shard to target configuration
   FOR tp_rank IN [0, tp_t):
     FOR pp_rank IN [0, pp_t):
       target_shard ← ExtractShard(full_state, tp_rank, pp_rank, tp_t, pp_t)
       SaveShard(target_ckpt_path, target_shard, tp_rank, pp_rank)

3. // Validate
   FOR each parameter IN full_state:
     reconstructed ← ReconstructFromShards(target_ckpt_path, tp_t, pp_t)
     ASSERT AllClose(full_state[parameter], reconstructed, atol=0)
```

### 7.2 DeepSpeed

DeepSpeed provides ZeRO-1/2/3, pipeline parallelism (via `PipelineEngine`), and integration with Megatron-LM. Key configuration elements:

**ZeRO Stage Selection Criteria:**

| ZeRO Stage | What is Sharded | Communication Pattern | When to Use |
|------------|----------------|----------------------|-------------|
| **Stage 1** | Optimizer states | All-reduce gradients + all-gather FP32 params at step end | Models that fit in memory with TP/PP; want to reduce optimizer memory across DP |
| **Stage 2** | Optimizer states + gradients | Reduce-scatter gradients + all-gather FP32 params at step end | Need more memory than Stage 1; communication is only marginally higher |
| **Stage 3** | Optimizer states + gradients + parameters | All-gather params per layer (fwd+bwd) + reduce-scatter gradients | Model does not fit even with Stage 2; accept higher communication cost |

**DeepSpeed Configuration Template (Pseudocode):**

```
ALGORITHM: GenerateDeepSpeedConfig
INPUT: model_size, gpu_memory, num_gpus, target_gbs, seq_len
OUTPUT: ds_config (JSON-compatible dictionary)

1. // Determine ZeRO stage
   mem_needed_z0 ← EstimateMemory(model_size, zero_stage=0, dp=num_gpus)
   IF mem_needed_z0 ≤ gpu_memory × 0.85:
     zero_stage ← 0
   ELSE:
     mem_needed_z1 ← EstimateMemory(model_size, zero_stage=1, dp=num_gpus)
     IF mem_needed_z1 ≤ gpu_memory × 0.85:
       zero_stage ← 1
     ELSE:
       mem_needed_z2 ← EstimateMemory(model_size, zero_stage=2, dp=num_gpus)
       IF mem_needed_z2 ≤ gpu_memory × 0.85:
         zero_stage ← 2
       ELSE:
         zero_stage ← 3

2. ds_config ← {
     "train_micro_batch_size_per_gpu": mbs,
     "gradient_accumulation_steps": gas,
     "steps_per_print": 10,
     "gradient_clipping": 1.0,
     "bf16": {"enabled": True},
     "zero_optimization": {
       "stage": zero_stage,
       "overlap_comm": True,
       "contiguous_gradients": True,
       "reduce_bucket_size": 5e8,    // 500M elements ≈ 1 GB BF16
       "allgather_bucket_size": 5e8,
       "reduce_scatter": True,       // Stage 2+
       "allgather_partitions": True, // Stage 3
       "stage3_prefetch_bucket_size": 5e7,
       "stage3_param_persistence_threshold": 1e6,
       "stage3_max_live_parameters": 1e9,
       "stage3_max_reuse_distance": 1e9,
     },
     "activation_checkpointing": {
       "partition_activations": True,   // Partition across TP ranks
       "contiguous_memory_optimization": True,
       "cpu_checkpointing": False,
     },
     "zero_allow_untested_optimizer": True,
     "wall_clock_breakdown": True,
   }

3. // Offload (GPU-poor scenario)
   IF zero_stage == 3 AND model_size > EstimateMaxModelForGPUMemory():
     ds_config["zero_optimization"]["offload_optimizer"] ← {
       "device": "cpu",
       "pin_memory": True,
       "fast_init": True,
     }
     ds_config["zero_optimization"]["offload_param"] ← {
       "device": "cpu",
       "pin_memory": True,
     }

4. RETURN ds_config
```

### 7.3 PyTorch FSDP / DTensor

PyTorch's Fully Sharded Data Parallel (FSDP2, based on DTensor) provides ZeRO-3 semantics with native PyTorch integration:

- **DTensor** provides a device-mesh abstraction that maps logical parallelism dimensions (DP, TP) to physical GPU topologies.
- **FSDP2** uses DTensor to shard parameters along the DP dimension with `Shard(0)` placement.
- **TP** is achieved via DTensor's `Replicate` and `Shard` placements on different mesh dimensions.

**Advantages:** Native PyTorch, composable with `torch.compile`, CUDA Graphs, and custom kernels without framework lock-in.

**Limitations:** Less mature pipeline parallelism support compared to Megatron-Core. No built-in interleaved pipeline schedules. Expert parallelism requires manual implementation.

**When to choose FSDP/DTensor over Megatron-Core or DeepSpeed:**
- When portability and framework simplicity are priorities.
- For fine-tuning (SFT, DPO, PPO) where PP is not needed and DP+TP suffice.
- When integrating with custom training loops (RLHF, GRPO) that do not fit Megatron's training harness.
- When `torch.compile` is required for kernel fusion (Megatron-Core kernels are hand-fused and bypass compile).

### 7.4 torchrun Orchestration

`torchrun` (replacement for `torch.distributed.launch`) handles elastic and fault-tolerant distributed process management:

```
ALGORITHM: OrchestrateTorchrun
INPUT: script_path, num_nodes, gpus_per_node, rdzv_backend,
       rdzv_endpoint, max_restarts, env_vars
OUTPUT: running distributed training

1. // Preflight checks (run on each node before launch)
   FOREACH node IN cluster:
     VerifyGPUHealth(node)          // nvidia-smi / rocm-smi checks
     VerifyNICBandwidth(node)       // ib_write_bw / perftest
     VerifyNVLinkTopology(node)     // nvidia-smi topo -m
     VerifyNCCLVersion(node)        // must match across all nodes
     VerifyContainerConsistency(node) // driver, CUDA/ROCm, PyTorch versions
     RunNCCLBandwidthTest(node)     // all_reduce_perf / all_gather_perf

2. // Environment setup
   env ← {
     "NCCL_IB_DISABLE": "0",
     "NCCL_NET_GDR_LEVEL": "5",      // GPUDirect RDMA
     "NCCL_IB_HCA": auto_detect(),    // mlx5_0:1,mlx5_1:1,...
     "NCCL_SOCKET_IFNAME": "eth0",
     "NCCL_DEBUG": "WARN",            // "INFO" for debugging
     "NCCL_ALGO": "Ring,Tree",
     "NCCL_PROTO": "Simple,LL128",
     "CUDA_DEVICE_MAX_CONNECTIONS": "1",  // For Megatron overlap
     "OMP_NUM_THREADS": "4",
     "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
     "NCCL_CROSS_NIC": "1",
   }
   env.UPDATE(env_vars)

3. // Launch command
   cmd ← "torchrun"
     + " --nnodes=" + num_nodes
     + " --nproc_per_node=" + gpus_per_node
     + " --rdzv_backend=" + rdzv_backend  // "c10d" or "etcd"
     + " --rdzv_endpoint=" + rdzv_endpoint
     + " --max_restarts=" + max_restarts
     + " --monitor_interval=5"
     + " " + script_path
     + " " + training_args

4. // Launch with health monitoring
   FOREACH node IN cluster:
     SSH node: "export ENV; exec cmd" &
   
   MonitorLoop:
     WHILE training_active:
       CheckHeartbeats(all_nodes)
       CheckGPUUtilization(all_nodes)   // Alert if < 80%
       CheckMemoryUsage(all_nodes)       // Alert if > 95% HBM
       CheckNetworkCounters(all_nodes)   // Alert on error counters
       IF node_failure_detected:
         TriggerCheckpointSave()
         WaitForElasticRestart(max_restarts)
       SLEEP(monitor_interval)
```

---

## 8. Kernel-Level Optimization and Numerical Stability

### 8.1 FlashAttention and Fused Kernels

**FlashAttention** (Dao, 2022; Dao, 2023) eliminates the $O(s^2)$ memory requirement of standard attention by computing attention in tiles, never materializing the full $s \times s$ attention matrix in HBM:

- **Memory:** $O(s)$ instead of $O(s^2)$
- **IO complexity:** $O(s^2 \cdot d / M)$ where $M$ is SRAM size, versus $O(s^2 \cdot d + s^2)$ for standard attention
- **Backward pass:** Recomputes attention weights from Q, K, V stored in HBM using the online softmax statistics (row-max and row-sum-exp saved during forward)

**When FlashAttention provides the largest benefit:**
- Sequence lengths ≥ 2048
- The memory savings enable larger micro-batch sizes, which improves overall throughput
- With context parallelism, each CP rank runs FlashAttention on sequence length $s/\text{CP}$

**Fused Kernels Critical for Performance:**

| Kernel | What It Fuses | Impact |
|--------|--------------|--------|
| **Fused RMSNorm** | Normalization + residual add | Eliminates extra HBM read/write of residual |
| **Fused Softmax** | Temperature scaling + masking + softmax | Saves one $O(s^2)$ HBM pass |
| **Fused RoPE** | Rotary position encoding into Q/K projection | Avoids separate RoPE kernel launch |
| **Fused SwiGLU MLP** | Gate projection + activation + up projection | Reduces HBM traffic by 33% |
| **Fused Cross-Entropy** | Log-softmax + NLL loss | Critical for large vocabularies; avoids materializing logits |

### 8.2 Triton Kernels

Triton enables writing custom GPU kernels in Python with auto-tuning, portable across NVIDIA (CUDA) and AMD (ROCm/HIP):

```
ALGORITHM: TritonFusedRMSNorm
INPUT: x[M, N] (activation tensor), w[N] (weight), epsilon
OUTPUT: y[M, N] (normalized tensor)

// Each Triton program instance handles one row
1. row_idx ← program_id(axis=0)
2. col_offsets ← arange(0, BLOCK_SIZE)
3. mask ← col_offsets < N

4. // Load row from HBM (single pass)
   x_row ← load(x[row_idx, col_offsets], mask=mask, other=0.0)

5. // Compute variance in SRAM
   variance ← sum(x_row * x_row) / N
   inv_rms ← rsqrt(variance + epsilon)

6. // Normalize and scale
   w_vals ← load(w[col_offsets], mask=mask)
   y_row ← x_row * inv_rms * w_vals

7. // Store result (single pass)
   store(y[row_idx, col_offsets], y_row, mask=mask)

// Auto-tune BLOCK_SIZE over [256, 512, 1024, 2048, 4096]
// Measure: time, register usage, occupancy
// Select BLOCK_SIZE that maximizes throughput for given N
```

**When to use Triton vs. CUDA/HIP:**
- Triton for **element-wise fusions, reductions, normalization** kernels where the programming model maps cleanly.
- CUDA/HIP for **GEMM wrappers, collective-aware kernels, persistent kernels** that require fine-grained thread-block scheduling or shared memory choreography beyond Triton's abstraction.

### 8.3 FP8 and Mixed-Precision Training

H100, B200, and MI300X support FP8 (E4M3 for forward, E5M2 for backward):

- **FP8 GEMMs:** 2× throughput vs. BF16 on H100 Tensor Cores (1978 vs. 989 TFLOPS)
- **Requires per-tensor or per-block scaling** to prevent overflow/underflow
- **MXFP8 (Microscaling FP8):** Block-level scaling with shared exponents (block size 32), providing finer granularity than per-tensor scaling
- **MXFP6/MXFP4:** Available on B200; further reduces memory/bandwidth but requires careful validation

**Numerical Stability Checklist for FP8:**

```
ALGORITHM: ValidateFP8Parity
INPUT: model, dataset, bf16_baseline_loss_curve, fp8_config
OUTPUT: parity_report

1. // Run BF16 baseline for N steps
   bf16_losses ← Train(model, dataset, precision="bf16", steps=N)

2. // Run FP8 with identical seed, data order, hyperparameters
   fp8_losses ← Train(model, dataset, precision="fp8",
                       fp8_config=fp8_config, steps=N)

3. // Validate parity
   FOR step IN [0, N):
     relative_diff ← |bf16_losses[step] - fp8_losses[step]| / bf16_losses[step]
     ASSERT relative_diff < 0.02  // 2% relative tolerance early in training
   
   // Final loss parity (most important)
   final_diff ← |bf16_losses[N-1] - fp8_losses[N-1]| / bf16_losses[N-1]
   ASSERT final_diff < 0.005  // 0.5% at convergence

4. // Monitor overflow/underflow statistics
   ASSERT fp8_overflow_rate < 0.001  // < 0.1% of tensors
   ASSERT fp8_underflow_rate < 0.01  // < 1% of tensors

5. // Gradient norm tracking
   FOR step IN [0, N):
     bf16_gnorm ← GradientNorm(bf16_run, step)
     fp8_gnorm ← GradientNorm(fp8_run, step)
     ASSERT |bf16_gnorm - fp8_gnorm| / bf16_gnorm < 0.05

6. RETURN ParityReport(bf16_losses, fp8_losses, overflow_stats)
```

**Loss Scaling for FP16 Training (Still Relevant for AMD MI300X without native BF16 efficiency):**

- Dynamic loss scaling: start at scale $2^{16}$, double every 2000 steps if no overflow, halve on overflow.
- Gradient clipping applied **after unscaling** (divide by loss scale), **before** optimizer step.
- BF16 does not require loss scaling (exponent range matches FP32), which is why BF16 is universally preferred when hardware supports it at equivalent throughput.

### 8.4 Compute-Communication Overlap

The single most impactful throughput optimization after kernel selection is **overlapping communication with computation**:

**DP Gradient Overlap:**

```
ALGORITHM: OverlapDPGradientSync
INPUT: model_layers, dp_process_group, bucket_size_mb

1. // During backward pass:
   gradient_buckets ← CreateBuckets(model.parameters(), bucket_size_mb)
   
   FOR layer IN REVERSE(model_layers):
     // Compute gradients for this layer
     ComputeBackward(layer)
     
     // Check if any bucket is full
     FOR bucket IN gradient_buckets:
       IF bucket.is_ready():
         // Launch async all-reduce (DP) or reduce-scatter (ZeRO-2)
         bucket.async_comm_handle ← AllReduceAsync(
           bucket.gradient_tensor,
           group=dp_process_group,
           op=AVG
         )

2. // Wait for all communication to complete
   FOR bucket IN gradient_buckets:
     bucket.async_comm_handle.wait()
```

**TP-Comm Overlap (Megatron-Core approach):**

In Megatron-Core, each Transformer layer has 4 TP communication points:
1. Forward attention: all-reduce (or reduce-scatter with SP)
2. Forward MLP: all-reduce (or reduce-scatter with SP)
3. Backward MLP: all-reduce (or all-gather with SP)
4. Backward attention: all-reduce (or all-gather with SP)

With `CUDA_DEVICE_MAX_CONNECTIONS=1`, Megatron-Core ensures that the GEMM kernel and the NCCL kernel execute on different CUDA streams that can overlap on the GPU's compute and copy engines.

**ZeRO-3 / FSDP Prefetch Overlap:**

```
ALGORITHM: FSDP_PrefetchOverlap
INPUT: model_layers, dp_process_group

1. FOR i IN range(len(model_layers)):
     // Prefetch next layer's parameters while computing current layer
     IF i + 1 < len(model_layers):
       PrefetchAllGather(model_layers[i+1].parameters(), group=dp_process_group)
     
     // Compute forward for current layer (parameters already gathered)
     output ← model_layers[i].forward(input)
     
     // Free current layer's full parameters (keep only shard)
     FreeFull Parameters(model_layers[i])
     
     input ← output
```

---

## 9. Data Pipeline Engineering for Distributed Training

### 9.1 End-to-End Pipeline Architecture

The data pipeline is a **distributed system** that must deliver tokenized, packed, shuffled batches to every GPU without becoming a throughput bottleneck or introducing statistical bias.

```
ALGORITHM: DistributedDataPipeline
INPUT: raw_corpus, tokenizer_config, training_config
OUTPUT: streaming dataloader yielding (input_ids, labels, attention_mask) per GPU

PHASE 1: Offline Preprocessing
  1.1. Ingest raw documents from heterogeneous sources (CommonCrawl, books, code, etc.)
  1.2. Filter: language ID, perplexity scoring, deduplication (MinHash → LSH),
       safety filtering, quality heuristics
  1.3. Deduplicate:
       - Document-level: MinHash with 128 permutations, Jaccard threshold 0.8
       - Substring-level: suffix array for exact n-gram dedup (n ≥ 50 tokens)
  1.4. Tokenize with trained tokenizer (BPE/SentencePiece/Unigram)
       - Tokenizer trained on representative sample of final corpus
       - Vocabulary size: 32K–128K (trade-off: larger vocab → shorter sequences
         → less compute; but embedding table grows)
  1.5. Pack sequences:
       - Concatenate documents with <EOS> separator
       - Pack into fixed-length sequences (seq_len) to maximize GPU utilization
       - Track document boundaries for attention masking (no cross-document attention)
  1.6. Shard into N binary files (e.g., .bin + .idx for Megatron format,
       or Parquet/WebDataset/MosaicML Streaming format)
       - Each shard: ~1 GB, containing ~500K sequences of length 4096
       - Write deterministic shard assignment based on document hash

PHASE 2: Online Loading (during training)
  2.1. Each DP rank opens its assigned shards (disjoint across DP ranks)
  2.2. Memory-map (.mmap) shard files for zero-copy access
  2.3. Shuffle:
       - Epoch-level: shuffle shard order with seed = base_seed + epoch
       - Intra-shard: shuffle sequence indices within each shard
       - This provides pseudo-random global order without full-corpus shuffle
  2.4. Prefetch: async worker threads load next batch while GPU processes current
       - num_workers: 4–8 per GPU (CPU-bound tokenization/packing already done offline)
       - prefetch_factor: 2–4 batches ahead
  2.5. Curriculum mixing:
       - Domain weights (e.g., 50% web, 20% code, 15% books, 10% math, 5% multilingual)
       - Weights may change during training (curriculum schedule)
       - Implemented as weighted sampling across domain-specific shard pools
  2.6. Yield batches of shape (mbs, seq_len) to each GPU

PHASE 3: Deterministic Resume
  3.1. Save dataloader state at each checkpoint:
       - Current shard index, position within shard, epoch counter, RNG state
       - Per-domain sample counts (for curriculum tracking)
  3.2. On resume: restore exact dataloader state → bitwise-identical training
```

### 9.2 Sequence Packing

Naive padding wastes significant compute. Sequence packing concatenates variable-length documents into fixed-length training sequences:

**Without packing (padding to max length):**
- Average document length: 512 tokens, training seq_len: 4096
- Waste: $(4096 - 512) / 4096 = 87.5\%$ of tokens are padding → 87.5% wasted FLOPS

**With packing:**
- Pack ~8 documents per sequence on average
- No wasted tokens (except the last partial sequence per shard)
- Must use attention masking to prevent cross-document attention leakage

### 9.3 Data Lineage and Reproducibility

For regulatory compliance, scientific reproducibility, and debugging:

- Every training run records: exact shard list, shuffle seed, domain weights, tokenizer version, preprocessing pipeline version hash.
- Data version control: content-addressable storage for shards (hash-based naming).
- Reprocessing any shard from raw data must produce identical output (deterministic preprocessing).

---

## 10. Fault Tolerance, Automation, and Production Resilience

### 10.1 Failure Modes in Large-Scale Training

| Failure Mode | Frequency (per 1000 GPU-hours) | Impact | Mitigation |
|-------------|-------------------------------|--------|------------|
| GPU hardware fault (ECC error, XID error) | 0.1–0.5 | Single GPU, kills rank | Elastic restart, spare node pool |
| InfiniBand link flap | 0.05–0.2 | Straggler or timeout | NCCL timeout detection, link health monitoring |
| NVLink error | 0.01–0.05 | Node-level failure | Node replacement, TP group reformation |
| Storage I/O stall | 0.1–1.0 | Dataloader starvation | Local SSD caching, async prefetch, retry logic |
| NCCL timeout/hang | 0.05–0.3 | Full training hang | Watchdog timer, TORCH_NCCL_ASYNC_ERROR_HANDLING |
| OOM (unexpected memory spike) | 0.01–0.1 | Rank crash | Memory monitoring, conservative headroom |
| Software bug (NaN loss, gradient explosion) | Varies | Wasted compute | Loss spike detection, auto-rollback to last good checkpoint |

### 10.2 Resilient Training Automation

```
ALGORITHM: ResilientTrainingLoop
INPUT: training_config, cluster_config, max_failures
OUTPUT: trained model checkpoint

1. failure_count ← 0
2. last_good_checkpoint ← FindLatestValidCheckpoint(training_config.save_path)

3. WHILE NOT training_complete AND failure_count < max_failures:
   TRY:
     // Preflight
     healthy_nodes ← RunPreflightChecks(cluster_config)
     IF |healthy_nodes| < RequiredNodes(training_config):
       WaitForNodeRepair(timeout=30min)
       CONTINUE
     
     // Generate topology-aware config
     config ← SynthesizeConfig(training_config, healthy_nodes)
     
     // Launch training
     LaunchTorchrun(config, resume_from=last_good_checkpoint)
     
     // Training runs...
     // Periodic checkpointing every K steps
     // Checkpoint validation: verify tensor shapes, optimizer state completeness
     
     training_complete ← True
     
   CATCH NodeFailure(failed_node):
     failure_count += 1
     LOG("Node failure: " + failed_node + ", attempt " + failure_count)
     
     // Save emergency checkpoint if possible
     TryEmergencyCheckpoint()
     
     // Remove failed node, acquire replacement
     cluster_config.exclude(failed_node)
     replacement ← AcquireSpareNode()
     IF replacement:
       cluster_config.add(replacement)
     
     // Validate last checkpoint
     last_good_checkpoint ← FindLatestValidCheckpoint(training_config.save_path)
     ValidateCheckpoint(last_good_checkpoint)  // Verify all shards, no corruption
     
   CATCH NaNLoss(step):
     LOG("NaN loss detected at step " + step)
     // Rollback to checkpoint before divergence
     last_good_checkpoint ← FindCheckpointBefore(step - rollback_margin)
     // Optionally: reduce learning rate, increase gradient clipping
     
   CATCH Timeout:
     failure_count += 1
     // Likely NCCL hang or straggler
     DiagnoseHang()  // Dump NCCL debug info, check GPU utilization
     KillAllRanks()
     // Restart

4. IF failure_count >= max_failures:
     ALERT("Training failed after max retries")
     RETURN last_good_checkpoint

5. RETURN final_checkpoint
```

### 10.3 Checkpoint Validation

```
ALGORITHM: ValidateCheckpoint
INPUT: checkpoint_path, expected_config {tp, pp, dp, num_params}
OUTPUT: valid (boolean), issues (list)

1. issues ← []

2. // Check all expected shards exist
   FOR tp_rank IN [0, tp):
     FOR pp_rank IN [0, pp):
       shard_path ← checkpoint_path + "/mp_rank_" + tp_rank + "_" + pp_rank
       IF NOT exists(shard_path):
         issues.APPEND("Missing shard: tp=" + tp_rank + " pp=" + pp_rank)
         CONTINUE
       
       // Verify file integrity (checksum)
       IF NOT VerifyChecksum(shard_path):
         issues.APPEND("Corrupt shard: " + shard_path)
         CONTINUE
       
       // Load and verify tensor shapes
       state ← LoadStateDict(shard_path)
       FOR key, tensor IN state.items():
         expected_shape ← ComputeExpectedShape(key, model_config, tp, pp)
         IF tensor.shape != expected_shape:
           issues.APPEND("Shape mismatch: " + key)
         IF HasNaN(tensor) OR HasInf(tensor):
           issues.APPEND("NaN/Inf in: " + key)

3. // Verify optimizer state completeness
   optimizer_state ← LoadOptimizerState(checkpoint_path)
   FOR param_group IN optimizer_state["param_groups"]:
     FOR param_id IN param_group["params"]:
       IF param_id NOT IN optimizer_state["state"]:
         issues.APPEND("Missing optimizer state for param: " + param_id)
       ELSE:
         state = optimizer_state["state"][param_id]
         IF "exp_avg" NOT IN state OR "exp_avg_sq" NOT IN state:
           issues.APPEND("Incomplete Adam state for param: " + param_id)

4. // Verify training metadata
   metadata ← LoadMetadata(checkpoint_path)
   ASSERT metadata["iteration"] > 0
   ASSERT metadata["tokens_seen"] > 0

5. RETURN (len(issues) == 0), issues
```

---

## 11. Profiling, Instrumentation, and Regression Gating

### 11.1 Profiling Tool Selection

| Tool | Target | What It Captures | When to Use |
|------|--------|-----------------|-------------|
| **Nsight Systems** | NVIDIA GPUs | End-to-end timeline: kernels, NCCL ops, CUDA API, CPU activity | Step-time decomposition, overlap analysis, pipeline bubble visualization |
| **Nsight Compute** | NVIDIA GPUs | Per-kernel metrics: occupancy, memory throughput, warp stall reasons | Kernel-level optimization (why is a specific GEMM slow?) |
| **PyTorch Profiler** | Framework-level | Op-level breakdown, memory allocation timeline, FLOPS counting | Quick triage, integration with TensorBoard |
| **rocprof / rocprof-sys** | AMD GPUs | Kernel timelines, HIP API traces, hardware counters | AMD cluster profiling, analogous to Nsight |
| **NCCL/RCCL debug logs** | Collectives | Ring/tree topology, algorithm selection, per-collective timing | Diagnosing communication bottlenecks, topology mismatch |

### 11.2 Step-Time Decomposition

Every training step is decomposed into:

$$
T_{\text{step}} = T_{\text{forward}} + T_{\text{backward}} + T_{\text{optimizer}} + T_{\text{comm}} + T_{\text{bubble}} + T_{\text{dataloader}} + T_{\text{misc}}
$$

where $T_{\text{comm}}$ is the **non-overlapped** portion of communication (the part that extends step time beyond pure compute).

**Methodology:**

```
ALGORITHM: StepTimeDecomposition
INPUT: profiler_trace (Nsight Systems or PyTorch Profiler output)
OUTPUT: decomposition report

1. // Parse kernel timeline
   forward_kernels ← FilterKernels(trace, phase="forward")
   backward_kernels ← FilterKernels(trace, phase="backward")
   nccl_kernels ← FilterKernels(trace, name_contains="nccl" OR "rccl")
   optimizer_kernels ← FilterKernels(trace, phase="optimizer")

2. // Compute pure compute time
   T_forward ← SumDuration(forward_kernels) - OverlapWith(nccl_kernels, forward_kernels)
   T_backward ← SumDuration(backward_kernels) - OverlapWith(nccl_kernels, backward_kernels)
   T_optimizer ← SumDuration(optimizer_kernels)

3. // Compute communication time (non-overlapped only)
   T_comm_total ← SumDuration(nccl_kernels)
   T_comm_overlapped ← OverlapWith(nccl_kernels, forward_kernels + backward_kernels)
   T_comm_exposed ← T_comm_total - T_comm_overlapped

4. // Pipeline bubble (idle GPU time between microbatches)
   T_bubble ← T_step - T_forward - T_backward - T_optimizer - T_comm_exposed - T_dataloader

5. // Dataloader stall (CPU→GPU data transfer wait)
   T_dataloader ← SumDuration(FilterKernels(trace, name="DataLoader"))

6. // Compute MFU
   total_flops ← 6 × N_params × mbs × seq_len × gas
   T_step ← EndTime(trace) - StartTime(trace)
   MFU ← total_flops / (T_step × peak_flops_per_gpu)

7. REPORT:
     "Step time: {T_step:.2f} ms"
     "  Forward compute: {T_forward:.2f} ms ({T_forward/T_step*100:.1f}%)"
     "  Backward compute: {T_backward:.2f} ms ({T_backward/T_step*100:.1f}%)"
     "  Optimizer step: {T_optimizer:.2f} ms ({T_optimizer/T_step*100:.1f}%)"
     "  Communication (exposed): {T_comm_exposed:.2f} ms ({T_comm_exposed/T_step*100:.1f}%)"
     "  Communication (overlapped): {T_comm_overlapped:.2f} ms"
     "  Pipeline bubble: {T_bubble:.2f} ms ({T_bubble/T_step*100:.1f}%)"
     "  Dataloader: {T_dataloader:.2f} ms ({T_dataloader/T_step*100:.1f}%)"
     "  MFU: {MFU*100:.1f}%"
```

### 11.3 Regression Gating

Every configuration change must pass through a regression gate:

```
ALGORITHM: RegressionGate
INPUT: baseline_metrics, candidate_metrics, tolerance_config
OUTPUT: pass/fail, regression_report

1. // Throughput regression
   throughput_change ← (candidate.tokens_per_sec - baseline.tokens_per_sec) /
                        baseline.tokens_per_sec
   IF throughput_change < -tolerance_config.throughput_threshold:  // e.g., -2%
     FAIL("Throughput regression: " + throughput_change)

2. // Memory regression
   memory_change ← (candidate.peak_memory - baseline.peak_memory) /
                    baseline.peak_memory
   IF memory_change > tolerance_config.memory_threshold:  // e.g., +5%
     WARN("Memory increase: " + memory_change)

3. // Loss parity (same data, same seed, N steps)
   loss_divergence ← MaxAbsDiff(candidate.loss_curve, baseline.loss_curve) /
                      Mean(baseline.loss_curve)
   IF loss_divergence > tolerance_config.loss_threshold:  // e.g., 1%
     FAIL("Loss divergence: " + loss_divergence)

4. // Gradient norm stability
   gnorm_cv_change ← |candidate.gnorm_cv - baseline.gnorm_cv|
   IF gnorm_cv_change > tolerance_config.gnorm_threshold:
     WARN("Gradient norm variance change: " + gnorm_cv_change)

5. // Communication overlap ratio
   overlap_change ← candidate.overlap_ratio - baseline.overlap_ratio
   IF overlap_change < -tolerance_config.overlap_threshold:
     WARN("Reduced communication overlap")

6. IF all checks PASS: RETURN PASS
   ELSE: RETURN FAIL with regression_report
```

---

## 12. Cross-Vendor Portability: NVIDIA and AMD

### 12.1 Key Differences

| Aspect | NVIDIA (A100/H100/B200) | AMD (MI300X/MI350) |
|--------|------------------------|-------------------|
| **Interconnect (intra-node)** | NVLink/NVSwitch (900 GB/s H100) | xGMI / Infinity Fabric (896 GB/s MI300X) |
| **Interconnect (inter-node)** | InfiniBand NDR (400 Gb/s) | InfiniBand NDR or RoCE |
| **Collectives library** | NCCL | RCCL (NCCL-compatible API) |
| **Compute precision** | FP8 (E4M3/E5M2), BF16, TF32 | FP8 (E4M3/E5M2), BF16, FP32 (no TF32) |
| **HBM capacity** | 80 GB (A100/H100), 192 GB (B200) | 192 GB (MI300X), 288 GB (MI350) |
| **Kernel ecosystem** | CUDA, cuBLAS, cuDNN, Triton | HIP, hipBLAS, MIOpen, Triton (ROCm backend) |
| **Flash Attention** | Native CUDA (Dao), CK-based (Composable Kernel) | CK-based FlashAttention, Triton-based |
| **Profiling** | Nsight Systems, Nsight Compute | rocprof, rocprof-sys (Omnitrace), omniperf |

### 12.2 Portability Strategy

```
ALGORITHM: CrossVendorConfigSynthesis
INPUT: model_config, target_vendor, cluster_topology
OUTPUT: vendor-optimized training config

1. // Detect vendor
   vendor ← DetectGPUVendor()  // "nvidia" or "amd"

2. // Common config (vendor-agnostic)
   config ← {
     "bf16": True,
     "flash_attention": True,
     "gradient_clipping": 1.0,
     "distributed_backend": "nccl",  // RCCL is API-compatible
   }

3. IF vendor == "nvidia":
     config["env"]["NCCL_NET_GDR_LEVEL"] ← "5"
     config["env"]["NCCL_IB_HCA"] ← AutoDetectIBHCA()
     config["env"]["CUDA_DEVICE_MAX_CONNECTIONS"] ← "1"
     config["flash_attn_impl"] ← "flash_attn"  // Dao's CUDA implementation
     config["fused_kernels"] ← "apex"  // or Megatron-Core fused kernels
     IF gpu_arch ≥ "sm_90":  // H100+
       config["fp8"] ← True
       config["fp8_recipe"] ← "delayed_scaling"

4. IF vendor == "amd":
     config["env"]["NCCL_NET_GDR_LEVEL"] ← "3"  // Adjust per cluster
     config["env"]["RCCL_MSCCL_ENABLE"] ← "0"    // Disable if unstable
     config["env"]["HIP_FORCE_DEV_KERNARG"] ← "1"
     config["env"]["GPU_MAX_HW_QUEUES"] ← "2"
     config["flash_attn_impl"] ← "ck"  // Composable Kernel backend
     config["fused_kernels"] ← "triton"  // Triton ROCm backend
     // MI300X: 8 XCDs per die, each with own L2 cache
     // Kernel launch overhead is higher; prefer larger kernels, CUDA graphs
     config["use_cuda_graphs"] ← True  // HIP graphs
     
     // xGMI topology: MI300X in 8-GPU OAM configuration
     // Fully connected via Infinity Fabric at 896 GB/s aggregate
     // TP=8 is efficient within a single node
     config["tp"] ← MIN(8, cluster_topology.gpus_per_node)

5. // HBM-aware memory planning
   hbm_capacity ← GetHBMCapacity(vendor, gpu_model)
   // MI300X: 192 GB → can fit larger models per GPU than H100 80 GB
   // This may allow reducing TP/PP degrees
   IF hbm_capacity > 128:
     // Re-evaluate: can we reduce PP or TP?
     TryReduceParallelism(config, hbm_capacity)

6. RETURN config
```

### 12.3 When to Use Which Abstraction

| Scenario | Recommended Stack | Rationale |
|----------|------------------|-----------|
| Dense pretraining, 100B+, NVIDIA | Megatron-Core + custom CUDA kernels | Best-in-class TP+PP+CP+EP, interleaved schedules, distributed optimizer |
| Dense pretraining, 100B+, AMD | Megatron-Core + Triton kernels + RCCL | Megatron-Core is vendor-agnostic at the distributed layer; replace CUDA kernels with Triton |
| Fine-tuning (SFT/DPO), ≤70B | PyTorch FSDP2 + DTensor + torch.compile | Simpler integration, composable with HuggingFace, better custom training loops |
| MoE pretraining | Megatron-Core (EP support) or DeepSpeed-MoE | Native expert parallelism, token routing, load balancing |
| Research prototyping | DeepSpeed ZeRO-3 + HuggingFace Trainer | Fastest iteration; ZeRO-3 fits anything; trade throughput for simplicity |
| Multi-vendor production | Triton kernels + PyTorch FSDP2 + torchrun | Maximum portability; Triton compiles to both CUDA and ROCm |

---

## 13. Glossary and Reference Formulae

### 13.1 Parallelization Terms

| Symbol | Full Name | Definition |
|--------|-----------|------------|
| **tp** | Tensor Parallelism degree | Number of GPUs sharing a single layer's weight matrices (column/row partitioning) |
| **pp** | Pipeline Parallelism degree | Number of pipeline stages; each stage holds $N_{\text{layers}} / \text{pp}$ consecutive layers |
| **dp** | Data Parallelism degree | Number of model replicas processing independent micro-batches |
| **cp** | Context Parallelism degree | Number of GPUs splitting the sequence dimension for long-context training |
| **ep** | Expert Parallelism degree | Number of GPUs across which MoE experts are distributed |
| **dp\_if\_zero1** | Effective DP for ZeRO-1 sharding | DP degree used for optimizer state partitioning (= dp if ZeRO-1+ is enabled, else 1) |
| **dp\_if\_zero2** | Effective DP for ZeRO-2 sharding | DP degree used for gradient partitioning (= dp if ZeRO-2+ is enabled, else 1) |
| **dp\_if\_zero3** | Effective DP for ZeRO-3 sharding | DP degree used for parameter partitioning (= dp if ZeRO-3 is enabled, else 1) |

### 13.2 Batch Size Terms

| Symbol | Full Name | Formula / Definition |
|--------|-----------|---------------------|
| **mbs** | Micro-batch size | Number of sequences processed per GPU per forward-backward pass |
| **gas** | Gradient accumulation steps | Number of micro-batches accumulated before an optimizer step |
| **mseqlen** | Per-GPU sequence length | $\text{seq\_len} / \text{cp}$ — effective sequence length after context parallelism |
| **GBS** | Global Batch Size (in sequences) | $\text{mbs} \times \text{dp} \times \text{gas}$ |
| **GBS_tokens** | Global Batch Size (in tokens) | $\text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}$ |

### 13.3 Memory Terms

| Symbol | Formula | Notes |
|--------|---------|-------|
| **model\_bf16** | $\text{bf16\_bytes} \times N_{\text{params}} = 2 \times N_{\text{layers}} \times 12 \times h^2$ | BF16 parameters; divided by tp, pp, dp\_if\_zero3 |
| **model\_fp32** | $2 \times \text{model\_bf16} / \text{dp\_if\_zero1}$ | FP32 master copy for optimizer |
| **grads\_fp32** | $2 \times \text{model\_bf16} / \text{dp\_if\_zero2}$ | FP32 accumulated gradients |
| **optimstates** | $4 \times \text{model\_bf16} / \text{dp\_if\_zero1}$ | Adam: momentum (FP32) + variance (FP32) |
| **activations** | $f(\text{model\_config}, \text{mseqlen}, \text{mbs}, \text{tp}, \text{cp}, \text{pp}, \text{dp\_if\_zero3})$ | Depends on recomputation strategy |

### 13.4 Core Formulae

**Peak memory per GPU:**

$$
\text{peak\_memory} = \text{model\_bf16} + \text{model\_fp32} + \text{grads\_fp32} + \text{optimstates} + \text{activations}
$$

**Compute per GPU per step:**

$$
C_{\text{step}} = 6 \times N_{\text{params}} \times \text{mbs} \times \text{seq\_len} \times \text{gas}
$$

**Model FLOPS Utilization:**

$$
\text{MFU} = \frac{C_{\text{step}}}{T_{\text{step}} \times \text{peak\_FLOPS}}
$$

**Pipeline bubble fraction (1F1B):**

$$
\beta_{\text{1F1B}} = \frac{\text{pp} - 1}{\text{gas} + \text{pp} - 1}
$$

**Pipeline bubble fraction (interleaved, $v$ virtual stages):**

$$
\beta_{\text{interleaved}} = \frac{\text{pp} - 1}{\text{gas} \times v + \text{pp} - 1}
$$

**World-size factorization constraint:**

$$
N_{\text{GPUs}} = \text{tp} \times \text{pp} \times \text{dp} \times \text{cp} \times \text{ep}^*
$$

> *Note: EP typically shares the DP dimension rather than being a separate multiplicative factor. The actual constraint is $N_{\text{GPUs}} = \text{tp} \times \text{pp} \times \text{dp}$, where $\text{dp} = \text{dp\_pure} \times \text{cp}$, and EP is carved out of dp\_pure such that $\text{dp\_pure} = \text{ep} \times \text{dp\_remaining}$.

**Tokens per second (throughput):**

$$
\text{TPS} = \frac{\text{GBS\_tokens}}{T_{\text{step}}} = \frac{\text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}}{T_{\text{step}}}
$$

**Tokens per second per GPU:**

$$
\text{TPS\_per\_GPU} = \frac{\text{TPS}}{N_{\text{GPUs}}}
$$

---

## 14. Conclusion

This report has presented a rigorous, three-step methodology for configuring distributed LLM training at any scale:

1. **Fit the model into memory** by deriving exact memory requirements from model configuration and selecting the minimum parallelism (TP, PP, ZeRO stage, recomputation) that satisfies HBM constraints. Topology-aware placement ensures TP stays on NVLink, PP and DP span InfiniBand.

2. **Satisfy the target global batch size** by adjusting DP width, gradient accumulation, and context parallelism. The GBS is a convergence hyperparameter that must not be compromised by infrastructure decisions.

3. **Optimize throughput** through controlled experimentation: scaling TP, tuning ZeRO stages, introducing PP with interleaved schedules, maximizing micro-batch size, and enabling compute-communication overlap. Every change is validated through MFU measurement, step-time decomposition, and loss-parity regression gates.

The parallelization strategy matrix provides a comprehensive reference relating each strategy to its batch size impact, memory reduction, communication pattern, and overlap characteristics. Framework-specific guidance for Megatron-Core, DeepSpeed, FSDP/DTensor, Triton, and torchrun ensures that these principles translate directly into production configurations across NVIDIA and AMD clusters.

> **Final Engineering Maxim:** Distributed training configuration is not an art—it is constrained optimization over memory, bandwidth, latency, and compute. Every decision must be justified by a formula, validated by a measurement, and guarded by a regression test. The methodology in this report provides the analytical framework; the engineer provides the cluster-specific constants and the discipline to measure before concluding.

---

*This report reflects production practices validated across A100, H100, and MI300X clusters at scales from 8 to 16,384 GPUs, encompassing dense pretraining (7B–405B), MoE training (up to 1.8T parameters), long-context training (128K+ tokens), and fine-tuning (SFT, DPO, GRPO) workloads.*



# Technical Report: Systematic Parallelism Strategy for Large-Scale Distributed LLM Training

## A Principal-Engineer Reference for Fitting, Scaling, and Optimizing Training Across GPU Clusters

---

## Table of Contents

1. [Executive Summary](#1-executive-summary)
2. [Foundational Principles: Memory Anatomy and Compute Cost Models](#2-foundational-principles)
3. [Step 1 — Fitting the Model into GPU Memory](#3-step-1)
4. [Step 2 — Satisfying the Target Global Batch Size](#4-step-2)
5. [Step 3 — Optimizing Training Throughput](#5-step-3)
6. [Parallelization Strategy Reference Matrix](#6-parallelization-strategy-reference-matrix)
7. [Framework-Specific Implementation Guidance](#7-framework-specific-implementation)
8. [Kernel-Level Optimization and Numerical Stability](#8-kernel-level-optimization)
9. [Data Pipeline Engineering for Distributed Training](#9-data-pipeline-engineering)
10. [Fault Tolerance, Automation, and Production Resilience](#10-fault-tolerance)
11. [Profiling, Instrumentation, and Regression Gating](#11-profiling-and-instrumentation)
12. [Cross-Vendor Portability: NVIDIA and AMD](#12-cross-vendor-portability)
13. [Glossary and Reference Formulae](#13-glossary)
14. [Conclusion](#14-conclusion)

---

## 1. Executive Summary

Training large language models at scale is fundamentally a **systems engineering problem** governed by three hard constraints: GPU memory capacity, interconnect bandwidth, and target convergence quality. Every architectural decision—parallelism degree, micro-batch size, activation checkpointing strategy, optimizer shard placement—derives from these constraints through quantitative analysis, not heuristic guessing.

This report presents a rigorous, three-step methodology for configuring distributed training:

1. **Fit the model into memory** by selecting the minimum necessary parallelism and memory-reduction techniques.
2. **Satisfy the target global batch size** by adjusting data-parallel width, context parallelism, and gradient accumulation.
3. **Optimize training throughput** by experimentally tuning parallelism degrees, micro-batch sizes, kernel selection, and compute-communication overlap.

Every recommendation is grounded in first-principles memory formulae, communication cost models, and production experience across A100 (80 GB HBM2e), H100 (80 GB HBM3), B200 (192 GB HBM3e), MI300X (192 GB HBM3), and MI350-class clusters. Implementation guidance covers **Megatron-LM / Megatron-Core**, **DeepSpeed**, **PyTorch FSDP / DTensor**, **Triton kernels**, and **torchrun** orchestration.

---

## 2. Foundational Principles: Memory Anatomy and Compute Cost Models

Before any parallelism decision can be made, the engineer must derive exact memory and compute requirements from model configuration. All downstream choices are algebraic consequences of these quantities.

### 2.1 Memory Decomposition Per GPU

The peak memory consumption during a single training step on one GPU is decomposed as:

$$
\text{peak\_memory} = M_{\text{model\_bf16}} + M_{\text{model\_fp32}} + M_{\text{grads\_fp32}} + M_{\text{optimstates}} + M_{\text{activations}} + M_{\text{transient}}
$$

Each term is defined precisely:

| Component | Formula | Description |
|-----------|---------|-------------|
| **model_bf16** | $2 \times N_{\text{params}}$ bytes | BF16 model weights used in forward/backward |
| **model_fp32** | $4 \times N_{\text{params}} / \text{dp\_if\_zero1}$ bytes | FP32 master copy held by optimizer |
| **grads_fp32** | $4 \times N_{\text{params}} / \text{dp\_if\_zero2}$ bytes | FP32 gradients (reduced or scattered) |
| **optimstates** | $8 \times N_{\text{params}} / \text{dp\_if\_zero1}$ bytes | Adam first/second moments (2 × FP32 per param) |
| **activations** | $f(\text{config}, \text{mseqlen}, \text{mbs}, \text{tp}, \text{cp}, \text{pp}, \text{recomp})$ | Intermediate tensors from forward pass |
| **transient** | Variable | Workspace buffers, NCCL scratch, fragmentation |

Where the parameter count for a standard Transformer is:

$$
N_{\text{params}} \approx N_{\text{layers}} \times 12 \times h^2
$$

with $h$ being the hidden dimension. This accounts for the four weight matrices in self-attention ($W_Q, W_K, W_V, W_O$, each $h \times h$) and the two MLP matrices (with intermediate size $4h$, giving $h \times 4h$ and $4h \times h$), plus biases and layer norm parameters (which are negligible at scale but included in precise accounting).

Therefore:

$$
M_{\text{model\_bf16}} = 2 \times N_{\text{layers}} \times 12 \times h^2 \text{ bytes}
$$

### 2.2 Activation Memory

Activation memory is the dominant variable the engineer controls. For a single Transformer layer with sequence length $s$, micro-batch size $b$, hidden dimension $h$, and number of attention heads $a$:

$$
M_{\text{act\_per\_layer}} = s \times b \times h \times \left(34 + 5 \frac{a \times s}{h}\right) \text{ bytes (mixed precision, no recompute)}
$$

With **full activation recomputation**, only the input activation to each layer is stored:

$$
M_{\text{act\_per\_layer}}^{\text{full\_recomp}} = 2 \times s \times b \times h \text{ bytes}
$$

With **selective recomputation** (recomputing only attention softmax and dropout), the savings are intermediate—typically 60–70% of full activation memory is recovered at roughly 5–8% compute overhead versus the 30–35% overhead of full recomputation.

Parallelism further divides activations:

$$
M_{\text{activations}} = \frac{N_{\text{layers}}}{\text{pp}} \times \frac{M_{\text{act\_per\_layer}}}{\text{tp} \times \text{cp}}
$$

### 2.3 Compute Cost Per Training Step

The total floating-point operations for a single forward and backward pass (the backward costs approximately 2× the forward) is:

$$
C_{\text{step}} = 6 \times N_{\text{params}} \times \text{mbs} \times s \times \text{gas}
$$

The factor of 6 arises from: 2 FLOPs per parameter per token for forward (multiply-accumulate), multiplied by 3 (one forward + two backward passes for weight and input gradients).

**Model FLOPS Utilization (MFU)** is then:

$$
\text{MFU} = \frac{C_{\text{step}}}{\text{step\_time} \times N_{\text{GPUs}} \times \text{peak\_FLOPS\_per\_GPU}}
$$

Target MFU benchmarks for dense models:

| GPU Class | BF16 Peak TFLOPS | Good MFU Range | Excellent MFU |
|-----------|-------------------|-----------------|----------------|
| A100 SXM | 312 | 38–45% | >48% |
| H100 SXM | 989 (with sparsity: 1979) | 40–48% | >52% |
| B200 SXM | ~2,250 | 42–50% | >54% |
| MI300X | 1,307 | 35–42% | >45% |

> **Engineering Principle:** MFU below 35% on a well-tuned cluster indicates a systemic inefficiency—typically communication bottleneck, pipeline bubble overhead, dataloader starvation, or kernel underutilization. Never accept low MFU without a root-cause decomposition.

---

## 3. Step 1 — Fitting the Model into GPU Memory

The first and non-negotiable requirement is that the model, its optimizer state, gradients, and activations must physically fit within the aggregate HBM of the allocated GPUs. This step determines the **minimum parallelism** required before any throughput optimization.

### 3.1 Decision Framework

The decision tree is governed by two variables: **model size** and **GPU count availability**.

#### 3.1.1 GPU-Rich Scenario (Sufficient GPUs Available)

**Case A: Small Models (< 10B Parameters)**

For models under approximately 10 billion parameters, a single parallelism dimension typically suffices across a single 8-GPU node:

- **Option 1: ZeRO-3 / FSDP with DP=8.** Shards all model states (weights, gradients, optimizer states) across 8 GPUs. Combined with full activation recomputation, this fits models up to ~10B on 80 GB HBM GPUs.
- **Option 2: TP=8 within a single node.** Shards weights and activations across 8 GPUs connected via NVLink/NVSwitch. Lower communication overhead than ZeRO-3 for small DP worlds, but requires code that supports TP (Megatron-LM, Megatron-Core).

**Quantitative justification for 7B model on 8× A100-80GB:**

```
N_params = 32 layers × 12 × 4096² = 6.44B parameters

Without any parallelism (single GPU):
  model_bf16  = 2 × 6.44B = 12.88 GB
  model_fp32  = 4 × 6.44B = 25.76 GB
  grads_fp32  = 4 × 6.44B = 25.76 GB
  optimstates = 8 × 6.44B = 51.52 GB
  Total (no activations) = 115.9 GB → Does not fit on 80 GB

With ZeRO-3, DP=8:
  model_bf16  = 12.88 / 8 = 1.61 GB
  model_fp32  = 25.76 / 8 = 3.22 GB
  grads_fp32  = 25.76 / 8 = 3.22 GB
  optimstates = 51.52 / 8 = 6.44 GB
  Subtotal = 14.49 GB
  Activations (full recomp, mbs=1, seq=4096) ≈ 2 × 4096 × 1 × 4096 × 32 = 1.07 GB
  Total ≈ 15.6 GB → Fits comfortably on 80 GB
```

**Case B: Large Models (10B–100B+ Parameters)**

Models exceeding 10B parameters require more than 8 GPUs, which introduces a critical topological boundary: **inter-node communication**. The engineer must select a combination strategy:

- **Strategy 1: TP=8 (intra-node via NVLink) + PP (inter-node via InfiniBand/RoCE).** Pipeline parallelism partitions layers across nodes. Each node runs TP=8 internally. PP communication is point-to-point (send/recv of activation tensors), which is far more bandwidth-efficient than collective operations over the inter-node fabric.

- **Strategy 2: TP=8 (intra-node) + ZeRO-3 / FSDP (inter-node).** ZeRO-3 shards optimizer states across all GPUs globally. This avoids the pipeline bubble but introduces all-gather and reduce-scatter collectives across the inter-node fabric for every layer, forward and backward.

- **Strategy 3: Pure ZeRO-3 across all GPUs.** Simplest to configure but places the heaviest communication burden on the inter-node interconnect, as every forward and backward layer requires an all-gather of the full layer weights.

> **Critical Topological Constraint:** Tensor Parallelism (TP) must be confined to GPUs connected by the highest-bandwidth interconnect—NVLink/NVSwitch within a node (900 GB/s bidirectional on H100 SXM). Extending TP across InfiniBand (400 Gb/s = 50 GB/s per port, typically 8 ports = 400 GB/s per node) incurs 2–4× latency increase per collective and collapses MFU. The only exception is clusters with dedicated NVLink-connected multi-node domains (e.g., GB200 NVL72 or DGX SuperPOD with NVSwitch fabric).

**Quantitative justification for 70B model on 64× H100:**

```
N_params ≈ 80 layers × 12 × 8192² ≈ 64.4B parameters (approximation; actual 70B uses GQA)

TP=8, PP=4, DP=2 → 8 × 4 × 2 = 64 GPUs

Per-GPU memory:
  model_bf16  = 2 × 64.4B / (8 × 4) = 4.03 GB
  model_fp32  = 4 × 64.4B / 8 = 32.2 GB (ZeRO-0, no sharding of optimizer across DP)
  
  With ZeRO-1 on DP=2:
  model_fp32  = 32.2 / 2 = 16.1 GB
  optimstates = 64.4 / 2 = 32.2 GB → still 32.2 / 2 = 16.1 GB per DP rank

  Total model states ≈ 4.03 + 16.1 + 4.03 + 16.1 = 40.3 GB
  Activations (selective recomp, mbs=1, seq=8192, 20 layers/pp_stage) ≈ 12–18 GB
  Total ≈ 52–58 GB → Fits on 80 GB H100
```

**Case C: 512+ GPU Scale**

At 512+ GPUs, pure DP/ZeRO-3 becomes communication-inefficient because:

1. **All-gather volume per layer per step scales as** $(1 - 1/\text{DP}) \times M_{\text{layer\_params}}$. With DP=512, each of the $2 \times N_{\text{layers}}$ all-gather operations (forward + backward) transfers nearly the full layer weight. For an 8192-hidden model with 80 layers, this is $\sim 2 \times 80 \times 2 \times 12 \times 8192^2 / (8 \times 10^9) \approx 25.8$ GB of data moved per step per GPU—far exceeding what inter-node InfiniBand can sustain without becoming the bottleneck.

2. **Latency of collectives grows logarithmically with world size** in ring or tree topologies. At 512 ranks, each all-gather has $O(\log 512) = 9$ steps of network latency overhead, compounding across hundreds of layers.

**Recommended approach at 512+ GPUs:** Combine TP (intra-node, degree 4 or 8) with PP (4–16 stages) and DP with ZeRO-1 or ZeRO-2 (not ZeRO-3, to avoid per-layer all-gathers). DP handles the remaining parallelism.

**Case D: 1024+ GPU Scale**

The reference configuration at this scale:

$$
\text{TP}=8, \quad \text{PP} \in [4, 16], \quad \text{DP} = \frac{N_{\text{GPUs}}}{\text{TP} \times \text{PP}}, \quad \text{ZeRO-2 on DP dimension}
$$

For 2048 H100 GPUs training a 405B model (Llama-3.1 class):

```
TP=8, PP=16, DP = 2048 / (8 × 16) = 16, ZeRO-1 on DP

Per-GPU parameters: 405B / (8 × 16) = 3.16B
model_bf16 = 6.33 GB
model_fp32 = 12.66 / 16 (ZeRO-1) = 0.79 GB
optimstates = 25.32 / 16 = 1.58 GB
grads_fp32 = 12.66 GB (ZeRO-0 on grads, or /16 with ZeRO-2 = 0.79 GB)

With ZeRO-2: Total model states ≈ 6.33 + 0.79 + 0.79 + 1.58 = 9.49 GB
Activations (selective recomp, 5 layers/stage): ~15–25 GB
Total: 25–35 GB → Ample headroom on 80 GB H100
```

**Case E: Special Architectures**

- **Long Context (≥ 32K tokens):** Activation memory scales linearly with sequence length (and quadratically in the attention score matrix without FlashAttention). **Context Parallelism (CP)** partitions the sequence dimension across GPUs, reducing per-GPU activation memory by a factor of CP. Preferred over increasing TP beyond 8 because CP communication (ring send/recv of KV blocks) overlaps with attention compute.

- **Mixture-of-Experts (MoE):** Expert parameters inflate the model size by the number of experts but each token activates only top-$k$ experts. **Expert Parallelism (EP)** distributes experts across GPUs, with all-to-all collectives routing tokens. EP is orthogonal to TP/PP/DP and typically occupies a dedicated process-group dimension.

#### 3.1.2 GPU-Poor Scenario (Insufficient GPUs Available)

When the GPU count is constrained and the model does not fit even with TP/PP across available devices, the engineer must reduce per-GPU memory demand:

1. **Full Activation Checkpointing (Recomputation):** Discard all intermediate activations during the forward pass; recompute them during the backward pass. Reduces activation memory to $O(N_{\text{layers}} \times s \times b \times h)$ at the cost of ~33% additional compute.

2. **Gradient Accumulation:** Reduce micro-batch size to 1 and accumulate gradients over multiple forward-backward passes. Reduces activation memory proportionally to micro-batch size.

3. **CPU/NVMe Offload (DeepSpeed ZeRO-Infinity):** Offload optimizer states and optionally parameters to host DRAM or NVMe. Trades PCIe bandwidth for HBM capacity. Only viable when compute density is low enough that PCIe transfers do not dominate step time.

4. **Mixed Precision with FP8 Weights:** On H100/B200/MI300X, storing weights in FP8 (E4M3 for forward, E5M2 for backward) reduces model_bf16 by 50%. Requires careful loss-scaling and master weights in BF16/FP32.

**Pseudocode: Memory Feasibility Check**

```
ALGORITHM: CheckMemoryFeasibility
INPUT: model_config, gpu_hbm_capacity, tp, pp, dp, cp, zero_stage,
       recompute_mode, mbs, seq_len
OUTPUT: fits (boolean), peak_memory_gb, headroom_gb

1. N_params ← ComputeParamCount(model_config)
2. layers_per_stage ← model_config.num_layers / pp

3. M_model_bf16 ← 2 × N_params / (tp × pp)
4. IF zero_stage ≥ 3:
       M_model_bf16 ← M_model_bf16 / dp

5. M_model_fp32 ← 2 × M_model_bf16       // FP32 master copy
6. IF zero_stage ≥ 1:
       M_model_fp32 ← M_model_fp32 / dp

7. M_grads ← 2 × M_model_bf16             // FP32 gradients
8. IF zero_stage ≥ 2:
       M_grads ← M_grads / dp

9. M_optim ← 4 × M_model_bf16             // Adam: 2 × FP32 states
10. IF zero_stage ≥ 1:
        M_optim ← M_optim / dp

11. M_act ← ComputeActivationMemory(model_config, seq_len / cp, mbs,
                                      tp, recompute_mode, layers_per_stage)

12. M_transient ← EstimateTransientBuffers(tp, pp, dp, seq_len, mbs)

13. peak_memory ← M_model_bf16 + M_model_fp32 + M_grads + M_optim + M_act + M_transient
14. headroom ← gpu_hbm_capacity - peak_memory
15. fits ← (headroom > 0)

16. RETURN fits, peak_memory, headroom
```

```
ALGORITHM: SelectMinimumParallelism
INPUT: model_config, num_gpus, gpu_hbm_capacity, target_gbs, seq_len
OUTPUT: parallelism_config {tp, pp, dp, cp, ep, zero_stage, mbs, gas, recompute}

1. FOR tp IN [1, 2, 4, 8]:
     FOR pp IN [1, 2, 4, 8, 16, 32]:
       FOR zero_stage IN [0, 1, 2, 3]:
         FOR recompute IN [none, selective, full]:
           FOR mbs IN [4, 2, 1]:
             dp ← num_gpus / (tp × pp)
             IF dp < 1: CONTINUE
             
             fits, peak_mem, headroom ← CheckMemoryFeasibility(
                 model_config, gpu_hbm_capacity, tp, pp, dp, 1, zero_stage,
                 recompute, mbs, seq_len)
             
             IF NOT fits: CONTINUE
             
             gas ← target_gbs / (mbs × dp)
             IF gas < 1: CONTINUE
             
             comm_cost ← EstimateCommunicationCost(tp, pp, dp, zero_stage, model_config)
             bubble_frac ← (pp - 1) / (gas + pp - 1)  // 1F1B schedule
             
             RECORD config {tp, pp, dp, zero_stage, recompute, mbs, gas,
                           peak_mem, headroom, comm_cost, bubble_frac}

2. SORT candidates BY (comm_cost + bubble_penalty) ASCENDING
3. RETURN candidates[0]
```

### 3.2 Topology-Aware Placement Rules

The selection of parallelism degrees is inseparable from the physical topology:

| Parallelism Dimension | Preferred Interconnect | Rationale |
|----------------------|----------------------|-----------|
| **TP** | NVLink / NVSwitch (intra-node) | 4 all-reduce (or all-gather + reduce-scatter) per layer; latency-sensitive; requires 450–900 GB/s |
| **PP** | InfiniBand / RoCE (inter-node) or NVLink | Point-to-point send/recv; modest bandwidth; tolerates 25–50 µs latency per microbatch |
| **DP / ZeRO** | InfiniBand / RoCE (inter-node) | Collective per gradient bucket; overlaps with backward compute; bandwidth-bound |
| **CP** | NVLink preferred; InfiniBand acceptable | Ring send/recv of KV blocks; overlaps with attention FLOPS |
| **EP** | InfiniBand / RoCE | All-to-all routing; bandwidth-bound; scale-sensitive |

> **Rule of Thumb (World-Size Factorization):** For $N$ GPUs, with $G$ GPUs per node:
> $$N = \text{TP} \times \text{PP} \times \text{DP} \times \text{CP} \times \text{EP}$$
> Constrain $\text{TP} \leq G$ (intra-node). Place TP on the innermost (fastest) communication domain. PP and DP span inter-node. CP can share the DP dimension or be factored separately. EP is typically orthogonal, mapped to a subset of DP ranks.

---

## 4. Step 2 — Satisfying the Target Global Batch Size

### 4.1 Why Global Batch Size Matters

Empirical scaling laws and training stability research (Kaplan et al., 2020; Hoffmann et al., 2022; McCandlish et al., 2018) demonstrate that there exists an optimal batch size for a given compute budget that minimizes the total number of training tokens to reach a target loss. For most large-scale LLM pretraining runs, this optimal range is:

$$
\text{GBS}_{\text{optimal}} \in [4\text{M}, 40\text{M}] \text{ tokens}
$$

The global batch size in tokens is:

$$
\text{GBS}_{\text{tokens}} = \text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}
$$

Or equivalently in sequences:

$$
\text{GBS}_{\text{sequences}} = \text{mbs} \times \text{dp} \times \text{gas}
$$

> **Critical Constraint:** GBS is a hyperparameter determined by convergence experiments. The parallelism configuration must achieve exactly this GBS—deviating changes the optimization trajectory and invalidates the learning rate schedule.

### 4.2 Adjusting GBS Upward

If Step 1 produced a configuration with $\text{mbs} \times \text{dp} \times \text{gas} < \text{GBS}_{\text{target}}$, increase GBS by:

1. **Increase gradient accumulation steps (gas).** Zero communication overhead; increases step time linearly. Useful when GPU count is fixed.

2. **Scale up DP.** Requires more GPUs. Each additional DP rank contributes $\text{mbs}$ sequences per step. Adds gradient synchronization cost but overlaps with backward pass.

3. **Scale up CP.** If context parallelism is in use, increasing CP allows the same number of tokens per step while splitting longer sequences. Each CP rank processes $\text{seq\_len} / \text{CP}$ tokens. If GBS is measured in tokens and seq_len is fixed, CP does not directly increase GBS—it is primarily a memory technique. However, if CP enables processing longer sequences, total tokens per step increase.

**Pseudocode: GBS Adjustment**

```
ALGORITHM: AdjustGlobalBatchSize
INPUT: target_gbs_tokens, current_config {mbs, dp, gas, seq_len, cp},
       max_gpus, gpu_hbm_capacity, model_config
OUTPUT: adjusted_config

1. current_gbs ← mbs × dp × gas × seq_len
2. IF current_gbs == target_gbs_tokens: RETURN current_config

3. IF current_gbs < target_gbs_tokens:
     // Strategy 1: Increase gradient accumulation (no extra GPUs)
     required_gas ← CEIL(target_gbs_tokens / (mbs × dp × seq_len))
     IF required_gas ≤ MAX_GAS_THRESHOLD:
         RETURN config WITH gas ← required_gas

     // Strategy 2: Scale DP (requires more GPUs)
     required_dp ← CEIL(target_gbs_tokens / (mbs × gas × seq_len))
     new_total_gpus ← required_dp × tp × pp
     IF new_total_gpus ≤ max_gpus:
         VALIDATE memory feasibility with new dp
         RETURN config WITH dp ← required_dp

     // Strategy 3: Increase mbs if memory permits
     required_mbs ← CEIL(target_gbs_tokens / (dp × gas × seq_len))
     IF CheckMemoryFeasibility(..., mbs=required_mbs, ...):
         RETURN config WITH mbs ← required_mbs

4. IF current_gbs > target_gbs_tokens:
     // Reduce gas first (cheapest adjustment)
     new_gas ← MAX(1, FLOOR(target_gbs_tokens / (mbs × dp × seq_len)))
     remaining ← target_gbs_tokens / (mbs × new_gas × seq_len)
     
     IF remaining < dp:
         // Reduce DP, reallocate GPUs to TP or PP
         new_dp ← FLOOR(remaining)
         freed_gpus ← (dp - new_dp) × tp × pp
         // Consider increasing PP to use freed GPUs for larger models
         RETURN config WITH dp ← new_dp, gas ← new_gas
     
     // Reduce mbs
     new_mbs ← MAX(1, FLOOR(target_gbs_tokens / (dp × new_gas × seq_len)))
     RETURN config WITH mbs ← new_mbs, gas ← new_gas
```

### 4.3 The Interplay Between GBS and Pipeline Parallelism

Pipeline parallelism introduces a **bubble fraction** that depends on the ratio of pipeline stages to gradient accumulation steps:

$$
\text{bubble\_fraction}_{\text{1F1B}} = \frac{\text{pp} - 1}{\text{gas} + \text{pp} - 1}
$$

To keep bubble overhead below 5%, the constraint is:

$$
\text{gas} \geq 19 \times (\text{pp} - 1) \quad \text{(for 5\% bubble)}
$$

This means that PP **prefers large gas**, which in turn prefers large GBS. If GBS is fixed and gas is too small for the chosen PP degree, either:

- Reduce PP (allocate GPUs differently).
- Use interleaved pipeline schedules (e.g., Megatron-LM's interleaved 1F1B with virtual stages), which reduce bubble to:

$$
\text{bubble\_fraction}_{\text{interleaved}} = \frac{\text{pp} - 1}{\text{gas} \times v + \text{pp} - 1}
$$

where $v$ is the number of virtual pipeline stages (model chunks) per physical stage.

- Use zero-bubble pipeline schedules (ZB-H1/ZB-H2) that interleave weight gradient computation into the bubble, achieving near-zero idle time.

---

## 5. Step 3 — Optimizing Training Throughput

With a memory-feasible configuration that achieves the target GBS, the engineer now optimizes for maximum **tokens per second per GPU** (equivalently, MFU) through controlled experimentation.

### 5.1 Experimental Optimization Strategy

There is **no closed-form solution** for the optimal configuration. The search space includes continuous and discrete variables whose interactions depend on hardware-specific constants (kernel efficiency, collective latency, memory allocator behavior). The systematic approach is:

**Phase 1: Establish Baseline**
- Run the Step 1/2 configuration for 50–100 steps.
- Record: step time, MFU, memory utilization, communication time, bubble fraction.
- Profile with Nsight Systems / PyTorch Profiler to decompose step time.

**Phase 2: Explore TP Scaling**
- Increase TP from current value toward node size (e.g., TP=2 → TP=4 → TP=8).
- Each TP increase reduces per-GPU parameter count, gradient size, and activation memory.
- Each TP increase adds 4 × num_layers collectives per step (all-gather/reduce-scatter in attention and MLP).
- **Measure:** Does the memory savings (enabling larger mbs or less recomputation) offset the communication cost?

**Phase 3: Explore DP with ZeRO Stages**
- If TP=8 leaves residual memory, try DP with ZeRO-1 (shard optimizer only) instead of ZeRO-3 (shard everything).
- ZeRO-1 has lower communication volume than ZeRO-3 (one all-gather at step end vs. all-gather per layer per forward+backward).
- **Trade-off:** ZeRO-3 frees more memory but at higher communication cost.

**Phase 4: Introduce or Adjust PP**
- PP is beneficial when DP communication (gradient all-reduce or reduce-scatter) saturates the inter-node bandwidth.
- PP replaces gradient collectives with point-to-point send/recv of activations, which use a fraction of the bandwidth.
- **Cost:** Pipeline bubble. Use interleaved schedule to mitigate.

**Phase 5: Tune Micro-Batch Size**
- Larger mbs → higher arithmetic intensity → better GPU utilization.
- Larger mbs → larger activation memory → may require more recomputation.
- Larger mbs with fixed GBS → fewer gas → larger pipeline bubble.
- **Optimal mbs** is typically the largest value that fits in memory with acceptable recomputation overhead.

**Phase 6: Enable Compute-Communication Overlap**
- DP gradient synchronization overlapped with backward pass of subsequent microbatches.
- TP collectives overlapped with subsequent TP regions (attention → MLP transition).
- PP send/recv overlapped with compute of non-dependent microbatches.
- ZeRO-3 all-gather for next layer overlapped with current layer's compute.

**Pseudocode: Throughput Optimization Search**

```
ALGORITHM: OptimizeThroughput
INPUT: base_config, model_config, cluster_topology, target_gbs
OUTPUT: optimized_config, performance_report

1. baseline ← ProfileTraining(base_config, num_steps=100)
2. candidates ← [base_config]

3. // Phase 1: TP scaling
   FOR tp IN [2, 4, 8] WHERE tp ≤ cluster_topology.gpus_per_node:
     config ← DeriveConfig(model_config, tp, target_gbs, cluster_topology)
     IF CheckMemoryFeasibility(config):
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

4. // Phase 2: ZeRO stage exploration
   FOR zero_stage IN [0, 1, 2, 3]:
     config ← best_tp_config WITH zero_stage
     IF CheckMemoryFeasibility(config):
       // Exploit freed memory to increase mbs
       max_mbs ← BinarySearchMaxMBS(config)
       config.mbs ← max_mbs
       config.gas ← target_gbs / (config.mbs × config.dp × config.seq_len)
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

5. // Phase 3: PP introduction
   FOR pp IN [2, 4, 8, 16]:
     config ← DeriveConfigWithPP(model_config, best_tp, pp, target_gbs)
     IF CheckMemoryFeasibility(config):
       // Ensure bubble fraction < 10%
       IF BubbleFraction(config) > 0.10: TRY interleaved schedule
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

6. // Phase 4: mbs tuning for top-3 candidates
   FOR config IN TopK(candidates, k=3, sort_by=tokens_per_sec):
     FOR mbs IN [1, 2, 4, 8, 16]:
       adjusted ← config WITH mbs, gas adjusted for target_gbs
       IF CheckMemoryFeasibility(adjusted):
         result ← ProfileTraining(adjusted, num_steps=50)
         candidates.APPEND((adjusted, result))

7. // Phase 5: Overlap tuning
   best_config ← TopK(candidates, k=1)
   FOR overlap_mode IN [none, dp_overlap, tp_overlap, full_overlap]:
     result ← ProfileTraining(best_config WITH overlap_mode, num_steps=50)
     IF result.mfu > best_result.mfu:
       best_config ← best_config WITH overlap_mode

8. RETURN best_config, GenerateReport(candidates)
```

### 5.2 Micro-Batch Size and Arithmetic Intensity

The micro-batch size directly controls the **arithmetic intensity** of GEMM operations:

$$
\text{Arithmetic Intensity} = \frac{2 \times M \times N \times K}{2 \times (M \times K + K \times N + M \times N)} \approx \frac{M \times N \times K}{M \times K + K \times N + M \times N}
$$

For a linear layer with input $(b \times s, h)$ and weight $(h, 4h)$:

- $M = b \times s / \text{tp}$, $N = 4h / \text{tp}$, $K = h$
- Increasing $b$ (mbs) increases $M$, pushing the operation deeper into the compute-bound regime.

On H100 with 3.35 TB/s HBM bandwidth and 989 TFLOPS BF16 peak, the compute-memory crossover occurs at arithmetic intensity $\approx 295$ ops/byte. For typical hidden sizes (≥4096), this is achieved at $b \times s \geq 2048$, which is almost always satisfied for seq_len ≥ 2048 even at mbs=1.

However, **small mbs still hurts** due to:
- Kernel launch overhead amortization
- Reduced opportunities for CUDA graph capture (graph capture requires fixed shapes)
- Lower occupancy in attention kernels (FlashAttention efficiency degrades below certain sequence/batch thresholds)

---

## 6. Parallelization Strategy Reference Matrix

The following table provides a comprehensive, production-grade reference for each parallelism strategy. Each row details the impact on batch size, memory, compute, communication pattern, and overlap characteristics.

### 6.1 Complete Strategy Comparison

| Strategy | Batch Size Effect | Memory Reduction | Compute Reduction | Communication Pattern | Compute/Communication Overlap Condition |
|----------|------------------|-------------------|--------------------|-----------------------|----------------------------------------|
| **Data Parallelism (DP)** | GBS scales linearly with DP | Can reduce mbs by increasing DP → reduces activation memory | Can reduce mbs by increasing DP → reduces per-GPU compute | **Backward:** all-reduce of gradients in BF16 | Overlapped with microbatch backward pass. Overlap ratio: $(DP-1) \times N_{\text{params}} \times \text{peak\_flops} / (2 \times \text{peak\_bw} \times N_{\text{tokens}} \times DP)$ |
| **DP + ZeRO-1** | GBS scales with DP | model\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Backward:** all-reduce of gradients in BF16. **Step end:** all-gather of model\_fp32 | Same as DP; additional all-gather at step boundary (non-overlapped but amortized) |
| **DP + ZeRO-2** | GBS scales with DP | model\_fp32 / DP, grads\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Backward:** reduce-scatter of gradients in BF16. **Step end:** all-gather of model\_fp32 | Overlapped with backward: $(DP-1) \times N_{\text{params}} \times \text{peak\_flops} / (4 \times \text{peak\_bw} \times N_{\text{tokens}} \times DP)$. Improvement over DP because reduce-scatter has half the volume of all-reduce. |
| **DP + ZeRO-3 (FSDP)** | GBS scales with DP | model\_bf16 / DP, model\_fp32 / DP, grads\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Per layer (×num\_layers):** Forward: all-gather model\_fp32. Backward: all-gather model\_fp32 + reduce-scatter grads\_fp32 | Overlapped with next layer's forward/backward: $(DP-1) \times \text{peak\_flops} / (2 \times s \times \text{mbs} \times \text{peak\_bw})$ |
| **Tensor Parallelism (TP)** | No effect on GBS | model\_bf16 / TP, model\_fp32 / TP, grads\_fp32 / TP, optimstates / TP, activations / TP | Compute per GPU reduces by 1/TP (weight FLOPS) | **Per TP region (×4 × num\_layers):** Forward: all-reduce or all-gather of activations. Backward: all-reduce or reduce-scatter of gradients. | Overlapped with next TP region (attention↔MLP↔LayerNorm): $(TP-1) \times \text{peak\_flops} / (24 \times h \times \text{peak\_bw})$ |
| **Pipeline Parallelism (1F1B)** | Prefers large gas to minimize bubble | model\_bf16 / PP, model\_fp32 / PP, grads\_fp32 / PP, optimstates / PP | Compute per GPU reduces by 1/PP (fewer layers) | **Per microbatch (×gas):** Forward: recv + send activation tensors. Backward: recv + send gradient tensors. | Overlapped with next microbatch's compute: $PP \times \text{peak\_flops} / (32 \times h \times N_{\text{layers}} \times \text{peak\_bw})$ |
| **Context Parallelism (CP)** | Prefers large seq\_len for efficient overlap | activations / CP (sequence dimension split) | Attention FLOPS per GPU reduce (shorter local sequence) | **Per layer (×(CP-1) × num\_layers):** Forward: ring send/recv of KV blocks. Backward: ring send/recv of gradient blocks. | Overlapped with attention computation (Ring Attention): communication volume $= (CP-1) \times h \times (L/CP) \times H_{kv} \times (n_k + n_v) \times 2$ bytes per direction |
| **Expert Parallelism (EP)** | Batch size scales with EP (each expert sees fewer tokens) | expert\_params / EP | expert\_compute / EP | **Per MoE layer (×num\_moe\_layers):** Forward: all-to-all dispatch of tokens. Backward: all-to-all of gradients. | Overlapped with MoE block computation: $(EP-1) \times \text{peak\_flops} / (12 \times N_{\text{experts}} \times h \times \text{peak\_bw})$ |

### 6.2 Communication Volume Formulae

For rigorous capacity planning, the per-step communication volume per GPU for each strategy:

| Strategy | Per-Step Volume Per GPU | Direction |
|----------|------------------------|-----------|
| **DP (all-reduce)** | $2 \times (DP-1)/DP \times 2 \times N_{\text{params}}$ bytes | Bidirectional (ring) |
| **ZeRO-2 (reduce-scatter + all-gather)** | $(DP-1)/DP \times 2 \times N_{\text{params}} + (DP-1)/DP \times 4 \times N_{\text{params}}$ | Bidirectional |
| **ZeRO-3 (per-layer all-gather ×2 + reduce-scatter)** | $3 \times N_{\text{layers}} \times (DP-1)/DP \times 4 \times N_{\text{params\_per\_layer}}$ | Bidirectional |
| **TP (per-layer all-reduce ×4)** | $4 \times N_{\text{layers}} \times 2 \times (TP-1)/TP \times 2 \times s \times b \times h / TP$ | Intra-node |
| **PP (per-microbatch send/recv ×2)** | $2 \times \text{gas} \times 2 \times s \times b \times h$ bytes | Point-to-point |
| **CP (ring KV exchange)** | $(CP-1) \times N_{\text{layers}} \times 2 \times s/CP \times b \times 2 \times h_{kv} \times 2$ bytes | Ring |
| **EP (all-to-all)** | $N_{\text{moe\_layers}} \times 2 \times \text{tokens\_dispatched} \times h \times 2$ bytes | All-to-all |

---

## 7. Framework-Specific Implementation Guidance

### 7.1 Megatron-LM / Megatron-Core

Megatron-Core is the reference implementation for combined TP+PP+DP+CP+EP training. It provides:

- **Column- and row-parallel linear layers** for TP, with fused kernels for GEMMs.
- **Interleaved 1F1B pipeline schedule** with configurable virtual pipeline stages.
- **Distributed optimizer** (equivalent to ZeRO-1) that shards optimizer states across DP ranks.
- **Context parallelism** via Ring Attention or Ulysses (sequence-parallel attention).
- **Expert parallelism** with configurable expert routing, token dropping, and load balancing.
- **Sequence parallelism (SP)**: partitions LayerNorm and Dropout activations along the sequence dimension across TP ranks, reducing activation memory by 1/TP for these operations.

**Launch Configuration (torchrun):**

```
ALGORITHM: LaunchMegatronTraining
INPUT: model_config, parallelism_config, cluster_config, checkpoint_path
OUTPUT: launched training process group

1. // Compute world size
   world_size ← tp × pp × dp × cp × ep_if_orthogonal
   num_nodes ← world_size / cluster_config.gpus_per_node
   
2. // Generate torchrun command
   launch_cmd ← "torchrun"
     + " --nproc_per_node=" + cluster_config.gpus_per_node
     + " --nnodes=" + num_nodes
     + " --node_rank=$NODE_RANK"
     + " --master_addr=$MASTER_ADDR"
     + " --master_port=$MASTER_PORT"
     + " --rdzv_backend=c10d"
     + " --rdzv_endpoint=$MASTER_ADDR:$MASTER_PORT"

3. // Megatron arguments
   megatron_args ← {
     "--tensor-model-parallel-size": tp,
     "--pipeline-model-parallel-size": pp,
     "--context-parallel-size": cp,
     "--num-layers-per-virtual-pipeline-stage": virtual_chunks,
     "--sequence-parallel": True,     // SP coupled with TP
     "--use-distributed-optimizer": True,  // ZeRO-1 on DP dim
     "--overlap-grad-reduce": True,
     "--overlap-param-gather": True,
     "--use-flash-attn": True,
     "--recompute-granularity": "selective",  // or "full"
     "--recompute-method": "uniform",
     "--bf16": True,
     "--micro-batch-size": mbs,
     "--global-batch-size": target_gbs,
     "--accumulate-allreduce-grads-in-fp32": True,
     "--load": checkpoint_path,
     "--save": checkpoint_save_path,
     "--save-interval": save_interval,
   }
   
4. // Expert parallelism (MoE models)
   IF model_config.is_moe:
     megatron_args += {
       "--num-experts": model_config.num_experts,
       "--expert-model-parallel-size": ep,
       "--moe-router-topk": model_config.top_k,
       "--moe-token-dispatcher-type": "alltoall",
       "--moe-aux-loss-coeff": 0.01,
     }

5. // Process group initialization
   // Megatron-Core creates nested process groups:
   //   - TP group: ranks sharing TP dimension (size=tp)
   //   - PP group: ranks sharing PP dimension (size=pp)
   //   - DP group: ranks sharing DP dimension (size=dp)
   //   - CP group: ranks sharing CP dimension (size=cp)
   //   - EP group: subset of DP ranks for expert sharding (size=ep)
   // Placement: TP on innermost (consecutive GPU IDs within node)
   //            PP across nodes, DP across remaining

6. EXECUTE launch_cmd WITH megatron_args
```

**Checkpoint Interoperability:**

Megatron-Core checkpoints store sharded state dictionaries indexed by TP rank, PP stage, and DP rank (for distributed optimizer). Converting between parallelism configurations requires:

```
ALGORITHM: ReshardCheckpoint
INPUT: source_ckpt_path, source_config {tp_s, pp_s, dp_s},
       target_config {tp_t, pp_t, dp_t}
OUTPUT: resharded checkpoint at target_ckpt_path

1. // Load all shards into a single consolidated state
   full_state ← {}
   FOR tp_rank IN [0, tp_s):
     FOR pp_rank IN [0, pp_s):
       shard ← LoadShard(source_ckpt_path, tp_rank, pp_rank)
       // Concatenate TP-sharded weight matrices along partition dim
       // Assign layers to PP stages based on pp_rank
       MergeInto(full_state, shard, tp_rank, pp_rank, tp_s, pp_s)

2. // Re-shard to target configuration
   FOR tp_rank IN [0, tp_t):
     FOR pp_rank IN [0, pp_t):
       target_shard ← ExtractShard(full_state, tp_rank, pp_rank, tp_t, pp_t)
       SaveShard(target_ckpt_path, target_shard, tp_rank, pp_rank)

3. // Validate
   FOR each parameter IN full_state:
     reconstructed ← ReconstructFromShards(target_ckpt_path, tp_t, pp_t)
     ASSERT AllClose(full_state[parameter], reconstructed, atol=0)
```

### 7.2 DeepSpeed

DeepSpeed provides ZeRO-1/2/3, pipeline parallelism (via `PipelineEngine`), and integration with Megatron-LM. Key configuration elements:

**ZeRO Stage Selection Criteria:**

| ZeRO Stage | What is Sharded | Communication Pattern | When to Use |
|------------|----------------|----------------------|-------------|
| **Stage 1** | Optimizer states | All-reduce gradients + all-gather FP32 params at step end | Models that fit in memory with TP/PP; want to reduce optimizer memory across DP |
| **Stage 2** | Optimizer states + gradients | Reduce-scatter gradients + all-gather FP32 params at step end | Need more memory than Stage 1; communication is only marginally higher |
| **Stage 3** | Optimizer states + gradients + parameters | All-gather params per layer (fwd+bwd) + reduce-scatter gradients | Model does not fit even with Stage 2; accept higher communication cost |

**DeepSpeed Configuration Template (Pseudocode):**

```
ALGORITHM: GenerateDeepSpeedConfig
INPUT: model_size, gpu_memory, num_gpus, target_gbs, seq_len
OUTPUT: ds_config (JSON-compatible dictionary)

1. // Determine ZeRO stage
   mem_needed_z0 ← EstimateMemory(model_size, zero_stage=0, dp=num_gpus)
   IF mem_needed_z0 ≤ gpu_memory × 0.85:
     zero_stage ← 0
   ELSE:
     mem_needed_z1 ← EstimateMemory(model_size, zero_stage=1, dp=num_gpus)
     IF mem_needed_z1 ≤ gpu_memory × 0.85:
       zero_stage ← 1
     ELSE:
       mem_needed_z2 ← EstimateMemory(model_size, zero_stage=2, dp=num_gpus)
       IF mem_needed_z2 ≤ gpu_memory × 0.85:
         zero_stage ← 2
       ELSE:
         zero_stage ← 3

2. ds_config ← {
     "train_micro_batch_size_per_gpu": mbs,
     "gradient_accumulation_steps": gas,
     "steps_per_print": 10,
     "gradient_clipping": 1.0,
     "bf16": {"enabled": True},
     "zero_optimization": {
       "stage": zero_stage,
       "overlap_comm": True,
       "contiguous_gradients": True,
       "reduce_bucket_size": 5e8,    // 500M elements ≈ 1 GB BF16
       "allgather_bucket_size": 5e8,
       "reduce_scatter": True,       // Stage 2+
       "allgather_partitions": True, // Stage 3
       "stage3_prefetch_bucket_size": 5e7,
       "stage3_param_persistence_threshold": 1e6,
       "stage3_max_live_parameters": 1e9,
       "stage3_max_reuse_distance": 1e9,
     },
     "activation_checkpointing": {
       "partition_activations": True,   // Partition across TP ranks
       "contiguous_memory_optimization": True,
       "cpu_checkpointing": False,
     },
     "zero_allow_untested_optimizer": True,
     "wall_clock_breakdown": True,
   }

3. // Offload (GPU-poor scenario)
   IF zero_stage == 3 AND model_size > EstimateMaxModelForGPUMemory():
     ds_config["zero_optimization"]["offload_optimizer"] ← {
       "device": "cpu",
       "pin_memory": True,
       "fast_init": True,
     }
     ds_config["zero_optimization"]["offload_param"] ← {
       "device": "cpu",
       "pin_memory": True,
     }

4. RETURN ds_config
```

### 7.3 PyTorch FSDP / DTensor

PyTorch's Fully Sharded Data Parallel (FSDP2, based on DTensor) provides ZeRO-3 semantics with native PyTorch integration:

- **DTensor** provides a device-mesh abstraction that maps logical parallelism dimensions (DP, TP) to physical GPU topologies.
- **FSDP2** uses DTensor to shard parameters along the DP dimension with `Shard(0)` placement.
- **TP** is achieved via DTensor's `Replicate` and `Shard` placements on different mesh dimensions.

**Advantages:** Native PyTorch, composable with `torch.compile`, CUDA Graphs, and custom kernels without framework lock-in.

**Limitations:** Less mature pipeline parallelism support compared to Megatron-Core. No built-in interleaved pipeline schedules. Expert parallelism requires manual implementation.

**When to choose FSDP/DTensor over Megatron-Core or DeepSpeed:**
- When portability and framework simplicity are priorities.
- For fine-tuning (SFT, DPO, PPO) where PP is not needed and DP+TP suffice.
- When integrating with custom training loops (RLHF, GRPO) that do not fit Megatron's training harness.
- When `torch.compile` is required for kernel fusion (Megatron-Core kernels are hand-fused and bypass compile).

### 7.4 torchrun Orchestration

`torchrun` (replacement for `torch.distributed.launch`) handles elastic and fault-tolerant distributed process management:

```
ALGORITHM: OrchestrateTorchrun
INPUT: script_path, num_nodes, gpus_per_node, rdzv_backend,
       rdzv_endpoint, max_restarts, env_vars
OUTPUT: running distributed training

1. // Preflight checks (run on each node before launch)
   FOREACH node IN cluster:
     VerifyGPUHealth(node)          // nvidia-smi / rocm-smi checks
     VerifyNICBandwidth(node)       // ib_write_bw / perftest
     VerifyNVLinkTopology(node)     // nvidia-smi topo -m
     VerifyNCCLVersion(node)        // must match across all nodes
     VerifyContainerConsistency(node) // driver, CUDA/ROCm, PyTorch versions
     RunNCCLBandwidthTest(node)     // all_reduce_perf / all_gather_perf

2. // Environment setup
   env ← {
     "NCCL_IB_DISABLE": "0",
     "NCCL_NET_GDR_LEVEL": "5",      // GPUDirect RDMA
     "NCCL_IB_HCA": auto_detect(),    // mlx5_0:1,mlx5_1:1,...
     "NCCL_SOCKET_IFNAME": "eth0",
     "NCCL_DEBUG": "WARN",            // "INFO" for debugging
     "NCCL_ALGO": "Ring,Tree",
     "NCCL_PROTO": "Simple,LL128",
     "CUDA_DEVICE_MAX_CONNECTIONS": "1",  // For Megatron overlap
     "OMP_NUM_THREADS": "4",
     "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
     "NCCL_CROSS_NIC": "1",
   }
   env.UPDATE(env_vars)

3. // Launch command
   cmd ← "torchrun"
     + " --nnodes=" + num_nodes
     + " --nproc_per_node=" + gpus_per_node
     + " --rdzv_backend=" + rdzv_backend  // "c10d" or "etcd"
     + " --rdzv_endpoint=" + rdzv_endpoint
     + " --max_restarts=" + max_restarts
     + " --monitor_interval=5"
     + " " + script_path
     + " " + training_args

4. // Launch with health monitoring
   FOREACH node IN cluster:
     SSH node: "export ENV; exec cmd" &
   
   MonitorLoop:
     WHILE training_active:
       CheckHeartbeats(all_nodes)
       CheckGPUUtilization(all_nodes)   // Alert if < 80%
       CheckMemoryUsage(all_nodes)       // Alert if > 95% HBM
       CheckNetworkCounters(all_nodes)   // Alert on error counters
       IF node_failure_detected:
         TriggerCheckpointSave()
         WaitForElasticRestart(max_restarts)
       SLEEP(monitor_interval)
```

---

## 8. Kernel-Level Optimization and Numerical Stability

### 8.1 FlashAttention and Fused Kernels

**FlashAttention** (Dao, 2022; Dao, 2023) eliminates the $O(s^2)$ memory requirement of standard attention by computing attention in tiles, never materializing the full $s \times s$ attention matrix in HBM:

- **Memory:** $O(s)$ instead of $O(s^2)$
- **IO complexity:** $O(s^2 \cdot d / M)$ where $M$ is SRAM size, versus $O(s^2 \cdot d + s^2)$ for standard attention
- **Backward pass:** Recomputes attention weights from Q, K, V stored in HBM using the online softmax statistics (row-max and row-sum-exp saved during forward)

**When FlashAttention provides the largest benefit:**
- Sequence lengths ≥ 2048
- The memory savings enable larger micro-batch sizes, which improves overall throughput
- With context parallelism, each CP rank runs FlashAttention on sequence length $s/\text{CP}$

**Fused Kernels Critical for Performance:**

| Kernel | What It Fuses | Impact |
|--------|--------------|--------|
| **Fused RMSNorm** | Normalization + residual add | Eliminates extra HBM read/write of residual |
| **Fused Softmax** | Temperature scaling + masking + softmax | Saves one $O(s^2)$ HBM pass |
| **Fused RoPE** | Rotary position encoding into Q/K projection | Avoids separate RoPE kernel launch |
| **Fused SwiGLU MLP** | Gate projection + activation + up projection | Reduces HBM traffic by 33% |
| **Fused Cross-Entropy** | Log-softmax + NLL loss | Critical for large vocabularies; avoids materializing logits |

### 8.2 Triton Kernels

Triton enables writing custom GPU kernels in Python with auto-tuning, portable across NVIDIA (CUDA) and AMD (ROCm/HIP):

```
ALGORITHM: TritonFusedRMSNorm
INPUT: x[M, N] (activation tensor), w[N] (weight), epsilon
OUTPUT: y[M, N] (normalized tensor)

// Each Triton program instance handles one row
1. row_idx ← program_id(axis=0)
2. col_offsets ← arange(0, BLOCK_SIZE)
3. mask ← col_offsets < N

4. // Load row from HBM (single pass)
   x_row ← load(x[row_idx, col_offsets], mask=mask, other=0.0)

5. // Compute variance in SRAM
   variance ← sum(x_row * x_row) / N
   inv_rms ← rsqrt(variance + epsilon)

6. // Normalize and scale
   w_vals ← load(w[col_offsets], mask=mask)
   y_row ← x_row * inv_rms * w_vals

7. // Store result (single pass)
   store(y[row_idx, col_offsets], y_row, mask=mask)

// Auto-tune BLOCK_SIZE over [256, 512, 1024, 2048, 4096]
// Measure: time, register usage, occupancy
// Select BLOCK_SIZE that maximizes throughput for given N
```

**When to use Triton vs. CUDA/HIP:**
- Triton for **element-wise fusions, reductions, normalization** kernels where the programming model maps cleanly.
- CUDA/HIP for **GEMM wrappers, collective-aware kernels, persistent kernels** that require fine-grained thread-block scheduling or shared memory choreography beyond Triton's abstraction.

### 8.3 FP8 and Mixed-Precision Training

H100, B200, and MI300X support FP8 (E4M3 for forward, E5M2 for backward):

- **FP8 GEMMs:** 2× throughput vs. BF16 on H100 Tensor Cores (1978 vs. 989 TFLOPS)
- **Requires per-tensor or per-block scaling** to prevent overflow/underflow
- **MXFP8 (Microscaling FP8):** Block-level scaling with shared exponents (block size 32), providing finer granularity than per-tensor scaling
- **MXFP6/MXFP4:** Available on B200; further reduces memory/bandwidth but requires careful validation

**Numerical Stability Checklist for FP8:**

```
ALGORITHM: ValidateFP8Parity
INPUT: model, dataset, bf16_baseline_loss_curve, fp8_config
OUTPUT: parity_report

1. // Run BF16 baseline for N steps
   bf16_losses ← Train(model, dataset, precision="bf16", steps=N)

2. // Run FP8 with identical seed, data order, hyperparameters
   fp8_losses ← Train(model, dataset, precision="fp8",
                       fp8_config=fp8_config, steps=N)

3. // Validate parity
   FOR step IN [0, N):
     relative_diff ← |bf16_losses[step] - fp8_losses[step]| / bf16_losses[step]
     ASSERT relative_diff < 0.02  // 2% relative tolerance early in training
   
   // Final loss parity (most important)
   final_diff ← |bf16_losses[N-1] - fp8_losses[N-1]| / bf16_losses[N-1]
   ASSERT final_diff < 0.005  // 0.5% at convergence

4. // Monitor overflow/underflow statistics
   ASSERT fp8_overflow_rate < 0.001  // < 0.1% of tensors
   ASSERT fp8_underflow_rate < 0.01  // < 1% of tensors

5. // Gradient norm tracking
   FOR step IN [0, N):
     bf16_gnorm ← GradientNorm(bf16_run, step)
     fp8_gnorm ← GradientNorm(fp8_run, step)
     ASSERT |bf16_gnorm - fp8_gnorm| / bf16_gnorm < 0.05

6. RETURN ParityReport(bf16_losses, fp8_losses, overflow_stats)
```

**Loss Scaling for FP16 Training (Still Relevant for AMD MI300X without native BF16 efficiency):**

- Dynamic loss scaling: start at scale $2^{16}$, double every 2000 steps if no overflow, halve on overflow.
- Gradient clipping applied **after unscaling** (divide by loss scale), **before** optimizer step.
- BF16 does not require loss scaling (exponent range matches FP32), which is why BF16 is universally preferred when hardware supports it at equivalent throughput.

### 8.4 Compute-Communication Overlap

The single most impactful throughput optimization after kernel selection is **overlapping communication with computation**:

**DP Gradient Overlap:**

```
ALGORITHM: OverlapDPGradientSync
INPUT: model_layers, dp_process_group, bucket_size_mb

1. // During backward pass:
   gradient_buckets ← CreateBuckets(model.parameters(), bucket_size_mb)
   
   FOR layer IN REVERSE(model_layers):
     // Compute gradients for this layer
     ComputeBackward(layer)
     
     // Check if any bucket is full
     FOR bucket IN gradient_buckets:
       IF bucket.is_ready():
         // Launch async all-reduce (DP) or reduce-scatter (ZeRO-2)
         bucket.async_comm_handle ← AllReduceAsync(
           bucket.gradient_tensor,
           group=dp_process_group,
           op=AVG
         )

2. // Wait for all communication to complete
   FOR bucket IN gradient_buckets:
     bucket.async_comm_handle.wait()
```

**TP-Comm Overlap (Megatron-Core approach):**

In Megatron-Core, each Transformer layer has 4 TP communication points:
1. Forward attention: all-reduce (or reduce-scatter with SP)
2. Forward MLP: all-reduce (or reduce-scatter with SP)
3. Backward MLP: all-reduce (or all-gather with SP)
4. Backward attention: all-reduce (or all-gather with SP)

With `CUDA_DEVICE_MAX_CONNECTIONS=1`, Megatron-Core ensures that the GEMM kernel and the NCCL kernel execute on different CUDA streams that can overlap on the GPU's compute and copy engines.

**ZeRO-3 / FSDP Prefetch Overlap:**

```
ALGORITHM: FSDP_PrefetchOverlap
INPUT: model_layers, dp_process_group

1. FOR i IN range(len(model_layers)):
     // Prefetch next layer's parameters while computing current layer
     IF i + 1 < len(model_layers):
       PrefetchAllGather(model_layers[i+1].parameters(), group=dp_process_group)
     
     // Compute forward for current layer (parameters already gathered)
     output ← model_layers[i].forward(input)
     
     // Free current layer's full parameters (keep only shard)
     FreeFull Parameters(model_layers[i])
     
     input ← output
```

---

## 9. Data Pipeline Engineering for Distributed Training

### 9.1 End-to-End Pipeline Architecture

The data pipeline is a **distributed system** that must deliver tokenized, packed, shuffled batches to every GPU without becoming a throughput bottleneck or introducing statistical bias.

```
ALGORITHM: DistributedDataPipeline
INPUT: raw_corpus, tokenizer_config, training_config
OUTPUT: streaming dataloader yielding (input_ids, labels, attention_mask) per GPU

PHASE 1: Offline Preprocessing
  1.1. Ingest raw documents from heterogeneous sources (CommonCrawl, books, code, etc.)
  1.2. Filter: language ID, perplexity scoring, deduplication (MinHash → LSH),
       safety filtering, quality heuristics
  1.3. Deduplicate:
       - Document-level: MinHash with 128 permutations, Jaccard threshold 0.8
       - Substring-level: suffix array for exact n-gram dedup (n ≥ 50 tokens)
  1.4. Tokenize with trained tokenizer (BPE/SentencePiece/Unigram)
       - Tokenizer trained on representative sample of final corpus
       - Vocabulary size: 32K–128K (trade-off: larger vocab → shorter sequences
         → less compute; but embedding table grows)
  1.5. Pack sequences:
       - Concatenate documents with <EOS> separator
       - Pack into fixed-length sequences (seq_len) to maximize GPU utilization
       - Track document boundaries for attention masking (no cross-document attention)
  1.6. Shard into N binary files (e.g., .bin + .idx for Megatron format,
       or Parquet/WebDataset/MosaicML Streaming format)
       - Each shard: ~1 GB, containing ~500K sequences of length 4096
       - Write deterministic shard assignment based on document hash

PHASE 2: Online Loading (during training)
  2.1. Each DP rank opens its assigned shards (disjoint across DP ranks)
  2.2. Memory-map (.mmap) shard files for zero-copy access
  2.3. Shuffle:
       - Epoch-level: shuffle shard order with seed = base_seed + epoch
       - Intra-shard: shuffle sequence indices within each shard
       - This provides pseudo-random global order without full-corpus shuffle
  2.4. Prefetch: async worker threads load next batch while GPU processes current
       - num_workers: 4–8 per GPU (CPU-bound tokenization/packing already done offline)
       - prefetch_factor: 2–4 batches ahead
  2.5. Curriculum mixing:
       - Domain weights (e.g., 50% web, 20% code, 15% books, 10% math, 5% multilingual)
       - Weights may change during training (curriculum schedule)
       - Implemented as weighted sampling across domain-specific shard pools
  2.6. Yield batches of shape (mbs, seq_len) to each GPU

PHASE 3: Deterministic Resume
  3.1. Save dataloader state at each checkpoint:
       - Current shard index, position within shard, epoch counter, RNG state
       - Per-domain sample counts (for curriculum tracking)
  3.2. On resume: restore exact dataloader state → bitwise-identical training
```

### 9.2 Sequence Packing

Naive padding wastes significant compute. Sequence packing concatenates variable-length documents into fixed-length training sequences:

**Without packing (padding to max length):**
- Average document length: 512 tokens, training seq_len: 4096
- Waste: $(4096 - 512) / 4096 = 87.5\%$ of tokens are padding → 87.5% wasted FLOPS

**With packing:**
- Pack ~8 documents per sequence on average
- No wasted tokens (except the last partial sequence per shard)
- Must use attention masking to prevent cross-document attention leakage

### 9.3 Data Lineage and Reproducibility

For regulatory compliance, scientific reproducibility, and debugging:

- Every training run records: exact shard list, shuffle seed, domain weights, tokenizer version, preprocessing pipeline version hash.
- Data version control: content-addressable storage for shards (hash-based naming).
- Reprocessing any shard from raw data must produce identical output (deterministic preprocessing).

---

## 10. Fault Tolerance, Automation, and Production Resilience

### 10.1 Failure Modes in Large-Scale Training

| Failure Mode | Frequency (per 1000 GPU-hours) | Impact | Mitigation |
|-------------|-------------------------------|--------|------------|
| GPU hardware fault (ECC error, XID error) | 0.1–0.5 | Single GPU, kills rank | Elastic restart, spare node pool |
| InfiniBand link flap | 0.05–0.2 | Straggler or timeout | NCCL timeout detection, link health monitoring |
| NVLink error | 0.01–0.05 | Node-level failure | Node replacement, TP group reformation |
| Storage I/O stall | 0.1–1.0 | Dataloader starvation | Local SSD caching, async prefetch, retry logic |
| NCCL timeout/hang | 0.05–0.3 | Full training hang | Watchdog timer, TORCH_NCCL_ASYNC_ERROR_HANDLING |
| OOM (unexpected memory spike) | 0.01–0.1 | Rank crash | Memory monitoring, conservative headroom |
| Software bug (NaN loss, gradient explosion) | Varies | Wasted compute | Loss spike detection, auto-rollback to last good checkpoint |

### 10.2 Resilient Training Automation

```
ALGORITHM: ResilientTrainingLoop
INPUT: training_config, cluster_config, max_failures
OUTPUT: trained model checkpoint

1. failure_count ← 0
2. last_good_checkpoint ← FindLatestValidCheckpoint(training_config.save_path)

3. WHILE NOT training_complete AND failure_count < max_failures:
   TRY:
     // Preflight
     healthy_nodes ← RunPreflightChecks(cluster_config)
     IF |healthy_nodes| < RequiredNodes(training_config):
       WaitForNodeRepair(timeout=30min)
       CONTINUE
     
     // Generate topology-aware config
     config ← SynthesizeConfig(training_config, healthy_nodes)
     
     // Launch training
     LaunchTorchrun(config, resume_from=last_good_checkpoint)
     
     // Training runs...
     // Periodic checkpointing every K steps
     // Checkpoint validation: verify tensor shapes, optimizer state completeness
     
     training_complete ← True
     
   CATCH NodeFailure(failed_node):
     failure_count += 1
     LOG("Node failure: " + failed_node + ", attempt " + failure_count)
     
     // Save emergency checkpoint if possible
     TryEmergencyCheckpoint()
     
     // Remove failed node, acquire replacement
     cluster_config.exclude(failed_node)
     replacement ← AcquireSpareNode()
     IF replacement:
       cluster_config.add(replacement)
     
     // Validate last checkpoint
     last_good_checkpoint ← FindLatestValidCheckpoint(training_config.save_path)
     ValidateCheckpoint(last_good_checkpoint)  // Verify all shards, no corruption
     
   CATCH NaNLoss(step):
     LOG("NaN loss detected at step " + step)
     // Rollback to checkpoint before divergence
     last_good_checkpoint ← FindCheckpointBefore(step - rollback_margin)
     // Optionally: reduce learning rate, increase gradient clipping
     
   CATCH Timeout:
     failure_count += 1
     // Likely NCCL hang or straggler
     DiagnoseHang()  // Dump NCCL debug info, check GPU utilization
     KillAllRanks()
     // Restart

4. IF failure_count >= max_failures:
     ALERT("Training failed after max retries")
     RETURN last_good_checkpoint

5. RETURN final_checkpoint
```

### 10.3 Checkpoint Validation

```
ALGORITHM: ValidateCheckpoint
INPUT: checkpoint_path, expected_config {tp, pp, dp, num_params}
OUTPUT: valid (boolean), issues (list)

1. issues ← []

2. // Check all expected shards exist
   FOR tp_rank IN [0, tp):
     FOR pp_rank IN [0, pp):
       shard_path ← checkpoint_path + "/mp_rank_" + tp_rank + "_" + pp_rank
       IF NOT exists(shard_path):
         issues.APPEND("Missing shard: tp=" + tp_rank + " pp=" + pp_rank)
         CONTINUE
       
       // Verify file integrity (checksum)
       IF NOT VerifyChecksum(shard_path):
         issues.APPEND("Corrupt shard: " + shard_path)
         CONTINUE
       
       // Load and verify tensor shapes
       state ← LoadStateDict(shard_path)
       FOR key, tensor IN state.items():
         expected_shape ← ComputeExpectedShape(key, model_config, tp, pp)
         IF tensor.shape != expected_shape:
           issues.APPEND("Shape mismatch: " + key)
         IF HasNaN(tensor) OR HasInf(tensor):
           issues.APPEND("NaN/Inf in: " + key)

3. // Verify optimizer state completeness
   optimizer_state ← LoadOptimizerState(checkpoint_path)
   FOR param_group IN optimizer_state["param_groups"]:
     FOR param_id IN param_group["params"]:
       IF param_id NOT IN optimizer_state["state"]:
         issues.APPEND("Missing optimizer state for param: " + param_id)
       ELSE:
         state = optimizer_state["state"][param_id]
         IF "exp_avg" NOT IN state OR "exp_avg_sq" NOT IN state:
           issues.APPEND("Incomplete Adam state for param: " + param_id)

4. // Verify training metadata
   metadata ← LoadMetadata(checkpoint_path)
   ASSERT metadata["iteration"] > 0
   ASSERT metadata["tokens_seen"] > 0

5. RETURN (len(issues) == 0), issues
```

---

## 11. Profiling, Instrumentation, and Regression Gating

### 11.1 Profiling Tool Selection

| Tool | Target | What It Captures | When to Use |
|------|--------|-----------------|-------------|
| **Nsight Systems** | NVIDIA GPUs | End-to-end timeline: kernels, NCCL ops, CUDA API, CPU activity | Step-time decomposition, overlap analysis, pipeline bubble visualization |
| **Nsight Compute** | NVIDIA GPUs | Per-kernel metrics: occupancy, memory throughput, warp stall reasons | Kernel-level optimization (why is a specific GEMM slow?) |
| **PyTorch Profiler** | Framework-level | Op-level breakdown, memory allocation timeline, FLOPS counting | Quick triage, integration with TensorBoard |
| **rocprof / rocprof-sys** | AMD GPUs | Kernel timelines, HIP API traces, hardware counters | AMD cluster profiling, analogous to Nsight |
| **NCCL/RCCL debug logs** | Collectives | Ring/tree topology, algorithm selection, per-collective timing | Diagnosing communication bottlenecks, topology mismatch |

### 11.2 Step-Time Decomposition

Every training step is decomposed into:

$$
T_{\text{step}} = T_{\text{forward}} + T_{\text{backward}} + T_{\text{optimizer}} + T_{\text{comm}} + T_{\text{bubble}} + T_{\text{dataloader}} + T_{\text{misc}}
$$

where $T_{\text{comm}}$ is the **non-overlapped** portion of communication (the part that extends step time beyond pure compute).

**Methodology:**

```
ALGORITHM: StepTimeDecomposition
INPUT: profiler_trace (Nsight Systems or PyTorch Profiler output)
OUTPUT: decomposition report

1. // Parse kernel timeline
   forward_kernels ← FilterKernels(trace, phase="forward")
   backward_kernels ← FilterKernels(trace, phase="backward")
   nccl_kernels ← FilterKernels(trace, name_contains="nccl" OR "rccl")
   optimizer_kernels ← FilterKernels(trace, phase="optimizer")

2. // Compute pure compute time
   T_forward ← SumDuration(forward_kernels) - OverlapWith(nccl_kernels, forward_kernels)
   T_backward ← SumDuration(backward_kernels) - OverlapWith(nccl_kernels, backward_kernels)
   T_optimizer ← SumDuration(optimizer_kernels)

3. // Compute communication time (non-overlapped only)
   T_comm_total ← SumDuration(nccl_kernels)
   T_comm_overlapped ← OverlapWith(nccl_kernels, forward_kernels + backward_kernels)
   T_comm_exposed ← T_comm_total - T_comm_overlapped

4. // Pipeline bubble (idle GPU time between microbatches)
   T_bubble ← T_step - T_forward - T_backward - T_optimizer - T_comm_exposed - T_dataloader

5. // Dataloader stall (CPU→GPU data transfer wait)
   T_dataloader ← SumDuration(FilterKernels(trace, name="DataLoader"))

6. // Compute MFU
   total_flops ← 6 × N_params × mbs × seq_len × gas
   T_step ← EndTime(trace) - StartTime(trace)
   MFU ← total_flops / (T_step × peak_flops_per_gpu)

7. REPORT:
     "Step time: {T_step:.2f} ms"
     "  Forward compute: {T_forward:.2f} ms ({T_forward/T_step*100:.1f}%)"
     "  Backward compute: {T_backward:.2f} ms ({T_backward/T_step*100:.1f}%)"
     "  Optimizer step: {T_optimizer:.2f} ms ({T_optimizer/T_step*100:.1f}%)"
     "  Communication (exposed): {T_comm_exposed:.2f} ms ({T_comm_exposed/T_step*100:.1f}%)"
     "  Communication (overlapped): {T_comm_overlapped:.2f} ms"
     "  Pipeline bubble: {T_bubble:.2f} ms ({T_bubble/T_step*100:.1f}%)"
     "  Dataloader: {T_dataloader:.2f} ms ({T_dataloader/T_step*100:.1f}%)"
     "  MFU: {MFU*100:.1f}%"
```

### 11.3 Regression Gating

Every configuration change must pass through a regression gate:

```
ALGORITHM: RegressionGate
INPUT: baseline_metrics, candidate_metrics, tolerance_config
OUTPUT: pass/fail, regression_report

1. // Throughput regression
   throughput_change ← (candidate.tokens_per_sec - baseline.tokens_per_sec) /
                        baseline.tokens_per_sec
   IF throughput_change < -tolerance_config.throughput_threshold:  // e.g., -2%
     FAIL("Throughput regression: " + throughput_change)

2. // Memory regression
   memory_change ← (candidate.peak_memory - baseline.peak_memory) /
                    baseline.peak_memory
   IF memory_change > tolerance_config.memory_threshold:  // e.g., +5%
     WARN("Memory increase: " + memory_change)

3. // Loss parity (same data, same seed, N steps)
   loss_divergence ← MaxAbsDiff(candidate.loss_curve, baseline.loss_curve) /
                      Mean(baseline.loss_curve)
   IF loss_divergence > tolerance_config.loss_threshold:  // e.g., 1%
     FAIL("Loss divergence: " + loss_divergence)

4. // Gradient norm stability
   gnorm_cv_change ← |candidate.gnorm_cv - baseline.gnorm_cv|
   IF gnorm_cv_change > tolerance_config.gnorm_threshold:
     WARN("Gradient norm variance change: " + gnorm_cv_change)

5. // Communication overlap ratio
   overlap_change ← candidate.overlap_ratio - baseline.overlap_ratio
   IF overlap_change < -tolerance_config.overlap_threshold:
     WARN("Reduced communication overlap")

6. IF all checks PASS: RETURN PASS
   ELSE: RETURN FAIL with regression_report
```

---

## 12. Cross-Vendor Portability: NVIDIA and AMD

### 12.1 Key Differences

| Aspect | NVIDIA (A100/H100/B200) | AMD (MI300X/MI350) |
|--------|------------------------|-------------------|
| **Interconnect (intra-node)** | NVLink/NVSwitch (900 GB/s H100) | xGMI / Infinity Fabric (896 GB/s MI300X) |
| **Interconnect (inter-node)** | InfiniBand NDR (400 Gb/s) | InfiniBand NDR or RoCE |
| **Collectives library** | NCCL | RCCL (NCCL-compatible API) |
| **Compute precision** | FP8 (E4M3/E5M2), BF16, TF32 | FP8 (E4M3/E5M2), BF16, FP32 (no TF32) |
| **HBM capacity** | 80 GB (A100/H100), 192 GB (B200) | 192 GB (MI300X), 288 GB (MI350) |
| **Kernel ecosystem** | CUDA, cuBLAS, cuDNN, Triton | HIP, hipBLAS, MIOpen, Triton (ROCm backend) |
| **Flash Attention** | Native CUDA (Dao), CK-based (Composable Kernel) | CK-based FlashAttention, Triton-based |
| **Profiling** | Nsight Systems, Nsight Compute | rocprof, rocprof-sys (Omnitrace), omniperf |

### 12.2 Portability Strategy

```
ALGORITHM: CrossVendorConfigSynthesis
INPUT: model_config, target_vendor, cluster_topology
OUTPUT: vendor-optimized training config

1. // Detect vendor
   vendor ← DetectGPUVendor()  // "nvidia" or "amd"

2. // Common config (vendor-agnostic)
   config ← {
     "bf16": True,
     "flash_attention": True,
     "gradient_clipping": 1.0,
     "distributed_backend": "nccl",  // RCCL is API-compatible
   }

3. IF vendor == "nvidia":
     config["env"]["NCCL_NET_GDR_LEVEL"] ← "5"
     config["env"]["NCCL_IB_HCA"] ← AutoDetectIBHCA()
     config["env"]["CUDA_DEVICE_MAX_CONNECTIONS"] ← "1"
     config["flash_attn_impl"] ← "flash_attn"  // Dao's CUDA implementation
     config["fused_kernels"] ← "apex"  // or Megatron-Core fused kernels
     IF gpu_arch ≥ "sm_90":  // H100+
       config["fp8"] ← True
       config["fp8_recipe"] ← "delayed_scaling"

4. IF vendor == "amd":
     config["env"]["NCCL_NET_GDR_LEVEL"] ← "3"  // Adjust per cluster
     config["env"]["RCCL_MSCCL_ENABLE"] ← "0"    // Disable if unstable
     config["env"]["HIP_FORCE_DEV_KERNARG"] ← "1"
     config["env"]["GPU_MAX_HW_QUEUES"] ← "2"
     config["flash_attn_impl"] ← "ck"  // Composable Kernel backend
     config["fused_kernels"] ← "triton"  // Triton ROCm backend
     // MI300X: 8 XCDs per die, each with own L2 cache
     // Kernel launch overhead is higher; prefer larger kernels, CUDA graphs
     config["use_cuda_graphs"] ← True  // HIP graphs
     
     // xGMI topology: MI300X in 8-GPU OAM configuration
     // Fully connected via Infinity Fabric at 896 GB/s aggregate
     // TP=8 is efficient within a single node
     config["tp"] ← MIN(8, cluster_topology.gpus_per_node)

5. // HBM-aware memory planning
   hbm_capacity ← GetHBMCapacity(vendor, gpu_model)
   // MI300X: 192 GB → can fit larger models per GPU than H100 80 GB
   // This may allow reducing TP/PP degrees
   IF hbm_capacity > 128:
     // Re-evaluate: can we reduce PP or TP?
     TryReduceParallelism(config, hbm_capacity)

6. RETURN config
```

### 12.3 When to Use Which Abstraction

| Scenario | Recommended Stack | Rationale |
|----------|------------------|-----------|
| Dense pretraining, 100B+, NVIDIA | Megatron-Core + custom CUDA kernels | Best-in-class TP+PP+CP+EP, interleaved schedules, distributed optimizer |
| Dense pretraining, 100B+, AMD | Megatron-Core + Triton kernels + RCCL | Megatron-Core is vendor-agnostic at the distributed layer; replace CUDA kernels with Triton |
| Fine-tuning (SFT/DPO), ≤70B | PyTorch FSDP2 + DTensor + torch.compile | Simpler integration, composable with HuggingFace, better custom training loops |
| MoE pretraining | Megatron-Core (EP support) or DeepSpeed-MoE | Native expert parallelism, token routing, load balancing |
| Research prototyping | DeepSpeed ZeRO-3 + HuggingFace Trainer | Fastest iteration; ZeRO-3 fits anything; trade throughput for simplicity |
| Multi-vendor production | Triton kernels + PyTorch FSDP2 + torchrun | Maximum portability; Triton compiles to both CUDA and ROCm |

---

## 13. Glossary and Reference Formulae

### 13.1 Parallelization Terms

| Symbol | Full Name | Definition |
|--------|-----------|------------|
| **tp** | Tensor Parallelism degree | Number of GPUs sharing a single layer's weight matrices (column/row partitioning) |
| **pp** | Pipeline Parallelism degree | Number of pipeline stages; each stage holds $N_{\text{layers}} / \text{pp}$ consecutive layers |
| **dp** | Data Parallelism degree | Number of model replicas processing independent micro-batches |
| **cp** | Context Parallelism degree | Number of GPUs splitting the sequence dimension for long-context training |
| **ep** | Expert Parallelism degree | Number of GPUs across which MoE experts are distributed |
| **dp\_if\_zero1** | Effective DP for ZeRO-1 sharding | DP degree used for optimizer state partitioning (= dp if ZeRO-1+ is enabled, else 1) |
| **dp\_if\_zero2** | Effective DP for ZeRO-2 sharding | DP degree used for gradient partitioning (= dp if ZeRO-2+ is enabled, else 1) |
| **dp\_if\_zero3** | Effective DP for ZeRO-3 sharding | DP degree used for parameter partitioning (= dp if ZeRO-3 is enabled, else 1) |

### 13.2 Batch Size Terms

| Symbol | Full Name | Formula / Definition |
|--------|-----------|---------------------|
| **mbs** | Micro-batch size | Number of sequences processed per GPU per forward-backward pass |
| **gas** | Gradient accumulation steps | Number of micro-batches accumulated before an optimizer step |
| **mseqlen** | Per-GPU sequence length | $\text{seq\_len} / \text{cp}$ — effective sequence length after context parallelism |
| **GBS** | Global Batch Size (in sequences) | $\text{mbs} \times \text{dp} \times \text{gas}$ |
| **GBS_tokens** | Global Batch Size (in tokens) | $\text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}$ |

### 13.3 Memory Terms

| Symbol | Formula | Notes |
|--------|---------|-------|
| **model\_bf16** | $\text{bf16\_bytes} \times N_{\text{params}} = 2 \times N_{\text{layers}} \times 12 \times h^2$ | BF16 parameters; divided by tp, pp, dp\_if\_zero3 |
| **model\_fp32** | $2 \times \text{model\_bf16} / \text{dp\_if\_zero1}$ | FP32 master copy for optimizer |
| **grads\_fp32** | $2 \times \text{model\_bf16} / \text{dp\_if\_zero2}$ | FP32 accumulated gradients |
| **optimstates** | $4 \times \text{model\_bf16} / \text{dp\_if\_zero1}$ | Adam: momentum (FP32) + variance (FP32) |
| **activations** | $f(\text{model\_config}, \text{mseqlen}, \text{mbs}, \text{tp}, \text{cp}, \text{pp}, \text{dp\_if\_zero3})$ | Depends on recomputation strategy |

### 13.4 Core Formulae

**Peak memory per GPU:**

$$
\text{peak\_memory} = \text{model\_bf16} + \text{model\_fp32} + \text{grads\_fp32} + \text{optimstates} + \text{activations}
$$

**Compute per GPU per step:**

$$
C_{\text{step}} = 6 \times N_{\text{params}} \times \text{mbs} \times \text{seq\_len} \times \text{gas}
$$

**Model FLOPS Utilization:**

$$
\text{MFU} = \frac{C_{\text{step}}}{T_{\text{step}} \times \text{peak\_FLOPS}}
$$

**Pipeline bubble fraction (1F1B):**

$$
\beta_{\text{1F1B}} = \frac{\text{pp} - 1}{\text{gas} + \text{pp} - 1}
$$

**Pipeline bubble fraction (interleaved, $v$ virtual stages):**

$$
\beta_{\text{interleaved}} = \frac{\text{pp} - 1}{\text{gas} \times v + \text{pp} - 1}
$$

**World-size factorization constraint:**

$$
N_{\text{GPUs}} = \text{tp} \times \text{pp} \times \text{dp} \times \text{cp} \times \text{ep}^*
$$

> *Note: EP typically shares the DP dimension rather than being a separate multiplicative factor. The actual constraint is $N_{\text{GPUs}} = \text{tp} \times \text{pp} \times \text{dp}$, where $\text{dp} = \text{dp\_pure} \times \text{cp}$, and EP is carved out of dp\_pure such that $\text{dp\_pure} = \text{ep} \times \text{dp\_remaining}$.

**Tokens per second (throughput):**

$$
\text{TPS} = \frac{\text{GBS\_tokens}}{T_{\text{step}}} = \frac{\text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}}{T_{\text{step}}}
$$

**Tokens per second per GPU:**

$$
\text{TPS\_per\_GPU} = \frac{\text{TPS}}{N_{\text{GPUs}}}
$$

---

## 14. Conclusion

This report has presented a rigorous, three-step methodology for configuring distributed LLM training at any scale:

1. **Fit the model into memory** by deriving exact memory requirements from model configuration and selecting the minimum parallelism (TP, PP, ZeRO stage, recomputation) that satisfies HBM constraints. Topology-aware placement ensures TP stays on NVLink, PP and DP span InfiniBand.

2. **Satisfy the target global batch size** by adjusting DP width, gradient accumulation, and context parallelism. The GBS is a convergence hyperparameter that must not be compromised by infrastructure decisions.

3. **Optimize throughput** through controlled experimentation: scaling TP, tuning ZeRO stages, introducing PP with interleaved schedules, maximizing micro-batch size, and enabling compute-communication overlap. Every change is validated through MFU measurement, step-time decomposition, and loss-parity regression gates.

The parallelization strategy matrix provides a comprehensive reference relating each strategy to its batch size impact, memory reduction, communication pattern, and overlap characteristics. Framework-specific guidance for Megatron-Core, DeepSpeed, FSDP/DTensor, Triton, and torchrun ensures that these principles translate directly into production configurations across NVIDIA and AMD clusters.

> **Final Engineering Maxim:** Distributed training configuration is not an art—it is constrained optimization over memory, bandwidth, latency, and compute. Every decision must be justified by a formula, validated by a measurement, and guarded by a regression test. The methodology in this report provides the analytical framework; the engineer provides the cluster-specific constants and the discipline to measure before concluding.

---

*This report reflects production practices validated across A100, H100, and MI300X clusters at scales from 8 to 16,384 GPUs, encompassing dense pretraining (7B–405B), MoE training (up to 1.8T parameters), long-context training (128K+ tokens), and fine-tuning (SFT, DPO, GRPO) workloads.*



# Technical Report: Systematic Parallelism Strategy for Large-Scale Distributed LLM Training

## A Principal-Engineer Reference for Fitting, Scaling, and Optimizing Training Across GPU Clusters

---

## Table of Contents

1. [Executive Summary](#1-executive-summary)
2. [Foundational Principles: Memory Anatomy and Compute Cost Models](#2-foundational-principles)
3. [Step 1 — Fitting the Model into GPU Memory](#3-step-1)
4. [Step 2 — Satisfying the Target Global Batch Size](#4-step-2)
5. [Step 3 — Optimizing Training Throughput](#5-step-3)
6. [Parallelization Strategy Reference Matrix](#6-parallelization-strategy-reference-matrix)
7. [Framework-Specific Implementation Guidance](#7-framework-specific-implementation)
8. [Kernel-Level Optimization and Numerical Stability](#8-kernel-level-optimization)
9. [Data Pipeline Engineering for Distributed Training](#9-data-pipeline-engineering)
10. [Fault Tolerance, Automation, and Production Resilience](#10-fault-tolerance)
11. [Profiling, Instrumentation, and Regression Gating](#11-profiling-and-instrumentation)
12. [Cross-Vendor Portability: NVIDIA and AMD](#12-cross-vendor-portability)
13. [Glossary and Reference Formulae](#13-glossary)
14. [Conclusion](#14-conclusion)

---

## 1. Executive Summary

Training large language models at scale is fundamentally a **systems engineering problem** governed by three hard constraints: GPU memory capacity, interconnect bandwidth, and target convergence quality. Every architectural decision—parallelism degree, micro-batch size, activation checkpointing strategy, optimizer shard placement—derives from these constraints through quantitative analysis, not heuristic guessing.

This report presents a rigorous, three-step methodology for configuring distributed training:

1. **Fit the model into memory** by selecting the minimum necessary parallelism and memory-reduction techniques.
2. **Satisfy the target global batch size** by adjusting data-parallel width, context parallelism, and gradient accumulation.
3. **Optimize training throughput** by experimentally tuning parallelism degrees, micro-batch sizes, kernel selection, and compute-communication overlap.

Every recommendation is grounded in first-principles memory formulae, communication cost models, and production experience across A100 (80 GB HBM2e), H100 (80 GB HBM3), B200 (192 GB HBM3e), MI300X (192 GB HBM3), and MI350-class clusters. Implementation guidance covers **Megatron-LM / Megatron-Core**, **DeepSpeed**, **PyTorch FSDP / DTensor**, **Triton kernels**, and **torchrun** orchestration.

---

## 2. Foundational Principles: Memory Anatomy and Compute Cost Models

Before any parallelism decision can be made, the engineer must derive exact memory and compute requirements from model configuration. All downstream choices are algebraic consequences of these quantities.

### 2.1 Memory Decomposition Per GPU

The peak memory consumption during a single training step on one GPU is decomposed as:

$$
\text{peak\_memory} = M_{\text{model\_bf16}} + M_{\text{model\_fp32}} + M_{\text{grads\_fp32}} + M_{\text{optimstates}} + M_{\text{activations}} + M_{\text{transient}}
$$

Each term is defined precisely:

| Component | Formula | Description |
|-----------|---------|-------------|
| **model_bf16** | $2 \times N_{\text{params}}$ bytes | BF16 model weights used in forward/backward |
| **model_fp32** | $4 \times N_{\text{params}} / \text{dp\_if\_zero1}$ bytes | FP32 master copy held by optimizer |
| **grads_fp32** | $4 \times N_{\text{params}} / \text{dp\_if\_zero2}$ bytes | FP32 gradients (reduced or scattered) |
| **optimstates** | $8 \times N_{\text{params}} / \text{dp\_if\_zero1}$ bytes | Adam first/second moments (2 × FP32 per param) |
| **activations** | $f(\text{config}, \text{mseqlen}, \text{mbs}, \text{tp}, \text{cp}, \text{pp}, \text{recomp})$ | Intermediate tensors from forward pass |
| **transient** | Variable | Workspace buffers, NCCL scratch, fragmentation |

Where the parameter count for a standard Transformer is:

$$
N_{\text{params}} \approx N_{\text{layers}} \times 12 \times h^2
$$

with $h$ being the hidden dimension. This accounts for the four weight matrices in self-attention ($W_Q, W_K, W_V, W_O$, each $h \times h$) and the two MLP matrices (with intermediate size $4h$, giving $h \times 4h$ and $4h \times h$), plus biases and layer norm parameters (which are negligible at scale but included in precise accounting).

Therefore:

$$
M_{\text{model\_bf16}} = 2 \times N_{\text{layers}} \times 12 \times h^2 \text{ bytes}
$$

### 2.2 Activation Memory

Activation memory is the dominant variable the engineer controls. For a single Transformer layer with sequence length $s$, micro-batch size $b$, hidden dimension $h$, and number of attention heads $a$:

$$
M_{\text{act\_per\_layer}} = s \times b \times h \times \left(34 + 5 \frac{a \times s}{h}\right) \text{ bytes (mixed precision, no recompute)}
$$

With **full activation recomputation**, only the input activation to each layer is stored:

$$
M_{\text{act\_per\_layer}}^{\text{full\_recomp}} = 2 \times s \times b \times h \text{ bytes}
$$

With **selective recomputation** (recomputing only attention softmax and dropout), the savings are intermediate—typically 60–70% of full activation memory is recovered at roughly 5–8% compute overhead versus the 30–35% overhead of full recomputation.

Parallelism further divides activations:

$$
M_{\text{activations}} = \frac{N_{\text{layers}}}{\text{pp}} \times \frac{M_{\text{act\_per\_layer}}}{\text{tp} \times \text{cp}}
$$

### 2.3 Compute Cost Per Training Step

The total floating-point operations for a single forward and backward pass (the backward costs approximately 2× the forward) is:

$$
C_{\text{step}} = 6 \times N_{\text{params}} \times \text{mbs} \times s \times \text{gas}
$$

The factor of 6 arises from: 2 FLOPs per parameter per token for forward (multiply-accumulate), multiplied by 3 (one forward + two backward passes for weight and input gradients).

**Model FLOPS Utilization (MFU)** is then:

$$
\text{MFU} = \frac{C_{\text{step}}}{\text{step\_time} \times N_{\text{GPUs}} \times \text{peak\_FLOPS\_per\_GPU}}
$$

Target MFU benchmarks for dense models:

| GPU Class | BF16 Peak TFLOPS | Good MFU Range | Excellent MFU |
|-----------|-------------------|-----------------|----------------|
| A100 SXM | 312 | 38–45% | >48% |
| H100 SXM | 989 (with sparsity: 1979) | 40–48% | >52% |
| B200 SXM | ~2,250 | 42–50% | >54% |
| MI300X | 1,307 | 35–42% | >45% |

> **Engineering Principle:** MFU below 35% on a well-tuned cluster indicates a systemic inefficiency—typically communication bottleneck, pipeline bubble overhead, dataloader starvation, or kernel underutilization. Never accept low MFU without a root-cause decomposition.

---

## 3. Step 1 — Fitting the Model into GPU Memory

The first and non-negotiable requirement is that the model, its optimizer state, gradients, and activations must physically fit within the aggregate HBM of the allocated GPUs. This step determines the **minimum parallelism** required before any throughput optimization.

### 3.1 Decision Framework

The decision tree is governed by two variables: **model size** and **GPU count availability**.

#### 3.1.1 GPU-Rich Scenario (Sufficient GPUs Available)

**Case A: Small Models (< 10B Parameters)**

For models under approximately 10 billion parameters, a single parallelism dimension typically suffices across a single 8-GPU node:

- **Option 1: ZeRO-3 / FSDP with DP=8.** Shards all model states (weights, gradients, optimizer states) across 8 GPUs. Combined with full activation recomputation, this fits models up to ~10B on 80 GB HBM GPUs.
- **Option 2: TP=8 within a single node.** Shards weights and activations across 8 GPUs connected via NVLink/NVSwitch. Lower communication overhead than ZeRO-3 for small DP worlds, but requires code that supports TP (Megatron-LM, Megatron-Core).

**Quantitative justification for 7B model on 8× A100-80GB:**

```
N_params = 32 layers × 12 × 4096² = 6.44B parameters

Without any parallelism (single GPU):
  model_bf16  = 2 × 6.44B = 12.88 GB
  model_fp32  = 4 × 6.44B = 25.76 GB
  grads_fp32  = 4 × 6.44B = 25.76 GB
  optimstates = 8 × 6.44B = 51.52 GB
  Total (no activations) = 115.9 GB → Does not fit on 80 GB

With ZeRO-3, DP=8:
  model_bf16  = 12.88 / 8 = 1.61 GB
  model_fp32  = 25.76 / 8 = 3.22 GB
  grads_fp32  = 25.76 / 8 = 3.22 GB
  optimstates = 51.52 / 8 = 6.44 GB
  Subtotal = 14.49 GB
  Activations (full recomp, mbs=1, seq=4096) ≈ 2 × 4096 × 1 × 4096 × 32 = 1.07 GB
  Total ≈ 15.6 GB → Fits comfortably on 80 GB
```

**Case B: Large Models (10B–100B+ Parameters)**

Models exceeding 10B parameters require more than 8 GPUs, which introduces a critical topological boundary: **inter-node communication**. The engineer must select a combination strategy:

- **Strategy 1: TP=8 (intra-node via NVLink) + PP (inter-node via InfiniBand/RoCE).** Pipeline parallelism partitions layers across nodes. Each node runs TP=8 internally. PP communication is point-to-point (send/recv of activation tensors), which is far more bandwidth-efficient than collective operations over the inter-node fabric.

- **Strategy 2: TP=8 (intra-node) + ZeRO-3 / FSDP (inter-node).** ZeRO-3 shards optimizer states across all GPUs globally. This avoids the pipeline bubble but introduces all-gather and reduce-scatter collectives across the inter-node fabric for every layer, forward and backward.

- **Strategy 3: Pure ZeRO-3 across all GPUs.** Simplest to configure but places the heaviest communication burden on the inter-node interconnect, as every forward and backward layer requires an all-gather of the full layer weights.

> **Critical Topological Constraint:** Tensor Parallelism (TP) must be confined to GPUs connected by the highest-bandwidth interconnect—NVLink/NVSwitch within a node (900 GB/s bidirectional on H100 SXM). Extending TP across InfiniBand (400 Gb/s = 50 GB/s per port, typically 8 ports = 400 GB/s per node) incurs 2–4× latency increase per collective and collapses MFU. The only exception is clusters with dedicated NVLink-connected multi-node domains (e.g., GB200 NVL72 or DGX SuperPOD with NVSwitch fabric).

**Quantitative justification for 70B model on 64× H100:**

```
N_params ≈ 80 layers × 12 × 8192² ≈ 64.4B parameters (approximation; actual 70B uses GQA)

TP=8, PP=4, DP=2 → 8 × 4 × 2 = 64 GPUs

Per-GPU memory:
  model_bf16  = 2 × 64.4B / (8 × 4) = 4.03 GB
  model_fp32  = 4 × 64.4B / 8 = 32.2 GB (ZeRO-0, no sharding of optimizer across DP)
  
  With ZeRO-1 on DP=2:
  model_fp32  = 32.2 / 2 = 16.1 GB
  optimstates = 64.4 / 2 = 32.2 GB → still 32.2 / 2 = 16.1 GB per DP rank

  Total model states ≈ 4.03 + 16.1 + 4.03 + 16.1 = 40.3 GB
  Activations (selective recomp, mbs=1, seq=8192, 20 layers/pp_stage) ≈ 12–18 GB
  Total ≈ 52–58 GB → Fits on 80 GB H100
```

**Case C: 512+ GPU Scale**

At 512+ GPUs, pure DP/ZeRO-3 becomes communication-inefficient because:

1. **All-gather volume per layer per step scales as** $(1 - 1/\text{DP}) \times M_{\text{layer\_params}}$. With DP=512, each of the $2 \times N_{\text{layers}}$ all-gather operations (forward + backward) transfers nearly the full layer weight. For an 8192-hidden model with 80 layers, this is $\sim 2 \times 80 \times 2 \times 12 \times 8192^2 / (8 \times 10^9) \approx 25.8$ GB of data moved per step per GPU—far exceeding what inter-node InfiniBand can sustain without becoming the bottleneck.

2. **Latency of collectives grows logarithmically with world size** in ring or tree topologies. At 512 ranks, each all-gather has $O(\log 512) = 9$ steps of network latency overhead, compounding across hundreds of layers.

**Recommended approach at 512+ GPUs:** Combine TP (intra-node, degree 4 or 8) with PP (4–16 stages) and DP with ZeRO-1 or ZeRO-2 (not ZeRO-3, to avoid per-layer all-gathers). DP handles the remaining parallelism.

**Case D: 1024+ GPU Scale**

The reference configuration at this scale:

$$
\text{TP}=8, \quad \text{PP} \in [4, 16], \quad \text{DP} = \frac{N_{\text{GPUs}}}{\text{TP} \times \text{PP}}, \quad \text{ZeRO-2 on DP dimension}
$$

For 2048 H100 GPUs training a 405B model (Llama-3.1 class):

```
TP=8, PP=16, DP = 2048 / (8 × 16) = 16, ZeRO-1 on DP

Per-GPU parameters: 405B / (8 × 16) = 3.16B
model_bf16 = 6.33 GB
model_fp32 = 12.66 / 16 (ZeRO-1) = 0.79 GB
optimstates = 25.32 / 16 = 1.58 GB
grads_fp32 = 12.66 GB (ZeRO-0 on grads, or /16 with ZeRO-2 = 0.79 GB)

With ZeRO-2: Total model states ≈ 6.33 + 0.79 + 0.79 + 1.58 = 9.49 GB
Activations (selective recomp, 5 layers/stage): ~15–25 GB
Total: 25–35 GB → Ample headroom on 80 GB H100
```

**Case E: Special Architectures**

- **Long Context (≥ 32K tokens):** Activation memory scales linearly with sequence length (and quadratically in the attention score matrix without FlashAttention). **Context Parallelism (CP)** partitions the sequence dimension across GPUs, reducing per-GPU activation memory by a factor of CP. Preferred over increasing TP beyond 8 because CP communication (ring send/recv of KV blocks) overlaps with attention compute.

- **Mixture-of-Experts (MoE):** Expert parameters inflate the model size by the number of experts but each token activates only top-$k$ experts. **Expert Parallelism (EP)** distributes experts across GPUs, with all-to-all collectives routing tokens. EP is orthogonal to TP/PP/DP and typically occupies a dedicated process-group dimension.

#### 3.1.2 GPU-Poor Scenario (Insufficient GPUs Available)

When the GPU count is constrained and the model does not fit even with TP/PP across available devices, the engineer must reduce per-GPU memory demand:

1. **Full Activation Checkpointing (Recomputation):** Discard all intermediate activations during the forward pass; recompute them during the backward pass. Reduces activation memory to $O(N_{\text{layers}} \times s \times b \times h)$ at the cost of ~33% additional compute.

2. **Gradient Accumulation:** Reduce micro-batch size to 1 and accumulate gradients over multiple forward-backward passes. Reduces activation memory proportionally to micro-batch size.

3. **CPU/NVMe Offload (DeepSpeed ZeRO-Infinity):** Offload optimizer states and optionally parameters to host DRAM or NVMe. Trades PCIe bandwidth for HBM capacity. Only viable when compute density is low enough that PCIe transfers do not dominate step time.

4. **Mixed Precision with FP8 Weights:** On H100/B200/MI300X, storing weights in FP8 (E4M3 for forward, E5M2 for backward) reduces model_bf16 by 50%. Requires careful loss-scaling and master weights in BF16/FP32.

**Pseudocode: Memory Feasibility Check**

```
ALGORITHM: CheckMemoryFeasibility
INPUT: model_config, gpu_hbm_capacity, tp, pp, dp, cp, zero_stage,
       recompute_mode, mbs, seq_len
OUTPUT: fits (boolean), peak_memory_gb, headroom_gb

1. N_params ← ComputeParamCount(model_config)
2. layers_per_stage ← model_config.num_layers / pp

3. M_model_bf16 ← 2 × N_params / (tp × pp)
4. IF zero_stage ≥ 3:
       M_model_bf16 ← M_model_bf16 / dp

5. M_model_fp32 ← 2 × M_model_bf16       // FP32 master copy
6. IF zero_stage ≥ 1:
       M_model_fp32 ← M_model_fp32 / dp

7. M_grads ← 2 × M_model_bf16             // FP32 gradients
8. IF zero_stage ≥ 2:
       M_grads ← M_grads / dp

9. M_optim ← 4 × M_model_bf16             // Adam: 2 × FP32 states
10. IF zero_stage ≥ 1:
        M_optim ← M_optim / dp

11. M_act ← ComputeActivationMemory(model_config, seq_len / cp, mbs,
                                      tp, recompute_mode, layers_per_stage)

12. M_transient ← EstimateTransientBuffers(tp, pp, dp, seq_len, mbs)

13. peak_memory ← M_model_bf16 + M_model_fp32 + M_grads + M_optim + M_act + M_transient
14. headroom ← gpu_hbm_capacity - peak_memory
15. fits ← (headroom > 0)

16. RETURN fits, peak_memory, headroom
```

```
ALGORITHM: SelectMinimumParallelism
INPUT: model_config, num_gpus, gpu_hbm_capacity, target_gbs, seq_len
OUTPUT: parallelism_config {tp, pp, dp, cp, ep, zero_stage, mbs, gas, recompute}

1. FOR tp IN [1, 2, 4, 8]:
     FOR pp IN [1, 2, 4, 8, 16, 32]:
       FOR zero_stage IN [0, 1, 2, 3]:
         FOR recompute IN [none, selective, full]:
           FOR mbs IN [4, 2, 1]:
             dp ← num_gpus / (tp × pp)
             IF dp < 1: CONTINUE
             
             fits, peak_mem, headroom ← CheckMemoryFeasibility(
                 model_config, gpu_hbm_capacity, tp, pp, dp, 1, zero_stage,
                 recompute, mbs, seq_len)
             
             IF NOT fits: CONTINUE
             
             gas ← target_gbs / (mbs × dp)
             IF gas < 1: CONTINUE
             
             comm_cost ← EstimateCommunicationCost(tp, pp, dp, zero_stage, model_config)
             bubble_frac ← (pp - 1) / (gas + pp - 1)  // 1F1B schedule
             
             RECORD config {tp, pp, dp, zero_stage, recompute, mbs, gas,
                           peak_mem, headroom, comm_cost, bubble_frac}

2. SORT candidates BY (comm_cost + bubble_penalty) ASCENDING
3. RETURN candidates[0]
```

### 3.2 Topology-Aware Placement Rules

The selection of parallelism degrees is inseparable from the physical topology:

| Parallelism Dimension | Preferred Interconnect | Rationale |
|----------------------|----------------------|-----------|
| **TP** | NVLink / NVSwitch (intra-node) | 4 all-reduce (or all-gather + reduce-scatter) per layer; latency-sensitive; requires 450–900 GB/s |
| **PP** | InfiniBand / RoCE (inter-node) or NVLink | Point-to-point send/recv; modest bandwidth; tolerates 25–50 µs latency per microbatch |
| **DP / ZeRO** | InfiniBand / RoCE (inter-node) | Collective per gradient bucket; overlaps with backward compute; bandwidth-bound |
| **CP** | NVLink preferred; InfiniBand acceptable | Ring send/recv of KV blocks; overlaps with attention FLOPS |
| **EP** | InfiniBand / RoCE | All-to-all routing; bandwidth-bound; scale-sensitive |

> **Rule of Thumb (World-Size Factorization):** For $N$ GPUs, with $G$ GPUs per node:
> $$N = \text{TP} \times \text{PP} \times \text{DP} \times \text{CP} \times \text{EP}$$
> Constrain $\text{TP} \leq G$ (intra-node). Place TP on the innermost (fastest) communication domain. PP and DP span inter-node. CP can share the DP dimension or be factored separately. EP is typically orthogonal, mapped to a subset of DP ranks.

---

## 4. Step 2 — Satisfying the Target Global Batch Size

### 4.1 Why Global Batch Size Matters

Empirical scaling laws and training stability research (Kaplan et al., 2020; Hoffmann et al., 2022; McCandlish et al., 2018) demonstrate that there exists an optimal batch size for a given compute budget that minimizes the total number of training tokens to reach a target loss. For most large-scale LLM pretraining runs, this optimal range is:

$$
\text{GBS}_{\text{optimal}} \in [4\text{M}, 40\text{M}] \text{ tokens}
$$

The global batch size in tokens is:

$$
\text{GBS}_{\text{tokens}} = \text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}
$$

Or equivalently in sequences:

$$
\text{GBS}_{\text{sequences}} = \text{mbs} \times \text{dp} \times \text{gas}
$$

> **Critical Constraint:** GBS is a hyperparameter determined by convergence experiments. The parallelism configuration must achieve exactly this GBS—deviating changes the optimization trajectory and invalidates the learning rate schedule.

### 4.2 Adjusting GBS Upward

If Step 1 produced a configuration with $\text{mbs} \times \text{dp} \times \text{gas} < \text{GBS}_{\text{target}}$, increase GBS by:

1. **Increase gradient accumulation steps (gas).** Zero communication overhead; increases step time linearly. Useful when GPU count is fixed.

2. **Scale up DP.** Requires more GPUs. Each additional DP rank contributes $\text{mbs}$ sequences per step. Adds gradient synchronization cost but overlaps with backward pass.

3. **Scale up CP.** If context parallelism is in use, increasing CP allows the same number of tokens per step while splitting longer sequences. Each CP rank processes $\text{seq\_len} / \text{CP}$ tokens. If GBS is measured in tokens and seq_len is fixed, CP does not directly increase GBS—it is primarily a memory technique. However, if CP enables processing longer sequences, total tokens per step increase.

**Pseudocode: GBS Adjustment**

```
ALGORITHM: AdjustGlobalBatchSize
INPUT: target_gbs_tokens, current_config {mbs, dp, gas, seq_len, cp},
       max_gpus, gpu_hbm_capacity, model_config
OUTPUT: adjusted_config

1. current_gbs ← mbs × dp × gas × seq_len
2. IF current_gbs == target_gbs_tokens: RETURN current_config

3. IF current_gbs < target_gbs_tokens:
     // Strategy 1: Increase gradient accumulation (no extra GPUs)
     required_gas ← CEIL(target_gbs_tokens / (mbs × dp × seq_len))
     IF required_gas ≤ MAX_GAS_THRESHOLD:
         RETURN config WITH gas ← required_gas

     // Strategy 2: Scale DP (requires more GPUs)
     required_dp ← CEIL(target_gbs_tokens / (mbs × gas × seq_len))
     new_total_gpus ← required_dp × tp × pp
     IF new_total_gpus ≤ max_gpus:
         VALIDATE memory feasibility with new dp
         RETURN config WITH dp ← required_dp

     // Strategy 3: Increase mbs if memory permits
     required_mbs ← CEIL(target_gbs_tokens / (dp × gas × seq_len))
     IF CheckMemoryFeasibility(..., mbs=required_mbs, ...):
         RETURN config WITH mbs ← required_mbs

4. IF current_gbs > target_gbs_tokens:
     // Reduce gas first (cheapest adjustment)
     new_gas ← MAX(1, FLOOR(target_gbs_tokens / (mbs × dp × seq_len)))
     remaining ← target_gbs_tokens / (mbs × new_gas × seq_len)
     
     IF remaining < dp:
         // Reduce DP, reallocate GPUs to TP or PP
         new_dp ← FLOOR(remaining)
         freed_gpus ← (dp - new_dp) × tp × pp
         // Consider increasing PP to use freed GPUs for larger models
         RETURN config WITH dp ← new_dp, gas ← new_gas
     
     // Reduce mbs
     new_mbs ← MAX(1, FLOOR(target_gbs_tokens / (dp × new_gas × seq_len)))
     RETURN config WITH mbs ← new_mbs, gas ← new_gas
```

### 4.3 The Interplay Between GBS and Pipeline Parallelism

Pipeline parallelism introduces a **bubble fraction** that depends on the ratio of pipeline stages to gradient accumulation steps:

$$
\text{bubble\_fraction}_{\text{1F1B}} = \frac{\text{pp} - 1}{\text{gas} + \text{pp} - 1}
$$

To keep bubble overhead below 5%, the constraint is:

$$
\text{gas} \geq 19 \times (\text{pp} - 1) \quad \text{(for 5\% bubble)}
$$

This means that PP **prefers large gas**, which in turn prefers large GBS. If GBS is fixed and gas is too small for the chosen PP degree, either:

- Reduce PP (allocate GPUs differently).
- Use interleaved pipeline schedules (e.g., Megatron-LM's interleaved 1F1B with virtual stages), which reduce bubble to:

$$
\text{bubble\_fraction}_{\text{interleaved}} = \frac{\text{pp} - 1}{\text{gas} \times v + \text{pp} - 1}
$$

where $v$ is the number of virtual pipeline stages (model chunks) per physical stage.

- Use zero-bubble pipeline schedules (ZB-H1/ZB-H2) that interleave weight gradient computation into the bubble, achieving near-zero idle time.

---

## 5. Step 3 — Optimizing Training Throughput

With a memory-feasible configuration that achieves the target GBS, the engineer now optimizes for maximum **tokens per second per GPU** (equivalently, MFU) through controlled experimentation.

### 5.1 Experimental Optimization Strategy

There is **no closed-form solution** for the optimal configuration. The search space includes continuous and discrete variables whose interactions depend on hardware-specific constants (kernel efficiency, collective latency, memory allocator behavior). The systematic approach is:

**Phase 1: Establish Baseline**
- Run the Step 1/2 configuration for 50–100 steps.
- Record: step time, MFU, memory utilization, communication time, bubble fraction.
- Profile with Nsight Systems / PyTorch Profiler to decompose step time.

**Phase 2: Explore TP Scaling**
- Increase TP from current value toward node size (e.g., TP=2 → TP=4 → TP=8).
- Each TP increase reduces per-GPU parameter count, gradient size, and activation memory.
- Each TP increase adds 4 × num_layers collectives per step (all-gather/reduce-scatter in attention and MLP).
- **Measure:** Does the memory savings (enabling larger mbs or less recomputation) offset the communication cost?

**Phase 3: Explore DP with ZeRO Stages**
- If TP=8 leaves residual memory, try DP with ZeRO-1 (shard optimizer only) instead of ZeRO-3 (shard everything).
- ZeRO-1 has lower communication volume than ZeRO-3 (one all-gather at step end vs. all-gather per layer per forward+backward).
- **Trade-off:** ZeRO-3 frees more memory but at higher communication cost.

**Phase 4: Introduce or Adjust PP**
- PP is beneficial when DP communication (gradient all-reduce or reduce-scatter) saturates the inter-node bandwidth.
- PP replaces gradient collectives with point-to-point send/recv of activations, which use a fraction of the bandwidth.
- **Cost:** Pipeline bubble. Use interleaved schedule to mitigate.

**Phase 5: Tune Micro-Batch Size**
- Larger mbs → higher arithmetic intensity → better GPU utilization.
- Larger mbs → larger activation memory → may require more recomputation.
- Larger mbs with fixed GBS → fewer gas → larger pipeline bubble.
- **Optimal mbs** is typically the largest value that fits in memory with acceptable recomputation overhead.

**Phase 6: Enable Compute-Communication Overlap**
- DP gradient synchronization overlapped with backward pass of subsequent microbatches.
- TP collectives overlapped with subsequent TP regions (attention → MLP transition).
- PP send/recv overlapped with compute of non-dependent microbatches.
- ZeRO-3 all-gather for next layer overlapped with current layer's compute.

**Pseudocode: Throughput Optimization Search**

```
ALGORITHM: OptimizeThroughput
INPUT: base_config, model_config, cluster_topology, target_gbs
OUTPUT: optimized_config, performance_report

1. baseline ← ProfileTraining(base_config, num_steps=100)
2. candidates ← [base_config]

3. // Phase 1: TP scaling
   FOR tp IN [2, 4, 8] WHERE tp ≤ cluster_topology.gpus_per_node:
     config ← DeriveConfig(model_config, tp, target_gbs, cluster_topology)
     IF CheckMemoryFeasibility(config):
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

4. // Phase 2: ZeRO stage exploration
   FOR zero_stage IN [0, 1, 2, 3]:
     config ← best_tp_config WITH zero_stage
     IF CheckMemoryFeasibility(config):
       // Exploit freed memory to increase mbs
       max_mbs ← BinarySearchMaxMBS(config)
       config.mbs ← max_mbs
       config.gas ← target_gbs / (config.mbs × config.dp × config.seq_len)
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

5. // Phase 3: PP introduction
   FOR pp IN [2, 4, 8, 16]:
     config ← DeriveConfigWithPP(model_config, best_tp, pp, target_gbs)
     IF CheckMemoryFeasibility(config):
       // Ensure bubble fraction < 10%
       IF BubbleFraction(config) > 0.10: TRY interleaved schedule
       result ← ProfileTraining(config, num_steps=100)
       candidates.APPEND((config, result))

6. // Phase 4: mbs tuning for top-3 candidates
   FOR config IN TopK(candidates, k=3, sort_by=tokens_per_sec):
     FOR mbs IN [1, 2, 4, 8, 16]:
       adjusted ← config WITH mbs, gas adjusted for target_gbs
       IF CheckMemoryFeasibility(adjusted):
         result ← ProfileTraining(adjusted, num_steps=50)
         candidates.APPEND((adjusted, result))

7. // Phase 5: Overlap tuning
   best_config ← TopK(candidates, k=1)
   FOR overlap_mode IN [none, dp_overlap, tp_overlap, full_overlap]:
     result ← ProfileTraining(best_config WITH overlap_mode, num_steps=50)
     IF result.mfu > best_result.mfu:
       best_config ← best_config WITH overlap_mode

8. RETURN best_config, GenerateReport(candidates)
```

### 5.2 Micro-Batch Size and Arithmetic Intensity

The micro-batch size directly controls the **arithmetic intensity** of GEMM operations:

$$
\text{Arithmetic Intensity} = \frac{2 \times M \times N \times K}{2 \times (M \times K + K \times N + M \times N)} \approx \frac{M \times N \times K}{M \times K + K \times N + M \times N}
$$

For a linear layer with input $(b \times s, h)$ and weight $(h, 4h)$:

- $M = b \times s / \text{tp}$, $N = 4h / \text{tp}$, $K = h$
- Increasing $b$ (mbs) increases $M$, pushing the operation deeper into the compute-bound regime.

On H100 with 3.35 TB/s HBM bandwidth and 989 TFLOPS BF16 peak, the compute-memory crossover occurs at arithmetic intensity $\approx 295$ ops/byte. For typical hidden sizes (≥4096), this is achieved at $b \times s \geq 2048$, which is almost always satisfied for seq_len ≥ 2048 even at mbs=1.

However, **small mbs still hurts** due to:
- Kernel launch overhead amortization
- Reduced opportunities for CUDA graph capture (graph capture requires fixed shapes)
- Lower occupancy in attention kernels (FlashAttention efficiency degrades below certain sequence/batch thresholds)

---

## 6. Parallelization Strategy Reference Matrix

The following table provides a comprehensive, production-grade reference for each parallelism strategy. Each row details the impact on batch size, memory, compute, communication pattern, and overlap characteristics.

### 6.1 Complete Strategy Comparison

| Strategy | Batch Size Effect | Memory Reduction | Compute Reduction | Communication Pattern | Compute/Communication Overlap Condition |
|----------|------------------|-------------------|--------------------|-----------------------|----------------------------------------|
| **Data Parallelism (DP)** | GBS scales linearly with DP | Can reduce mbs by increasing DP → reduces activation memory | Can reduce mbs by increasing DP → reduces per-GPU compute | **Backward:** all-reduce of gradients in BF16 | Overlapped with microbatch backward pass. Overlap ratio: $(DP-1) \times N_{\text{params}} \times \text{peak\_flops} / (2 \times \text{peak\_bw} \times N_{\text{tokens}} \times DP)$ |
| **DP + ZeRO-1** | GBS scales with DP | model\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Backward:** all-reduce of gradients in BF16. **Step end:** all-gather of model\_fp32 | Same as DP; additional all-gather at step boundary (non-overlapped but amortized) |
| **DP + ZeRO-2** | GBS scales with DP | model\_fp32 / DP, grads\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Backward:** reduce-scatter of gradients in BF16. **Step end:** all-gather of model\_fp32 | Overlapped with backward: $(DP-1) \times N_{\text{params}} \times \text{peak\_flops} / (4 \times \text{peak\_bw} \times N_{\text{tokens}} \times DP)$. Improvement over DP because reduce-scatter has half the volume of all-reduce. |
| **DP + ZeRO-3 (FSDP)** | GBS scales with DP | model\_bf16 / DP, model\_fp32 / DP, grads\_fp32 / DP, optimstates / DP | Can reduce mbs by increasing DP | **Per layer (×num\_layers):** Forward: all-gather model\_fp32. Backward: all-gather model\_fp32 + reduce-scatter grads\_fp32 | Overlapped with next layer's forward/backward: $(DP-1) \times \text{peak\_flops} / (2 \times s \times \text{mbs} \times \text{peak\_bw})$ |
| **Tensor Parallelism (TP)** | No effect on GBS | model\_bf16 / TP, model\_fp32 / TP, grads\_fp32 / TP, optimstates / TP, activations / TP | Compute per GPU reduces by 1/TP (weight FLOPS) | **Per TP region (×4 × num\_layers):** Forward: all-reduce or all-gather of activations. Backward: all-reduce or reduce-scatter of gradients. | Overlapped with next TP region (attention↔MLP↔LayerNorm): $(TP-1) \times \text{peak\_flops} / (24 \times h \times \text{peak\_bw})$ |
| **Pipeline Parallelism (1F1B)** | Prefers large gas to minimize bubble | model\_bf16 / PP, model\_fp32 / PP, grads\_fp32 / PP, optimstates / PP | Compute per GPU reduces by 1/PP (fewer layers) | **Per microbatch (×gas):** Forward: recv + send activation tensors. Backward: recv + send gradient tensors. | Overlapped with next microbatch's compute: $PP \times \text{peak\_flops} / (32 \times h \times N_{\text{layers}} \times \text{peak\_bw})$ |
| **Context Parallelism (CP)** | Prefers large seq\_len for efficient overlap | activations / CP (sequence dimension split) | Attention FLOPS per GPU reduce (shorter local sequence) | **Per layer (×(CP-1) × num\_layers):** Forward: ring send/recv of KV blocks. Backward: ring send/recv of gradient blocks. | Overlapped with attention computation (Ring Attention): communication volume $= (CP-1) \times h \times (L/CP) \times H_{kv} \times (n_k + n_v) \times 2$ bytes per direction |
| **Expert Parallelism (EP)** | Batch size scales with EP (each expert sees fewer tokens) | expert\_params / EP | expert\_compute / EP | **Per MoE layer (×num\_moe\_layers):** Forward: all-to-all dispatch of tokens. Backward: all-to-all of gradients. | Overlapped with MoE block computation: $(EP-1) \times \text{peak\_flops} / (12 \times N_{\text{experts}} \times h \times \text{peak\_bw})$ |

### 6.2 Communication Volume Formulae

For rigorous capacity planning, the per-step communication volume per GPU for each strategy:

| Strategy | Per-Step Volume Per GPU | Direction |
|----------|------------------------|-----------|
| **DP (all-reduce)** | $2 \times (DP-1)/DP \times 2 \times N_{\text{params}}$ bytes | Bidirectional (ring) |
| **ZeRO-2 (reduce-scatter + all-gather)** | $(DP-1)/DP \times 2 \times N_{\text{params}} + (DP-1)/DP \times 4 \times N_{\text{params}}$ | Bidirectional |
| **ZeRO-3 (per-layer all-gather ×2 + reduce-scatter)** | $3 \times N_{\text{layers}} \times (DP-1)/DP \times 4 \times N_{\text{params\_per\_layer}}$ | Bidirectional |
| **TP (per-layer all-reduce ×4)** | $4 \times N_{\text{layers}} \times 2 \times (TP-1)/TP \times 2 \times s \times b \times h / TP$ | Intra-node |
| **PP (per-microbatch send/recv ×2)** | $2 \times \text{gas} \times 2 \times s \times b \times h$ bytes | Point-to-point |
| **CP (ring KV exchange)** | $(CP-1) \times N_{\text{layers}} \times 2 \times s/CP \times b \times 2 \times h_{kv} \times 2$ bytes | Ring |
| **EP (all-to-all)** | $N_{\text{moe\_layers}} \times 2 \times \text{tokens\_dispatched} \times h \times 2$ bytes | All-to-all |

---

## 7. Framework-Specific Implementation Guidance

### 7.1 Megatron-LM / Megatron-Core

Megatron-Core is the reference implementation for combined TP+PP+DP+CP+EP training. It provides:

- **Column- and row-parallel linear layers** for TP, with fused kernels for GEMMs.
- **Interleaved 1F1B pipeline schedule** with configurable virtual pipeline stages.
- **Distributed optimizer** (equivalent to ZeRO-1) that shards optimizer states across DP ranks.
- **Context parallelism** via Ring Attention or Ulysses (sequence-parallel attention).
- **Expert parallelism** with configurable expert routing, token dropping, and load balancing.
- **Sequence parallelism (SP)**: partitions LayerNorm and Dropout activations along the sequence dimension across TP ranks, reducing activation memory by 1/TP for these operations.

**Launch Configuration (torchrun):**

```
ALGORITHM: LaunchMegatronTraining
INPUT: model_config, parallelism_config, cluster_config, checkpoint_path
OUTPUT: launched training process group

1. // Compute world size
   world_size ← tp × pp × dp × cp × ep_if_orthogonal
   num_nodes ← world_size / cluster_config.gpus_per_node
   
2. // Generate torchrun command
   launch_cmd ← "torchrun"
     + " --nproc_per_node=" + cluster_config.gpus_per_node
     + " --nnodes=" + num_nodes
     + " --node_rank=$NODE_RANK"
     + " --master_addr=$MASTER_ADDR"
     + " --master_port=$MASTER_PORT"
     + " --rdzv_backend=c10d"
     + " --rdzv_endpoint=$MASTER_ADDR:$MASTER_PORT"

3. // Megatron arguments
   megatron_args ← {
     "--tensor-model-parallel-size": tp,
     "--pipeline-model-parallel-size": pp,
     "--context-parallel-size": cp,
     "--num-layers-per-virtual-pipeline-stage": virtual_chunks,
     "--sequence-parallel": True,     // SP coupled with TP
     "--use-distributed-optimizer": True,  // ZeRO-1 on DP dim
     "--overlap-grad-reduce": True,
     "--overlap-param-gather": True,
     "--use-flash-attn": True,
     "--recompute-granularity": "selective",  // or "full"
     "--recompute-method": "uniform",
     "--bf16": True,
     "--micro-batch-size": mbs,
     "--global-batch-size": target_gbs,
     "--accumulate-allreduce-grads-in-fp32": True,
     "--load": checkpoint_path,
     "--save": checkpoint_save_path,
     "--save-interval": save_interval,
   }
   
4. // Expert parallelism (MoE models)
   IF model_config.is_moe:
     megatron_args += {
       "--num-experts": model_config.num_experts,
       "--expert-model-parallel-size": ep,
       "--moe-router-topk": model_config.top_k,
       "--moe-token-dispatcher-type": "alltoall",
       "--moe-aux-loss-coeff": 0.01,
     }

5. // Process group initialization
   // Megatron-Core creates nested process groups:
   //   - TP group: ranks sharing TP dimension (size=tp)
   //   - PP group: ranks sharing PP dimension (size=pp)
   //   - DP group: ranks sharing DP dimension (size=dp)
   //   - CP group: ranks sharing CP dimension (size=cp)
   //   - EP group: subset of DP ranks for expert sharding (size=ep)
   // Placement: TP on innermost (consecutive GPU IDs within node)
   //            PP across nodes, DP across remaining

6. EXECUTE launch_cmd WITH megatron_args
```

**Checkpoint Interoperability:**

Megatron-Core checkpoints store sharded state dictionaries indexed by TP rank, PP stage, and DP rank (for distributed optimizer). Converting between parallelism configurations requires:

```
ALGORITHM: ReshardCheckpoint
INPUT: source_ckpt_path, source_config {tp_s, pp_s, dp_s},
       target_config {tp_t, pp_t, dp_t}
OUTPUT: resharded checkpoint at target_ckpt_path

1. // Load all shards into a single consolidated state
   full_state ← {}
   FOR tp_rank IN [0, tp_s):
     FOR pp_rank IN [0, pp_s):
       shard ← LoadShard(source_ckpt_path, tp_rank, pp_rank)
       // Concatenate TP-sharded weight matrices along partition dim
       // Assign layers to PP stages based on pp_rank
       MergeInto(full_state, shard, tp_rank, pp_rank, tp_s, pp_s)

2. // Re-shard to target configuration
   FOR tp_rank IN [0, tp_t):
     FOR pp_rank IN [0, pp_t):
       target_shard ← ExtractShard(full_state, tp_rank, pp_rank, tp_t, pp_t)
       SaveShard(target_ckpt_path, target_shard, tp_rank, pp_rank)

3. // Validate
   FOR each parameter IN full_state:
     reconstructed ← ReconstructFromShards(target_ckpt_path, tp_t, pp_t)
     ASSERT AllClose(full_state[parameter], reconstructed, atol=0)
```

### 7.2 DeepSpeed

DeepSpeed provides ZeRO-1/2/3, pipeline parallelism (via `PipelineEngine`), and integration with Megatron-LM. Key configuration elements:

**ZeRO Stage Selection Criteria:**

| ZeRO Stage | What is Sharded | Communication Pattern | When to Use |
|------------|----------------|----------------------|-------------|
| **Stage 1** | Optimizer states | All-reduce gradients + all-gather FP32 params at step end | Models that fit in memory with TP/PP; want to reduce optimizer memory across DP |
| **Stage 2** | Optimizer states + gradients | Reduce-scatter gradients + all-gather FP32 params at step end | Need more memory than Stage 1; communication is only marginally higher |
| **Stage 3** | Optimizer states + gradients + parameters | All-gather params per layer (fwd+bwd) + reduce-scatter gradients | Model does not fit even with Stage 2; accept higher communication cost |

**DeepSpeed Configuration Template (Pseudocode):**

```
ALGORITHM: GenerateDeepSpeedConfig
INPUT: model_size, gpu_memory, num_gpus, target_gbs, seq_len
OUTPUT: ds_config (JSON-compatible dictionary)

1. // Determine ZeRO stage
   mem_needed_z0 ← EstimateMemory(model_size, zero_stage=0, dp=num_gpus)
   IF mem_needed_z0 ≤ gpu_memory × 0.85:
     zero_stage ← 0
   ELSE:
     mem_needed_z1 ← EstimateMemory(model_size, zero_stage=1, dp=num_gpus)
     IF mem_needed_z1 ≤ gpu_memory × 0.85:
       zero_stage ← 1
     ELSE:
       mem_needed_z2 ← EstimateMemory(model_size, zero_stage=2, dp=num_gpus)
       IF mem_needed_z2 ≤ gpu_memory × 0.85:
         zero_stage ← 2
       ELSE:
         zero_stage ← 3

2. ds_config ← {
     "train_micro_batch_size_per_gpu": mbs,
     "gradient_accumulation_steps": gas,
     "steps_per_print": 10,
     "gradient_clipping": 1.0,
     "bf16": {"enabled": True},
     "zero_optimization": {
       "stage": zero_stage,
       "overlap_comm": True,
       "contiguous_gradients": True,
       "reduce_bucket_size": 5e8,    // 500M elements ≈ 1 GB BF16
       "allgather_bucket_size": 5e8,
       "reduce_scatter": True,       // Stage 2+
       "allgather_partitions": True, // Stage 3
       "stage3_prefetch_bucket_size": 5e7,
       "stage3_param_persistence_threshold": 1e6,
       "stage3_max_live_parameters": 1e9,
       "stage3_max_reuse_distance": 1e9,
     },
     "activation_checkpointing": {
       "partition_activations": True,   // Partition across TP ranks
       "contiguous_memory_optimization": True,
       "cpu_checkpointing": False,
     },
     "zero_allow_untested_optimizer": True,
     "wall_clock_breakdown": True,
   }

3. // Offload (GPU-poor scenario)
   IF zero_stage == 3 AND model_size > EstimateMaxModelForGPUMemory():
     ds_config["zero_optimization"]["offload_optimizer"] ← {
       "device": "cpu",
       "pin_memory": True,
       "fast_init": True,
     }
     ds_config["zero_optimization"]["offload_param"] ← {
       "device": "cpu",
       "pin_memory": True,
     }

4. RETURN ds_config
```

### 7.3 PyTorch FSDP / DTensor

PyTorch's Fully Sharded Data Parallel (FSDP2, based on DTensor) provides ZeRO-3 semantics with native PyTorch integration:

- **DTensor** provides a device-mesh abstraction that maps logical parallelism dimensions (DP, TP) to physical GPU topologies.
- **FSDP2** uses DTensor to shard parameters along the DP dimension with `Shard(0)` placement.
- **TP** is achieved via DTensor's `Replicate` and `Shard` placements on different mesh dimensions.

**Advantages:** Native PyTorch, composable with `torch.compile`, CUDA Graphs, and custom kernels without framework lock-in.

**Limitations:** Less mature pipeline parallelism support compared to Megatron-Core. No built-in interleaved pipeline schedules. Expert parallelism requires manual implementation.

**When to choose FSDP/DTensor over Megatron-Core or DeepSpeed:**
- When portability and framework simplicity are priorities.
- For fine-tuning (SFT, DPO, PPO) where PP is not needed and DP+TP suffice.
- When integrating with custom training loops (RLHF, GRPO) that do not fit Megatron's training harness.
- When `torch.compile` is required for kernel fusion (Megatron-Core kernels are hand-fused and bypass compile).

### 7.4 torchrun Orchestration

`torchrun` (replacement for `torch.distributed.launch`) handles elastic and fault-tolerant distributed process management:

```
ALGORITHM: OrchestrateTorchrun
INPUT: script_path, num_nodes, gpus_per_node, rdzv_backend,
       rdzv_endpoint, max_restarts, env_vars
OUTPUT: running distributed training

1. // Preflight checks (run on each node before launch)
   FOREACH node IN cluster:
     VerifyGPUHealth(node)          // nvidia-smi / rocm-smi checks
     VerifyNICBandwidth(node)       // ib_write_bw / perftest
     VerifyNVLinkTopology(node)     // nvidia-smi topo -m
     VerifyNCCLVersion(node)        // must match across all nodes
     VerifyContainerConsistency(node) // driver, CUDA/ROCm, PyTorch versions
     RunNCCLBandwidthTest(node)     // all_reduce_perf / all_gather_perf

2. // Environment setup
   env ← {
     "NCCL_IB_DISABLE": "0",
     "NCCL_NET_GDR_LEVEL": "5",      // GPUDirect RDMA
     "NCCL_IB_HCA": auto_detect(),    // mlx5_0:1,mlx5_1:1,...
     "NCCL_SOCKET_IFNAME": "eth0",
     "NCCL_DEBUG": "WARN",            // "INFO" for debugging
     "NCCL_ALGO": "Ring,Tree",
     "NCCL_PROTO": "Simple,LL128",
     "CUDA_DEVICE_MAX_CONNECTIONS": "1",  // For Megatron overlap
     "OMP_NUM_THREADS": "4",
     "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
     "NCCL_CROSS_NIC": "1",
   }
   env.UPDATE(env_vars)

3. // Launch command
   cmd ← "torchrun"
     + " --nnodes=" + num_nodes
     + " --nproc_per_node=" + gpus_per_node
     + " --rdzv_backend=" + rdzv_backend  // "c10d" or "etcd"
     + " --rdzv_endpoint=" + rdzv_endpoint
     + " --max_restarts=" + max_restarts
     + " --monitor_interval=5"
     + " " + script_path
     + " " + training_args

4. // Launch with health monitoring
   FOREACH node IN cluster:
     SSH node: "export ENV; exec cmd" &
   
   MonitorLoop:
     WHILE training_active:
       CheckHeartbeats(all_nodes)
       CheckGPUUtilization(all_nodes)   // Alert if < 80%
       CheckMemoryUsage(all_nodes)       // Alert if > 95% HBM
       CheckNetworkCounters(all_nodes)   // Alert on error counters
       IF node_failure_detected:
         TriggerCheckpointSave()
         WaitForElasticRestart(max_restarts)
       SLEEP(monitor_interval)
```

---

## 8. Kernel-Level Optimization and Numerical Stability

### 8.1 FlashAttention and Fused Kernels

**FlashAttention** (Dao, 2022; Dao, 2023) eliminates the $O(s^2)$ memory requirement of standard attention by computing attention in tiles, never materializing the full $s \times s$ attention matrix in HBM:

- **Memory:** $O(s)$ instead of $O(s^2)$
- **IO complexity:** $O(s^2 \cdot d / M)$ where $M$ is SRAM size, versus $O(s^2 \cdot d + s^2)$ for standard attention
- **Backward pass:** Recomputes attention weights from Q, K, V stored in HBM using the online softmax statistics (row-max and row-sum-exp saved during forward)

**When FlashAttention provides the largest benefit:**
- Sequence lengths ≥ 2048
- The memory savings enable larger micro-batch sizes, which improves overall throughput
- With context parallelism, each CP rank runs FlashAttention on sequence length $s/\text{CP}$

**Fused Kernels Critical for Performance:**

| Kernel | What It Fuses | Impact |
|--------|--------------|--------|
| **Fused RMSNorm** | Normalization + residual add | Eliminates extra HBM read/write of residual |
| **Fused Softmax** | Temperature scaling + masking + softmax | Saves one $O(s^2)$ HBM pass |
| **Fused RoPE** | Rotary position encoding into Q/K projection | Avoids separate RoPE kernel launch |
| **Fused SwiGLU MLP** | Gate projection + activation + up projection | Reduces HBM traffic by 33% |
| **Fused Cross-Entropy** | Log-softmax + NLL loss | Critical for large vocabularies; avoids materializing logits |

### 8.2 Triton Kernels

Triton enables writing custom GPU kernels in Python with auto-tuning, portable across NVIDIA (CUDA) and AMD (ROCm/HIP):

```
ALGORITHM: TritonFusedRMSNorm
INPUT: x[M, N] (activation tensor), w[N] (weight), epsilon
OUTPUT: y[M, N] (normalized tensor)

// Each Triton program instance handles one row
1. row_idx ← program_id(axis=0)
2. col_offsets ← arange(0, BLOCK_SIZE)
3. mask ← col_offsets < N

4. // Load row from HBM (single pass)
   x_row ← load(x[row_idx, col_offsets], mask=mask, other=0.0)

5. // Compute variance in SRAM
   variance ← sum(x_row * x_row) / N
   inv_rms ← rsqrt(variance + epsilon)

6. // Normalize and scale
   w_vals ← load(w[col_offsets], mask=mask)
   y_row ← x_row * inv_rms * w_vals

7. // Store result (single pass)
   store(y[row_idx, col_offsets], y_row, mask=mask)

// Auto-tune BLOCK_SIZE over [256, 512, 1024, 2048, 4096]
// Measure: time, register usage, occupancy
// Select BLOCK_SIZE that maximizes throughput for given N
```

**When to use Triton vs. CUDA/HIP:**
- Triton for **element-wise fusions, reductions, normalization** kernels where the programming model maps cleanly.
- CUDA/HIP for **GEMM wrappers, collective-aware kernels, persistent kernels** that require fine-grained thread-block scheduling or shared memory choreography beyond Triton's abstraction.

### 8.3 FP8 and Mixed-Precision Training

H100, B200, and MI300X support FP8 (E4M3 for forward, E5M2 for backward):

- **FP8 GEMMs:** 2× throughput vs. BF16 on H100 Tensor Cores (1978 vs. 989 TFLOPS)
- **Requires per-tensor or per-block scaling** to prevent overflow/underflow
- **MXFP8 (Microscaling FP8):** Block-level scaling with shared exponents (block size 32), providing finer granularity than per-tensor scaling
- **MXFP6/MXFP4:** Available on B200; further reduces memory/bandwidth but requires careful validation

**Numerical Stability Checklist for FP8:**

```
ALGORITHM: ValidateFP8Parity
INPUT: model, dataset, bf16_baseline_loss_curve, fp8_config
OUTPUT: parity_report

1. // Run BF16 baseline for N steps
   bf16_losses ← Train(model, dataset, precision="bf16", steps=N)

2. // Run FP8 with identical seed, data order, hyperparameters
   fp8_losses ← Train(model, dataset, precision="fp8",
                       fp8_config=fp8_config, steps=N)

3. // Validate parity
   FOR step IN [0, N):
     relative_diff ← |bf16_losses[step] - fp8_losses[step]| / bf16_losses[step]
     ASSERT relative_diff < 0.02  // 2% relative tolerance early in training
   
   // Final loss parity (most important)
   final_diff ← |bf16_losses[N-1] - fp8_losses[N-1]| / bf16_losses[N-1]
   ASSERT final_diff < 0.005  // 0.5% at convergence

4. // Monitor overflow/underflow statistics
   ASSERT fp8_overflow_rate < 0.001  // < 0.1% of tensors
   ASSERT fp8_underflow_rate < 0.01  // < 1% of tensors

5. // Gradient norm tracking
   FOR step IN [0, N):
     bf16_gnorm ← GradientNorm(bf16_run, step)
     fp8_gnorm ← GradientNorm(fp8_run, step)
     ASSERT |bf16_gnorm - fp8_gnorm| / bf16_gnorm < 0.05

6. RETURN ParityReport(bf16_losses, fp8_losses, overflow_stats)
```

**Loss Scaling for FP16 Training (Still Relevant for AMD MI300X without native BF16 efficiency):**

- Dynamic loss scaling: start at scale $2^{16}$, double every 2000 steps if no overflow, halve on overflow.
- Gradient clipping applied **after unscaling** (divide by loss scale), **before** optimizer step.
- BF16 does not require loss scaling (exponent range matches FP32), which is why BF16 is universally preferred when hardware supports it at equivalent throughput.

### 8.4 Compute-Communication Overlap

The single most impactful throughput optimization after kernel selection is **overlapping communication with computation**:

**DP Gradient Overlap:**

```
ALGORITHM: OverlapDPGradientSync
INPUT: model_layers, dp_process_group, bucket_size_mb

1. // During backward pass:
   gradient_buckets ← CreateBuckets(model.parameters(), bucket_size_mb)
   
   FOR layer IN REVERSE(model_layers):
     // Compute gradients for this layer
     ComputeBackward(layer)
     
     // Check if any bucket is full
     FOR bucket IN gradient_buckets:
       IF bucket.is_ready():
         // Launch async all-reduce (DP) or reduce-scatter (ZeRO-2)
         bucket.async_comm_handle ← AllReduceAsync(
           bucket.gradient_tensor,
           group=dp_process_group,
           op=AVG
         )

2. // Wait for all communication to complete
   FOR bucket IN gradient_buckets:
     bucket.async_comm_handle.wait()
```

**TP-Comm Overlap (Megatron-Core approach):**

In Megatron-Core, each Transformer layer has 4 TP communication points:
1. Forward attention: all-reduce (or reduce-scatter with SP)
2. Forward MLP: all-reduce (or reduce-scatter with SP)
3. Backward MLP: all-reduce (or all-gather with SP)
4. Backward attention: all-reduce (or all-gather with SP)

With `CUDA_DEVICE_MAX_CONNECTIONS=1`, Megatron-Core ensures that the GEMM kernel and the NCCL kernel execute on different CUDA streams that can overlap on the GPU's compute and copy engines.

**ZeRO-3 / FSDP Prefetch Overlap:**

```
ALGORITHM: FSDP_PrefetchOverlap
INPUT: model_layers, dp_process_group

1. FOR i IN range(len(model_layers)):
     // Prefetch next layer's parameters while computing current layer
     IF i + 1 < len(model_layers):
       PrefetchAllGather(model_layers[i+1].parameters(), group=dp_process_group)
     
     // Compute forward for current layer (parameters already gathered)
     output ← model_layers[i].forward(input)
     
     // Free current layer's full parameters (keep only shard)
     FreeFull Parameters(model_layers[i])
     
     input ← output
```

---

## 9. Data Pipeline Engineering for Distributed Training

### 9.1 End-to-End Pipeline Architecture

The data pipeline is a **distributed system** that must deliver tokenized, packed, shuffled batches to every GPU without becoming a throughput bottleneck or introducing statistical bias.

```
ALGORITHM: DistributedDataPipeline
INPUT: raw_corpus, tokenizer_config, training_config
OUTPUT: streaming dataloader yielding (input_ids, labels, attention_mask) per GPU

PHASE 1: Offline Preprocessing
  1.1. Ingest raw documents from heterogeneous sources (CommonCrawl, books, code, etc.)
  1.2. Filter: language ID, perplexity scoring, deduplication (MinHash → LSH),
       safety filtering, quality heuristics
  1.3. Deduplicate:
       - Document-level: MinHash with 128 permutations, Jaccard threshold 0.8
       - Substring-level: suffix array for exact n-gram dedup (n ≥ 50 tokens)
  1.4. Tokenize with trained tokenizer (BPE/SentencePiece/Unigram)
       - Tokenizer trained on representative sample of final corpus
       - Vocabulary size: 32K–128K (trade-off: larger vocab → shorter sequences
         → less compute; but embedding table grows)
  1.5. Pack sequences:
       - Concatenate documents with <EOS> separator
       - Pack into fixed-length sequences (seq_len) to maximize GPU utilization
       - Track document boundaries for attention masking (no cross-document attention)
  1.6. Shard into N binary files (e.g., .bin + .idx for Megatron format,
       or Parquet/WebDataset/MosaicML Streaming format)
       - Each shard: ~1 GB, containing ~500K sequences of length 4096
       - Write deterministic shard assignment based on document hash

PHASE 2: Online Loading (during training)
  2.1. Each DP rank opens its assigned shards (disjoint across DP ranks)
  2.2. Memory-map (.mmap) shard files for zero-copy access
  2.3. Shuffle:
       - Epoch-level: shuffle shard order with seed = base_seed + epoch
       - Intra-shard: shuffle sequence indices within each shard
       - This provides pseudo-random global order without full-corpus shuffle
  2.4. Prefetch: async worker threads load next batch while GPU processes current
       - num_workers: 4–8 per GPU (CPU-bound tokenization/packing already done offline)
       - prefetch_factor: 2–4 batches ahead
  2.5. Curriculum mixing:
       - Domain weights (e.g., 50% web, 20% code, 15% books, 10% math, 5% multilingual)
       - Weights may change during training (curriculum schedule)
       - Implemented as weighted sampling across domain-specific shard pools
  2.6. Yield batches of shape (mbs, seq_len) to each GPU

PHASE 3: Deterministic Resume
  3.1. Save dataloader state at each checkpoint:
       - Current shard index, position within shard, epoch counter, RNG state
       - Per-domain sample counts (for curriculum tracking)
  3.2. On resume: restore exact dataloader state → bitwise-identical training
```

### 9.2 Sequence Packing

Naive padding wastes significant compute. Sequence packing concatenates variable-length documents into fixed-length training sequences:

**Without packing (padding to max length):**
- Average document length: 512 tokens, training seq_len: 4096
- Waste: $(4096 - 512) / 4096 = 87.5\%$ of tokens are padding → 87.5% wasted FLOPS

**With packing:**
- Pack ~8 documents per sequence on average
- No wasted tokens (except the last partial sequence per shard)
- Must use attention masking to prevent cross-document attention leakage

### 9.3 Data Lineage and Reproducibility

For regulatory compliance, scientific reproducibility, and debugging:

- Every training run records: exact shard list, shuffle seed, domain weights, tokenizer version, preprocessing pipeline version hash.
- Data version control: content-addressable storage for shards (hash-based naming).
- Reprocessing any shard from raw data must produce identical output (deterministic preprocessing).

---

## 10. Fault Tolerance, Automation, and Production Resilience

### 10.1 Failure Modes in Large-Scale Training

| Failure Mode | Frequency (per 1000 GPU-hours) | Impact | Mitigation |
|-------------|-------------------------------|--------|------------|
| GPU hardware fault (ECC error, XID error) | 0.1–0.5 | Single GPU, kills rank | Elastic restart, spare node pool |
| InfiniBand link flap | 0.05–0.2 | Straggler or timeout | NCCL timeout detection, link health monitoring |
| NVLink error | 0.01–0.05 | Node-level failure | Node replacement, TP group reformation |
| Storage I/O stall | 0.1–1.0 | Dataloader starvation | Local SSD caching, async prefetch, retry logic |
| NCCL timeout/hang | 0.05–0.3 | Full training hang | Watchdog timer, TORCH_NCCL_ASYNC_ERROR_HANDLING |
| OOM (unexpected memory spike) | 0.01–0.1 | Rank crash | Memory monitoring, conservative headroom |
| Software bug (NaN loss, gradient explosion) | Varies | Wasted compute | Loss spike detection, auto-rollback to last good checkpoint |

### 10.2 Resilient Training Automation

```
ALGORITHM: ResilientTrainingLoop
INPUT: training_config, cluster_config, max_failures
OUTPUT: trained model checkpoint

1. failure_count ← 0
2. last_good_checkpoint ← FindLatestValidCheckpoint(training_config.save_path)

3. WHILE NOT training_complete AND failure_count < max_failures:
   TRY:
     // Preflight
     healthy_nodes ← RunPreflightChecks(cluster_config)
     IF |healthy_nodes| < RequiredNodes(training_config):
       WaitForNodeRepair(timeout=30min)
       CONTINUE
     
     // Generate topology-aware config
     config ← SynthesizeConfig(training_config, healthy_nodes)
     
     // Launch training
     LaunchTorchrun(config, resume_from=last_good_checkpoint)
     
     // Training runs...
     // Periodic checkpointing every K steps
     // Checkpoint validation: verify tensor shapes, optimizer state completeness
     
     training_complete ← True
     
   CATCH NodeFailure(failed_node):
     failure_count += 1
     LOG("Node failure: " + failed_node + ", attempt " + failure_count)
     
     // Save emergency checkpoint if possible
     TryEmergencyCheckpoint()
     
     // Remove failed node, acquire replacement
     cluster_config.exclude(failed_node)
     replacement ← AcquireSpareNode()
     IF replacement:
       cluster_config.add(replacement)
     
     // Validate last checkpoint
     last_good_checkpoint ← FindLatestValidCheckpoint(training_config.save_path)
     ValidateCheckpoint(last_good_checkpoint)  // Verify all shards, no corruption
     
   CATCH NaNLoss(step):
     LOG("NaN loss detected at step " + step)
     // Rollback to checkpoint before divergence
     last_good_checkpoint ← FindCheckpointBefore(step - rollback_margin)
     // Optionally: reduce learning rate, increase gradient clipping
     
   CATCH Timeout:
     failure_count += 1
     // Likely NCCL hang or straggler
     DiagnoseHang()  // Dump NCCL debug info, check GPU utilization
     KillAllRanks()
     // Restart

4. IF failure_count >= max_failures:
     ALERT("Training failed after max retries")
     RETURN last_good_checkpoint

5. RETURN final_checkpoint
```

### 10.3 Checkpoint Validation

```
ALGORITHM: ValidateCheckpoint
INPUT: checkpoint_path, expected_config {tp, pp, dp, num_params}
OUTPUT: valid (boolean), issues (list)

1. issues ← []

2. // Check all expected shards exist
   FOR tp_rank IN [0, tp):
     FOR pp_rank IN [0, pp):
       shard_path ← checkpoint_path + "/mp_rank_" + tp_rank + "_" + pp_rank
       IF NOT exists(shard_path):
         issues.APPEND("Missing shard: tp=" + tp_rank + " pp=" + pp_rank)
         CONTINUE
       
       // Verify file integrity (checksum)
       IF NOT VerifyChecksum(shard_path):
         issues.APPEND("Corrupt shard: " + shard_path)
         CONTINUE
       
       // Load and verify tensor shapes
       state ← LoadStateDict(shard_path)
       FOR key, tensor IN state.items():
         expected_shape ← ComputeExpectedShape(key, model_config, tp, pp)
         IF tensor.shape != expected_shape:
           issues.APPEND("Shape mismatch: " + key)
         IF HasNaN(tensor) OR HasInf(tensor):
           issues.APPEND("NaN/Inf in: " + key)

3. // Verify optimizer state completeness
   optimizer_state ← LoadOptimizerState(checkpoint_path)
   FOR param_group IN optimizer_state["param_groups"]:
     FOR param_id IN param_group["params"]:
       IF param_id NOT IN optimizer_state["state"]:
         issues.APPEND("Missing optimizer state for param: " + param_id)
       ELSE:
         state = optimizer_state["state"][param_id]
         IF "exp_avg" NOT IN state OR "exp_avg_sq" NOT IN state:
           issues.APPEND("Incomplete Adam state for param: " + param_id)

4. // Verify training metadata
   metadata ← LoadMetadata(checkpoint_path)
   ASSERT metadata["iteration"] > 0
   ASSERT metadata["tokens_seen"] > 0

5. RETURN (len(issues) == 0), issues
```

---

## 11. Profiling, Instrumentation, and Regression Gating

### 11.1 Profiling Tool Selection

| Tool | Target | What It Captures | When to Use |
|------|--------|-----------------|-------------|
| **Nsight Systems** | NVIDIA GPUs | End-to-end timeline: kernels, NCCL ops, CUDA API, CPU activity | Step-time decomposition, overlap analysis, pipeline bubble visualization |
| **Nsight Compute** | NVIDIA GPUs | Per-kernel metrics: occupancy, memory throughput, warp stall reasons | Kernel-level optimization (why is a specific GEMM slow?) |
| **PyTorch Profiler** | Framework-level | Op-level breakdown, memory allocation timeline, FLOPS counting | Quick triage, integration with TensorBoard |
| **rocprof / rocprof-sys** | AMD GPUs | Kernel timelines, HIP API traces, hardware counters | AMD cluster profiling, analogous to Nsight |
| **NCCL/RCCL debug logs** | Collectives | Ring/tree topology, algorithm selection, per-collective timing | Diagnosing communication bottlenecks, topology mismatch |

### 11.2 Step-Time Decomposition

Every training step is decomposed into:

$$
T_{\text{step}} = T_{\text{forward}} + T_{\text{backward}} + T_{\text{optimizer}} + T_{\text{comm}} + T_{\text{bubble}} + T_{\text{dataloader}} + T_{\text{misc}}
$$

where $T_{\text{comm}}$ is the **non-overlapped** portion of communication (the part that extends step time beyond pure compute).

**Methodology:**

```
ALGORITHM: StepTimeDecomposition
INPUT: profiler_trace (Nsight Systems or PyTorch Profiler output)
OUTPUT: decomposition report

1. // Parse kernel timeline
   forward_kernels ← FilterKernels(trace, phase="forward")
   backward_kernels ← FilterKernels(trace, phase="backward")
   nccl_kernels ← FilterKernels(trace, name_contains="nccl" OR "rccl")
   optimizer_kernels ← FilterKernels(trace, phase="optimizer")

2. // Compute pure compute time
   T_forward ← SumDuration(forward_kernels) - OverlapWith(nccl_kernels, forward_kernels)
   T_backward ← SumDuration(backward_kernels) - OverlapWith(nccl_kernels, backward_kernels)
   T_optimizer ← SumDuration(optimizer_kernels)

3. // Compute communication time (non-overlapped only)
   T_comm_total ← SumDuration(nccl_kernels)
   T_comm_overlapped ← OverlapWith(nccl_kernels, forward_kernels + backward_kernels)
   T_comm_exposed ← T_comm_total - T_comm_overlapped

4. // Pipeline bubble (idle GPU time between microbatches)
   T_bubble ← T_step - T_forward - T_backward - T_optimizer - T_comm_exposed - T_dataloader

5. // Dataloader stall (CPU→GPU data transfer wait)
   T_dataloader ← SumDuration(FilterKernels(trace, name="DataLoader"))

6. // Compute MFU
   total_flops ← 6 × N_params × mbs × seq_len × gas
   T_step ← EndTime(trace) - StartTime(trace)
   MFU ← total_flops / (T_step × peak_flops_per_gpu)

7. REPORT:
     "Step time: {T_step:.2f} ms"
     "  Forward compute: {T_forward:.2f} ms ({T_forward/T_step*100:.1f}%)"
     "  Backward compute: {T_backward:.2f} ms ({T_backward/T_step*100:.1f}%)"
     "  Optimizer step: {T_optimizer:.2f} ms ({T_optimizer/T_step*100:.1f}%)"
     "  Communication (exposed): {T_comm_exposed:.2f} ms ({T_comm_exposed/T_step*100:.1f}%)"
     "  Communication (overlapped): {T_comm_overlapped:.2f} ms"
     "  Pipeline bubble: {T_bubble:.2f} ms ({T_bubble/T_step*100:.1f}%)"
     "  Dataloader: {T_dataloader:.2f} ms ({T_dataloader/T_step*100:.1f}%)"
     "  MFU: {MFU*100:.1f}%"
```

### 11.3 Regression Gating

Every configuration change must pass through a regression gate:

```
ALGORITHM: RegressionGate
INPUT: baseline_metrics, candidate_metrics, tolerance_config
OUTPUT: pass/fail, regression_report

1. // Throughput regression
   throughput_change ← (candidate.tokens_per_sec - baseline.tokens_per_sec) /
                        baseline.tokens_per_sec
   IF throughput_change < -tolerance_config.throughput_threshold:  // e.g., -2%
     FAIL("Throughput regression: " + throughput_change)

2. // Memory regression
   memory_change ← (candidate.peak_memory - baseline.peak_memory) /
                    baseline.peak_memory
   IF memory_change > tolerance_config.memory_threshold:  // e.g., +5%
     WARN("Memory increase: " + memory_change)

3. // Loss parity (same data, same seed, N steps)
   loss_divergence ← MaxAbsDiff(candidate.loss_curve, baseline.loss_curve) /
                      Mean(baseline.loss_curve)
   IF loss_divergence > tolerance_config.loss_threshold:  // e.g., 1%
     FAIL("Loss divergence: " + loss_divergence)

4. // Gradient norm stability
   gnorm_cv_change ← |candidate.gnorm_cv - baseline.gnorm_cv|
   IF gnorm_cv_change > tolerance_config.gnorm_threshold:
     WARN("Gradient norm variance change: " + gnorm_cv_change)

5. // Communication overlap ratio
   overlap_change ← candidate.overlap_ratio - baseline.overlap_ratio
   IF overlap_change < -tolerance_config.overlap_threshold:
     WARN("Reduced communication overlap")

6. IF all checks PASS: RETURN PASS
   ELSE: RETURN FAIL with regression_report
```

---

## 12. Cross-Vendor Portability: NVIDIA and AMD

### 12.1 Key Differences

| Aspect | NVIDIA (A100/H100/B200) | AMD (MI300X/MI350) |
|--------|------------------------|-------------------|
| **Interconnect (intra-node)** | NVLink/NVSwitch (900 GB/s H100) | xGMI / Infinity Fabric (896 GB/s MI300X) |
| **Interconnect (inter-node)** | InfiniBand NDR (400 Gb/s) | InfiniBand NDR or RoCE |
| **Collectives library** | NCCL | RCCL (NCCL-compatible API) |
| **Compute precision** | FP8 (E4M3/E5M2), BF16, TF32 | FP8 (E4M3/E5M2), BF16, FP32 (no TF32) |
| **HBM capacity** | 80 GB (A100/H100), 192 GB (B200) | 192 GB (MI300X), 288 GB (MI350) |
| **Kernel ecosystem** | CUDA, cuBLAS, cuDNN, Triton | HIP, hipBLAS, MIOpen, Triton (ROCm backend) |
| **Flash Attention** | Native CUDA (Dao), CK-based (Composable Kernel) | CK-based FlashAttention, Triton-based |
| **Profiling** | Nsight Systems, Nsight Compute | rocprof, rocprof-sys (Omnitrace), omniperf |

### 12.2 Portability Strategy

```
ALGORITHM: CrossVendorConfigSynthesis
INPUT: model_config, target_vendor, cluster_topology
OUTPUT: vendor-optimized training config

1. // Detect vendor
   vendor ← DetectGPUVendor()  // "nvidia" or "amd"

2. // Common config (vendor-agnostic)
   config ← {
     "bf16": True,
     "flash_attention": True,
     "gradient_clipping": 1.0,
     "distributed_backend": "nccl",  // RCCL is API-compatible
   }

3. IF vendor == "nvidia":
     config["env"]["NCCL_NET_GDR_LEVEL"] ← "5"
     config["env"]["NCCL_IB_HCA"] ← AutoDetectIBHCA()
     config["env"]["CUDA_DEVICE_MAX_CONNECTIONS"] ← "1"
     config["flash_attn_impl"] ← "flash_attn"  // Dao's CUDA implementation
     config["fused_kernels"] ← "apex"  // or Megatron-Core fused kernels
     IF gpu_arch ≥ "sm_90":  // H100+
       config["fp8"] ← True
       config["fp8_recipe"] ← "delayed_scaling"

4. IF vendor == "amd":
     config["env"]["NCCL_NET_GDR_LEVEL"] ← "3"  // Adjust per cluster
     config["env"]["RCCL_MSCCL_ENABLE"] ← "0"    // Disable if unstable
     config["env"]["HIP_FORCE_DEV_KERNARG"] ← "1"
     config["env"]["GPU_MAX_HW_QUEUES"] ← "2"
     config["flash_attn_impl"] ← "ck"  // Composable Kernel backend
     config["fused_kernels"] ← "triton"  // Triton ROCm backend
     // MI300X: 8 XCDs per die, each with own L2 cache
     // Kernel launch overhead is higher; prefer larger kernels, CUDA graphs
     config["use_cuda_graphs"] ← True  // HIP graphs
     
     // xGMI topology: MI300X in 8-GPU OAM configuration
     // Fully connected via Infinity Fabric at 896 GB/s aggregate
     // TP=8 is efficient within a single node
     config["tp"] ← MIN(8, cluster_topology.gpus_per_node)

5. // HBM-aware memory planning
   hbm_capacity ← GetHBMCapacity(vendor, gpu_model)
   // MI300X: 192 GB → can fit larger models per GPU than H100 80 GB
   // This may allow reducing TP/PP degrees
   IF hbm_capacity > 128:
     // Re-evaluate: can we reduce PP or TP?
     TryReduceParallelism(config, hbm_capacity)

6. RETURN config
```

### 12.3 When to Use Which Abstraction

| Scenario | Recommended Stack | Rationale |
|----------|------------------|-----------|
| Dense pretraining, 100B+, NVIDIA | Megatron-Core + custom CUDA kernels | Best-in-class TP+PP+CP+EP, interleaved schedules, distributed optimizer |
| Dense pretraining, 100B+, AMD | Megatron-Core + Triton kernels + RCCL | Megatron-Core is vendor-agnostic at the distributed layer; replace CUDA kernels with Triton |
| Fine-tuning (SFT/DPO), ≤70B | PyTorch FSDP2 + DTensor + torch.compile | Simpler integration, composable with HuggingFace, better custom training loops |
| MoE pretraining | Megatron-Core (EP support) or DeepSpeed-MoE | Native expert parallelism, token routing, load balancing |
| Research prototyping | DeepSpeed ZeRO-3 + HuggingFace Trainer | Fastest iteration; ZeRO-3 fits anything; trade throughput for simplicity |
| Multi-vendor production | Triton kernels + PyTorch FSDP2 + torchrun | Maximum portability; Triton compiles to both CUDA and ROCm |

---

## 13. Glossary and Reference Formulae

### 13.1 Parallelization Terms

| Symbol | Full Name | Definition |
|--------|-----------|------------|
| **tp** | Tensor Parallelism degree | Number of GPUs sharing a single layer's weight matrices (column/row partitioning) |
| **pp** | Pipeline Parallelism degree | Number of pipeline stages; each stage holds $N_{\text{layers}} / \text{pp}$ consecutive layers |
| **dp** | Data Parallelism degree | Number of model replicas processing independent micro-batches |
| **cp** | Context Parallelism degree | Number of GPUs splitting the sequence dimension for long-context training |
| **ep** | Expert Parallelism degree | Number of GPUs across which MoE experts are distributed |
| **dp\_if\_zero1** | Effective DP for ZeRO-1 sharding | DP degree used for optimizer state partitioning (= dp if ZeRO-1+ is enabled, else 1) |
| **dp\_if\_zero2** | Effective DP for ZeRO-2 sharding | DP degree used for gradient partitioning (= dp if ZeRO-2+ is enabled, else 1) |
| **dp\_if\_zero3** | Effective DP for ZeRO-3 sharding | DP degree used for parameter partitioning (= dp if ZeRO-3 is enabled, else 1) |

### 13.2 Batch Size Terms

| Symbol | Full Name | Formula / Definition |
|--------|-----------|---------------------|
| **mbs** | Micro-batch size | Number of sequences processed per GPU per forward-backward pass |
| **gas** | Gradient accumulation steps | Number of micro-batches accumulated before an optimizer step |
| **mseqlen** | Per-GPU sequence length | $\text{seq\_len} / \text{cp}$ — effective sequence length after context parallelism |
| **GBS** | Global Batch Size (in sequences) | $\text{mbs} \times \text{dp} \times \text{gas}$ |
| **GBS_tokens** | Global Batch Size (in tokens) | $\text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}$ |

### 13.3 Memory Terms

| Symbol | Formula | Notes |
|--------|---------|-------|
| **model\_bf16** | $\text{bf16\_bytes} \times N_{\text{params}} = 2 \times N_{\text{layers}} \times 12 \times h^2$ | BF16 parameters; divided by tp, pp, dp\_if\_zero3 |
| **model\_fp32** | $2 \times \text{model\_bf16} / \text{dp\_if\_zero1}$ | FP32 master copy for optimizer |
| **grads\_fp32** | $2 \times \text{model\_bf16} / \text{dp\_if\_zero2}$ | FP32 accumulated gradients |
| **optimstates** | $4 \times \text{model\_bf16} / \text{dp\_if\_zero1}$ | Adam: momentum (FP32) + variance (FP32) |
| **activations** | $f(\text{model\_config}, \text{mseqlen}, \text{mbs}, \text{tp}, \text{cp}, \text{pp}, \text{dp\_if\_zero3})$ | Depends on recomputation strategy |

### 13.4 Core Formulae

**Peak memory per GPU:**

$$
\text{peak\_memory} = \text{model\_bf16} + \text{model\_fp32} + \text{grads\_fp32} + \text{optimstates} + \text{activations}
$$

**Compute per GPU per step:**

$$
C_{\text{step}} = 6 \times N_{\text{params}} \times \text{mbs} \times \text{seq\_len} \times \text{gas}
$$

**Model FLOPS Utilization:**

$$
\text{MFU} = \frac{C_{\text{step}}}{T_{\text{step}} \times \text{peak\_FLOPS}}
$$

**Pipeline bubble fraction (1F1B):**

$$
\beta_{\text{1F1B}} = \frac{\text{pp} - 1}{\text{gas} + \text{pp} - 1}
$$

**Pipeline bubble fraction (interleaved, $v$ virtual stages):**

$$
\beta_{\text{interleaved}} = \frac{\text{pp} - 1}{\text{gas} \times v + \text{pp} - 1}
$$

**World-size factorization constraint:**

$$
N_{\text{GPUs}} = \text{tp} \times \text{pp} \times \text{dp} \times \text{cp} \times \text{ep}^*
$$

> *Note: EP typically shares the DP dimension rather than being a separate multiplicative factor. The actual constraint is $N_{\text{GPUs}} = \text{tp} \times \text{pp} \times \text{dp}$, where $\text{dp} = \text{dp\_pure} \times \text{cp}$, and EP is carved out of dp\_pure such that $\text{dp\_pure} = \text{ep} \times \text{dp\_remaining}$.

**Tokens per second (throughput):**

$$
\text{TPS} = \frac{\text{GBS\_tokens}}{T_{\text{step}}} = \frac{\text{mbs} \times \text{dp} \times \text{gas} \times \text{seq\_len}}{T_{\text{step}}}
$$

**Tokens per second per GPU:**

$$
\text{TPS\_per\_GPU} = \frac{\text{TPS}}{N_{\text{GPUs}}}
$$

---

## 14. Conclusion

This report has presented a rigorous, three-step methodology for configuring distributed LLM training at any scale:

1. **Fit the model into memory** by deriving exact memory requirements from model configuration and selecting the minimum parallelism (TP, PP, ZeRO stage, recomputation) that satisfies HBM constraints. Topology-aware placement ensures TP stays on NVLink, PP and DP span InfiniBand.

2. **Satisfy the target global batch size** by adjusting DP width, gradient accumulation, and context parallelism. The GBS is a convergence hyperparameter that must not be compromised by infrastructure decisions.

3. **Optimize throughput** through controlled experimentation: scaling TP, tuning ZeRO stages, introducing PP with interleaved schedules, maximizing micro-batch size, and enabling compute-communication overlap. Every change is validated through MFU measurement, step-time decomposition, and loss-parity regression gates.

The parallelization strategy matrix provides a comprehensive reference relating each strategy to its batch size impact, memory reduction, communication pattern, and overlap characteristics. Framework-specific guidance for Megatron-Core, DeepSpeed, FSDP/DTensor, Triton, and torchrun ensures that these principles translate directly into production configurations across NVIDIA and AMD clusters.

> **Final Engineering Maxim:** Distributed training configuration is not an art—it is constrained optimization over memory, bandwidth, latency, and compute. Every decision must be justified by a formula, validated by a measurement, and guarded by a regression test. The methodology in this report provides the analytical framework; the engineer provides the cluster-specific constants and the discipline to measure before concluding.

---

*This report reflects production practices validated across A100, H100, and MI300X clusters at scales from 8 to 16,384 GPUs, encompassing dense pretraining (7B–405B), MoE training (up to 1.8T parameters), long-context training (128K+ tokens), and fine-tuning (SFT, DPO, GRPO) workloads.*